## Cell 1 — S2 Environment và Published Shared Bundle

Cell này thực hiện:

1. Khóa identity S2 — MobileViT-S, confirmation seed 2026.
2. Kiểm tra GPU và strict deterministic mode.
3. Tìm đúng một published Phase 2A shared bundle.
4. Xác minh ba manifest của shared bundle.
5. Xác minh toàn bộ artifact theo size và SHA-256.
6. Xác minh 10 required artifacts.
7. Xác minh S2 encoder initial checkpoint.
8. Xác minh shared decoder initial checkpoint.
9. Khóa training, validation, reproducibility và augmentation contracts.
10. Kiểm tra exact-resume audit.
11. Không đọc dataset hoặc caption.
12. Không mở Held-out Test.
13. Chưa bắt đầu training.

In [ ]:
import csv
import hashlib
import json
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()

STUDENT_ID = "S2"
STUDENT_NAME = "MobileViT-S"
DEVELOPMENT_SEED = 2026

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
RUN_ROOT = (
    KAGGLE_WORKING_ROOT
    / ""
    / "phase2_student_baselines"
    / "s2_mobilevit_s_seed2026"
)
RUN_CONFIG_DIR = RUN_ROOT / "configs"
RUN_AUDIT_DIR = RUN_ROOT / "audits"
RUN_LOG_DIR = RUN_ROOT / "logs"
RUN_CHECKPOINT_DIR = RUN_ROOT / "checkpoints"

for directory in [
    RUN_ROOT,
    RUN_CONFIG_DIR,
    RUN_AUDIT_DIR,
    RUN_LOG_DIR,
    RUN_CHECKPOINT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def read_json(path):
    with open(path, "r", encoding="utf-8") as source_file:
        return json.load(source_file)


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as output_file:
        json.dump(
            payload,
            output_file,
            ensure_ascii=False,
            indent=2,
        )


def sha256_file(path, chunk_size=4 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as source_file:
        while True:
            chunk = source_file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    candidate = (root / relative_path).resolve()
    assert candidate.is_relative_to(root), (
        f"Đường dẫn thoát khỏi bundle root: {relative_path}"
    )
    return candidate


def set_deterministic_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(
        True,
        warn_only=False,
    )


assert KAGGLE_INPUT_ROOT.is_dir(), (
    "Không tìm thấy /kaggle/input. Cell phải chạy trên Kaggle."
)
assert torch.cuda.is_available(), (
    "GPU chưa được bật. Vào Settings -> Accelerator -> GPU."
)
set_deterministic_seed(DEVELOPMENT_SEED)

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_properties = torch.cuda.get_device_properties(0)


print("=" * 72)
print("DISCOVER PUBLISHED PHASE 2 SHARED BUNDLE")
print("=" * 72)
print()

ready_marker_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
    )
)
print("READY marker matches:", len(ready_marker_matches))
for marker_match in ready_marker_matches:
    print("-", marker_match)

assert len(ready_marker_matches) == 1, (
    "Cần gắn đúng một version output của Phase 2A Shared Pipeline. "
    f"Hiện tìm thấy {len(ready_marker_matches)} marker."
)

SHARED_BUNDLE_READY_PATH = (
    ready_marker_matches[0].resolve()
)
SHARED_BUNDLE_ROOT = (
    SHARED_BUNDLE_READY_PATH.parent.resolve()
)
SHARED_BUNDLE_READY_SHA256 = sha256_file(
    SHARED_BUNDLE_READY_PATH
)
SHARED_BUNDLE_READY = read_json(
    SHARED_BUNDLE_READY_PATH
)

assert SHARED_BUNDLE_READY["status"] == "ready"
assert SHARED_BUNDLE_READY[
    "full_training_authorized"
] is True
assert SHARED_BUNDLE_READY["relocatable"] is True
assert SHARED_BUNDLE_READY["student_ids"] == [
    "S1",
    "S2",
    "S3",
]
assert SHARED_BUNDLE_READY[
    "data_policy"
]["test_set_status"] == "sealed"
assert SHARED_BUNDLE_READY[
    "data_policy"
]["test_set_used"] is False
assert SHARED_BUNDLE_READY[
    "data_policy"
]["flickr_used"] is False
assert SHARED_BUNDLE_READY[
    "data_policy"
]["full_training_started"] is False


manifest_paths = {}
for manifest_name in [
    "csv",
    "json",
    "sha256sums",
]:
    manifest_record = SHARED_BUNDLE_READY[
        "manifests"
    ][manifest_name]
    manifest_path = resolve_inside(
        SHARED_BUNDLE_ROOT,
        manifest_record["relative_path"],
    )
    assert manifest_path.is_file()
    assert sha256_file(manifest_path) == (
        manifest_record["sha256"]
    )
    manifest_paths[manifest_name] = manifest_path

with open(
    manifest_paths["csv"],
    "r",
    encoding="utf-8",
    newline="",
) as manifest_file:
    manifest_rows = list(
        csv.DictReader(manifest_file)
    )
manifest_json = read_json(
    manifest_paths["json"]
)

assert len(manifest_rows) == int(
    SHARED_BUNDLE_READY["artifact_count"]
)
assert len(manifest_rows) == int(
    manifest_json["artifact_count"]
)
assert len(manifest_rows) > 0

csv_relative_paths = [
    row["relative_path"]
    for row in manifest_rows
]
assert len(csv_relative_paths) == len(
    set(csv_relative_paths)
)
json_rows_by_path = {
    row["relative_path"]: row
    for row in manifest_json["artifacts"]
}
assert set(csv_relative_paths) == set(
    json_rows_by_path
)

verified_bytes = 0
for row in manifest_rows:
    artifact_path = resolve_inside(
        SHARED_BUNDLE_ROOT,
        row["relative_path"],
    )
    expected_size = int(row["size_bytes"])
    assert artifact_path.is_file(), (
        f"Thiếu shared artifact: {artifact_path}"
    )
    assert artifact_path.stat().st_size == (
        expected_size
    )
    assert sha256_file(artifact_path) == (
        row["sha256"]
    )

    json_row = json_rows_by_path[
        row["relative_path"]
    ]
    assert json_row["role"] == row["role"]
    assert int(json_row["size_bytes"]) == (
        expected_size
    )
    assert json_row["sha256"] == row["sha256"]
    verified_bytes += expected_size

assert verified_bytes == int(
    SHARED_BUNDLE_READY["total_size_bytes"]
)


required_artifact_paths = {}
for artifact_name, relative_path in (
    SHARED_BUNDLE_READY[
        "required_artifacts"
    ].items()
):
    artifact_path = resolve_inside(
        SHARED_BUNDLE_ROOT,
        relative_path,
    )
    assert artifact_path.is_file()
    assert relative_path in csv_relative_paths
    required_artifact_paths[
        artifact_name
    ] = artifact_path

expected_required_names = {
    "full_training_ready",
    "training_contract",
    "validation_policy",
    "reproducibility_contract",
    "stateless_augmentation_contract",
    "exact_resume_audit",
    "shared_decoder_checkpoint",
    "s1_encoder_checkpoint",
    "s2_encoder_checkpoint",
    "s3_encoder_checkpoint",
}
assert set(required_artifact_paths) == (
    expected_required_names
)

training_contract = read_json(
    required_artifact_paths[
        "training_contract"
    ]
)
validation_policy = read_json(
    required_artifact_paths[
        "validation_policy"
    ]
)
reproducibility_contract = read_json(
    required_artifact_paths[
        "reproducibility_contract"
    ]
)
stateless_augmentation_contract = read_json(
    required_artifact_paths[
        "stateless_augmentation_contract"
    ]
)
exact_resume_audit = read_json(
    required_artifact_paths[
        "exact_resume_audit"
    ]
)

assert training_contract["status"] == "locked"
assert training_contract[
    "contract_version"
] == SHARED_BUNDLE_READY[
    "training_contract_version"
]
assert training_contract["scope"][
    "primary_dataset"
] == "MS COCO 2014 only"
assert training_contract["scope"][
    "split_protocol"
] == "Karpathy"
assert training_contract["scope"][
    "student_models"
]["S2"] == "MobileViT-S"
assert training_contract["scope"][
    "test_set_status"
] == "sealed"
assert validation_policy["status"] == "locked"
assert reproducibility_contract["status"] == "locked"
assert int(
    reproducibility_contract["development_seed"]
) == 42
assert DEVELOPMENT_SEED in {
    int(seed)
    for seed in reproducibility_contract[
        "confirmation_seeds"
    ]
}
assert DEVELOPMENT_SEED != int(
    reproducibility_contract["development_seed"]
)
assert stateless_augmentation_contract[
    "status"
] == "locked"
assert exact_resume_audit["status"] == "passed"

# Explicitly verify the S2 initial encoder checkpoint is part of the
# published, checksummed shared bundle.
S2_ENCODER_INITIAL_CHECKPOINT_PATH = (
    required_artifact_paths[
        "s2_encoder_checkpoint"
    ]
)
SHARED_DECODER_INITIAL_CHECKPOINT_PATH = (
    required_artifact_paths[
        "shared_decoder_checkpoint"
    ]
)
assert S2_ENCODER_INITIAL_CHECKPOINT_PATH.is_file()
assert SHARED_DECODER_INITIAL_CHECKPOINT_PATH.is_file()


CELL_AUDIT_PATH = (
    RUN_AUDIT_DIR
    / "s2_cell01_environment_and_shared_bundle_audit.json"
)
cell_elapsed_seconds = (
    time.perf_counter() - CELL_STARTED_AT
)

cell_audit = {
    "schema_version": (
        "phase2_s2_cell01_environment_bundle_audit_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "passed",
    "student": {
        "student_id": STUDENT_ID,
        "student_name": STUDENT_NAME,
        "seed": DEVELOPMENT_SEED,
        "run_role": "confirmation",
        "shared_initial_checkpoint_seed": 42,
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "pytorch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "pytorch_cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "gpu": gpu_name,
        "gpu_capability": list(
            torch.cuda.get_device_capability(0)
        ),
        "gpu_memory_bytes": int(
            gpu_properties.total_memory
        ),
        "strict_deterministic": True,
    },
    "shared_bundle": {
        "root": str(SHARED_BUNDLE_ROOT),
        "ready_marker": str(
            SHARED_BUNDLE_READY_PATH
        ),
        "ready_marker_sha256": (
            SHARED_BUNDLE_READY_SHA256
        ),
        "artifact_count": len(manifest_rows),
        "verified_bytes": verified_bytes,
        "manifest_csv_sha256": (
            sha256_file(
                manifest_paths["csv"]
            )
        ),
        "manifest_json_sha256": (
            sha256_file(
                manifest_paths["json"]
            )
        ),
        "sha256sums_sha256": (
            sha256_file(
                manifest_paths["sha256sums"]
            )
        ),
        "required_artifacts": {
            name: str(path)
            for name, path
            in sorted(
                required_artifact_paths.items()
            )
        },
        "s2_encoder_checkpoint": str(
            S2_ENCODER_INITIAL_CHECKPOINT_PATH
        ),
        "shared_decoder_checkpoint": str(
            SHARED_DECODER_INITIAL_CHECKPOINT_PATH
        ),
    },
    "data_policy": {
        "dataset_read": False,
        "caption_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "training_started": False,
    },
    "elapsed_seconds": cell_elapsed_seconds,
}
write_json(CELL_AUDIT_PATH, cell_audit)
CELL_AUDIT_SHA256 = sha256_file(
    CELL_AUDIT_PATH
)


print()
print("=" * 72)
print("S2 CONFIRMATION ENVIRONMENT")
print("=" * 72)
print("Student                 :", STUDENT_ID, "—", STUDENT_NAME)
print("Seed                    :", DEVELOPMENT_SEED)
print("Python                  :", platform.python_version())
print("PyTorch                 :", torch.__version__)
print("CUDA                    :", torch.version.cuda)
print("GPU                     :", gpu_name)
print(
    "GPU memory              :",
    f"{gpu_properties.total_memory / (1024 ** 3):.2f} GiB",
)
print("Strict deterministic    : True")

print()
print("=" * 72)
print("PHASE 2 S2 CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Confirmation seed         :", DEVELOPMENT_SEED)
print("Published shared bundle   : VERIFIED")
print("Relocatable bundle        : True")
print("Manifest files            : 3 / 3 VERIFIED")
print(
    "Manifest rows             :",
    f"{len(manifest_rows)} / {len(manifest_rows)} PASSED",
)
print("Required artifacts        : 10 / 10 PASSED")
print("Initial checkpoints       : 4 / 4 VERIFIED")
print("S2 encoder checkpoint     : VERIFIED")
print("Shared decoder checkpoint : VERIFIED")
print("Training contract         : LOCKED")
print("Validation policy         : LOCKED")
print("Reproducibility contract  : LOCKED")
print("Stateless augmentation    : LOCKED")
print("Exact-resume audit        : PASSED")
print("Dataset read              : False")
print("Caption read              : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print("Run root                  :", RUN_ROOT)
print("Audit                     :", CELL_AUDIT_PATH)
print("Audit SHA-256             :", CELL_AUDIT_SHA256)
print(
    "Elapsed                   :",
    round(cell_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : discover and verify Phase 1 formal "
    "output plus MS COCO image roots"
)

## Cell 2 — Xác minh Phase 1 và MS COCO Inputs

Cell này thực hiện:

1. Tìm đúng một Phase 1 formal READY marker.
2. Xác minh SHA256SUMS gồm đúng 32 artifact.
3. Hash lại 7 development artifacts của Phase 1.
4. Xác minh experiment contract, vocabulary và sequence contract.
5. Xác minh vocabulary size 10.080 và min frequency 5.
6. Xác minh maximum decoder steps bằng 31.
7. Chỉ kiểm tra sự tồn tại của Test Parquet, không mở hoặc hash.
8. Tìm đúng thư mục ảnh train2014 và val2014.
9. Probe trực tiếp một ảnh từ mỗi source partition.
10. Khóa expected counts 82.783 train và 40.504 validation images.
11. Không đọc caption payload hoặc Held-out Test.
12. Chưa bắt đầu training.

In [ ]:
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID",
    "STUDENT_NAME",
    "DEVELOPMENT_SEED",
    "KAGGLE_INPUT_ROOT",
    "RUN_CONFIG_DIR",
    "RUN_AUDIT_DIR",
    "SHARED_BUNDLE_ROOT",
    "SHARED_BUNDLE_READY_PATH",
    "SHARED_BUNDLE_READY_SHA256",
    "read_json",
    "write_json",
    "sha256_file",
    "resolve_inside",
]
missing_runtime_names = [
    name for name in required_runtime_names
    if name not in globals()
]
assert not missing_runtime_names, (
    "Thiếu biến từ S2 Cell 1: "
    f"{missing_runtime_names}. Hãy chạy lại Cell 1 trước."
)
assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026


def parse_sha256sums(path):
    checksum_pattern = re.compile(
        r"^([0-9a-fA-F]{64})  (.+)$"
    )
    rows = {}
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as checksum_file:
        for line_number, raw_line in enumerate(
            checksum_file,
            start=1,
        ):
            line = raw_line.rstrip("\r\n")
            if not line:
                continue
            match = checksum_pattern.fullmatch(
                line
            )
            assert match is not None, (
                "SHA256SUMS sai định dạng tại "
                f"dòng {line_number}"
            )
            checksum = match.group(1).lower()
            relative_path = (
                match.group(2)
                .replace("\\", "/")
            )
            assert relative_path not in rows
            rows[relative_path] = checksum
    return rows


def resolve_phase1_artifact(
    phase1_root,
    relative_path,
):
    phase1_root = Path(
        phase1_root
    ).resolve()
    artifact_path = (
        phase1_root / relative_path
    ).resolve()
    assert artifact_path.is_relative_to(
        phase1_root
    )
    assert artifact_path.is_file(), (
        f"Thiếu Phase 1 artifact: {artifact_path}"
    )
    assert artifact_path.stat().st_size > 0
    return artifact_path


def first_matching_image(
    directory,
    prefix,
):
    for candidate in Path(
        directory
    ).iterdir():
        if (
            candidate.is_file()
            and candidate.suffix.lower()
            in {".jpg", ".jpeg"}
            and candidate.name.startswith(
                prefix
            )
        ):
            return candidate
    return None


print("=" * 72)
print("DISCOVER PHASE 1 FORMAL OUTPUT")
print("=" * 72)
print()

phase1_ready_matches = sorted(
    KAGGLE_INPUT_ROOT.rglob(
        "PHASE1_FORMAL_DELIVERABLES_READY.json"
    )
)
print(
    "READY marker matches:",
    len(phase1_ready_matches),
)
for marker_path in phase1_ready_matches:
    print("-", marker_path)

assert len(phase1_ready_matches) == 1, (
    "Cần gắn đúng một version Phase 1 formal output. "
    f"Hiện tìm thấy {len(phase1_ready_matches)} marker."
)

PHASE1_FORMAL_READY_PATH = (
    phase1_ready_matches[0].resolve()
)
PHASE1_ROOT = (
    PHASE1_FORMAL_READY_PATH.parent.resolve()
)
PHASE1_FORMAL_READY = read_json(
    PHASE1_FORMAL_READY_PATH
)
assert PHASE1_FORMAL_READY["status"] == "ready"
assert PHASE1_FORMAL_READY[
    "phase"
] == "phase1_mscoco_data_contract"
assert PHASE1_FORMAL_READY[
    "test_set_status"
] == "sealed"
assert PHASE1_FORMAL_READY[
    "test_caption_exported"
] is False
assert PHASE1_FORMAL_READY[
    "test_used"
] is False
assert PHASE1_FORMAL_READY[
    "flickr_used"
] is False


phase2_input_contract_path = resolve_inside(
    SHARED_BUNDLE_ROOT,
    "configs/phase2_input_contract.json",
)
phase2_input_contract = read_json(
    phase2_input_contract_path
)
assert phase2_input_contract[
    "status"
] == "passed"
assert phase2_input_contract[
    "policy"
]["test_set_status"] == "sealed"
assert phase2_input_contract[
    "policy"
]["test_used"] is False
assert phase2_input_contract[
    "policy"
]["flickr_used"] is False
assert phase2_input_contract[
    "phase1"
]["checksum_rows_verified"] == 32


PHASE1_SHA256SUMS_PATH = (
    PHASE1_ROOT / "SHA256SUMS.txt"
)
assert PHASE1_SHA256SUMS_PATH.is_file()
phase1_sha256sums_sha256 = sha256_file(
    PHASE1_SHA256SUMS_PATH
)
assert phase1_sha256sums_sha256 == (
    PHASE1_FORMAL_READY[
        "sha256sums_sha256"
    ]
)
assert phase1_sha256sums_sha256 == (
    phase2_input_contract[
        "phase1"
    ]["sha256sums_sha256"]
)
phase1_checksum_rows = parse_sha256sums(
    PHASE1_SHA256SUMS_PATH
)
assert len(phase1_checksum_rows) == 32


development_artifact_relatives = {
    "experiment_contract": (
        "configs/experiment_contract_v2.json"
    ),
    "image_manifest_parquet": (
        "parquet/mscoco_image_manifest.parquet"
    ),
    "train_parquet": (
        "parquet/mscoco_train.parquet"
    ),
    "validation_parquet": (
        "parquet/mscoco_validation.parquet"
    ),
    "vocabulary": (
        "vocabulary/word_vocabulary_minfreq5.json"
    ),
    "tokenizer_contract": (
        "vocabulary/tokenizer_contract.json"
    ),
    "sequence_contract": (
        "vocabulary/sequence_encoding_contract.json"
    ),
}
PHASE1_DEVELOPMENT_ARTIFACT_PATHS = {}
phase1_development_hashes = {}

print()
print("=" * 72)
print("VERIFY PHASE 1 DEVELOPMENT ARTIFACTS")
print("=" * 72)
print()

for artifact_name, relative_path in (
    development_artifact_relatives.items()
):
    assert relative_path in phase1_checksum_rows, (
        f"Thiếu checksum Phase 1 cho {relative_path}"
    )
    artifact_path = resolve_phase1_artifact(
        PHASE1_ROOT,
        relative_path,
    )
    actual_sha256 = sha256_file(
        artifact_path
    )
    assert actual_sha256 == (
        phase1_checksum_rows[relative_path]
    ), (
        f"Checksum không khớp: {relative_path}"
    )
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
        artifact_name
    ] = artifact_path
    phase1_development_hashes[
        artifact_name
    ] = actual_sha256
    print(
        f"{artifact_name:24s}: VERIFIED | "
        f"{artifact_path.stat().st_size:,} bytes"
    )


# Held-out Test declaration only. Do not open, hash, inspect schema/metadata,
# load captions or decode images.
TEST_PARQUET_RELATIVE = (
    "parquet/mscoco_test.parquet"
)
assert TEST_PARQUET_RELATIVE in (
    phase1_checksum_rows
)
PHASE1_TEST_PARQUET_PATH = (
    PHASE1_ROOT
    / TEST_PARQUET_RELATIVE
).resolve()
assert PHASE1_TEST_PARQUET_PATH.is_relative_to(
    PHASE1_ROOT
)
assert PHASE1_TEST_PARQUET_PATH.is_file()
TEST_PARQUET_OPENED = False
TEST_PARQUET_HASHED = False


experiment_contract = read_json(
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
        "experiment_contract"
    ]
)
vocabulary = read_json(
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
        "vocabulary"
    ]
)
tokenizer_contract = read_json(
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
        "tokenizer_contract"
    ]
)
sequence_contract = read_json(
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
        "sequence_contract"
    ]
)
assert experiment_contract[
    "dataset"
]["name"] == "MS COCO"
assert str(
    experiment_contract[
        "dataset"
    ]["release"]
) == "2014"
assert experiment_contract[
    "dataset"
]["split_protocol"] == "Karpathy"
assert experiment_contract[
    "dataset"
]["experiment_splits"]["test"][
    "status"
] == "sealed"
assert vocabulary[
    "vocabulary_size"
] == 10_080
assert len(
    vocabulary["id_to_token"]
) == 10_080
assert tokenizer_contract[
    "minimum_frequency"
] == 5
assert sequence_contract[
    "length_policy"
]["maximum_decoder_time_steps"] == 31


print()
print("=" * 72)
print("DISCOVER MS COCO IMAGE ROOTS")
print("=" * 72)
print()

locked_coco = phase2_input_contract[
    "mscoco_images"
]
expected_train_leaf = Path(
    locked_coco["train_leaf"]
)
expected_val_leaf = Path(
    locked_coco["val_leaf"]
)

if (
    expected_train_leaf.is_dir()
    and expected_val_leaf.is_dir()
):
    COCO_DATASET_ROOT = Path(
        locked_coco["dataset_root"]
    ).resolve()
    COCO_TRAIN_IMAGES_LEAF_DIR = (
        expected_train_leaf.resolve()
    )
    COCO_VAL_IMAGES_LEAF_DIR = (
        expected_val_leaf.resolve()
    )
else:
    train_leaf_matches = []
    val_leaf_matches = []
    for candidate in (
        KAGGLE_INPUT_ROOT.rglob(
            "train2014"
        )
    ):
        if (
            candidate.is_dir()
            and first_matching_image(
                candidate,
                "COCO_train2014_",
            )
            is not None
        ):
            train_leaf_matches.append(
                candidate.resolve()
            )
    for candidate in (
        KAGGLE_INPUT_ROOT.rglob(
            "val2014"
        )
    ):
        if (
            candidate.is_dir()
            and first_matching_image(
                candidate,
                "COCO_val2014_",
            )
            is not None
        ):
            val_leaf_matches.append(
                candidate.resolve()
            )
    assert len(train_leaf_matches) == 1
    assert len(val_leaf_matches) == 1
    COCO_TRAIN_IMAGES_LEAF_DIR = (
        train_leaf_matches[0]
    )
    COCO_VAL_IMAGES_LEAF_DIR = (
        val_leaf_matches[0]
    )
    COCO_DATASET_ROOT = (
        COCO_TRAIN_IMAGES_LEAF_DIR.parents[1]
    )

assert COCO_TRAIN_IMAGES_LEAF_DIR.is_relative_to(
    KAGGLE_INPUT_ROOT.resolve()
)
assert COCO_VAL_IMAGES_LEAF_DIR.is_relative_to(
    KAGGLE_INPUT_ROOT.resolve()
)
assert COCO_TRAIN_IMAGES_LEAF_DIR.name == (
    "train2014"
)
assert COCO_VAL_IMAGES_LEAF_DIR.name == (
    "val2014"
)

train_probe = first_matching_image(
    COCO_TRAIN_IMAGES_LEAF_DIR,
    "COCO_train2014_",
)
val_probe = first_matching_image(
    COCO_VAL_IMAGES_LEAF_DIR,
    "COCO_val2014_",
)
assert train_probe is not None
assert val_probe is not None

EXPECTED_COCO_TRAIN_IMAGE_COUNT = int(
    locked_coco["train_image_count"]
)
EXPECTED_COCO_VAL_IMAGE_COUNT = int(
    locked_coco["val_image_count"]
)
assert EXPECTED_COCO_TRAIN_IMAGE_COUNT == (
    82_783
)
assert EXPECTED_COCO_VAL_IMAGE_COUNT == (
    40_504
)

print("Dataset root :", COCO_DATASET_ROOT)
print(
    "Train leaf   :",
    COCO_TRAIN_IMAGES_LEAF_DIR,
)
print("Train probe  :", train_probe.name)
print(
    "Val leaf     :",
    COCO_VAL_IMAGES_LEAF_DIR,
)
print("Val probe    :", val_probe.name)


S2_INPUT_CONTRACT_PATH = (
    RUN_CONFIG_DIR
    / "s2_seed2026_input_contract.json"
)
S2_CELL02_AUDIT_PATH = (
    RUN_AUDIT_DIR
    / "s2_cell02_data_input_audit.json"
)
cell_elapsed_seconds = (
    time.perf_counter() - CELL_STARTED_AT
)

s2_input_contract = {
    "contract_version": (
        "phase2_s2_seed2026_input_contract_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "locked",
    "student": {
        "student_id": STUDENT_ID,
        "student_name": STUDENT_NAME,
        "seed": DEVELOPMENT_SEED,
    },
    "shared_bundle": {
        "root": str(SHARED_BUNDLE_ROOT),
        "ready_marker": str(
            SHARED_BUNDLE_READY_PATH
        ),
        "ready_marker_sha256": (
            SHARED_BUNDLE_READY_SHA256
        ),
    },
    "phase1": {
        "root": str(PHASE1_ROOT),
        "formal_ready_marker": str(
            PHASE1_FORMAL_READY_PATH
        ),
        "sha256sums": str(
            PHASE1_SHA256SUMS_PATH
        ),
        "sha256sums_sha256": (
            phase1_sha256sums_sha256
        ),
        "checksum_row_count": len(
            phase1_checksum_rows
        ),
        "development_artifacts": {
            name: {
                "path": str(
                    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
                        name
                    ]
                ),
                "sha256": (
                    phase1_development_hashes[
                        name
                    ]
                ),
            }
            for name in (
                development_artifact_relatives
            )
        },
    },
    "mscoco_images": {
        "dataset_root": str(
            COCO_DATASET_ROOT
        ),
        "train_leaf": str(
            COCO_TRAIN_IMAGES_LEAF_DIR
        ),
        "val_leaf": str(
            COCO_VAL_IMAGES_LEAF_DIR
        ),
        "expected_train_images": (
            EXPECTED_COCO_TRAIN_IMAGE_COUNT
        ),
        "expected_val_images": (
            EXPECTED_COCO_VAL_IMAGE_COUNT
        ),
        "count_source": (
            "verified Phase 2A input contract"
        ),
        "direct_image_probes_passed": True,
    },
    "policy": {
        "development_artifact_bytes_hashed": True,
        "caption_payload_decoded": False,
        "test_parquet_declared": True,
        "test_parquet_opened": False,
        "test_parquet_hashed": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "training_started": False,
    },
}
write_json(
    S2_INPUT_CONTRACT_PATH,
    s2_input_contract,
)

s2_cell02_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "phase1_formal_ready": True,
    "phase1_checksum_rows": len(
        phase1_checksum_rows
    ),
    "phase1_sha256sums_verified": True,
    "development_artifacts_verified": len(
        development_artifact_relatives
    ),
    "coco_train_leaf_found": True,
    "coco_val_leaf_found": True,
    "coco_direct_image_probes_passed": True,
    "expected_train_images": (
        EXPECTED_COCO_TRAIN_IMAGE_COUNT
    ),
    "expected_val_images": (
        EXPECTED_COCO_VAL_IMAGE_COUNT
    ),
    "caption_payload_decoded": False,
    "test_parquet_opened": False,
    "test_parquet_hashed": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "training_started": False,
    "input_contract": str(
        S2_INPUT_CONTRACT_PATH
    ),
    "elapsed_seconds": (
        cell_elapsed_seconds
    ),
}
write_json(
    S2_CELL02_AUDIT_PATH,
    s2_cell02_audit,
)
S2_INPUT_CONTRACT_SHA256 = sha256_file(
    S2_INPUT_CONTRACT_PATH
)
S2_CELL02_AUDIT_SHA256 = sha256_file(
    S2_CELL02_AUDIT_PATH
)


print()
print("=" * 72)
print("PHASE 2 S2 CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Phase 1 formal READY      : VERIFIED")
print("Phase 1 SHA256SUMS        : VERIFIED")
print("Phase 1 checksum rows     : 32")
print("Development artifacts    : 7 / 7 VERIFIED")
print("Train Parquet             : VERIFIED")
print("Validation Parquet        : VERIFIED")
print("Image manifest Parquet    : VERIFIED")
print("Vocabulary/contracts      : 4 / 4 VERIFIED")
print("COCO train leaf           : FOUND")
print("COCO val leaf             : FOUND")
print("Direct image probes       : 2 / 2 PASSED")
print("Expected train images     : 82783")
print("Expected val images       : 40504")
print("Caption payload decoded   : False")
print("Test Parquet opened       : False")
print("Test Parquet hashed       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print("Input contract            :", S2_INPUT_CONTRACT_PATH)
print(
    "Input contract SHA-256    :",
    S2_INPUT_CONTRACT_SHA256,
)
print("Audit                     :", S2_CELL02_AUDIT_PATH)
print(
    "Audit SHA-256             :",
    S2_CELL02_AUDIT_SHA256,
)
print(
    "Elapsed                   :",
    round(cell_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : load Parquet schemas and build "
    "the S2 train/validation image-caption indexes"
)

## Cell 3 — Development Indexes và Deterministic Image-Balanced Sampler

Cell này thực hiện:

1. Kiểm tra schema Train, Validation và Image Manifest Parquet.
2. Xác minh row counts của ba Parquet.
3. Dựng bản đồ đường dẫn cho 113.287 ảnh Train.
4. Dựng bản đồ đường dẫn cho 5.000 ảnh Validation.
5. Chỉ lưu membership ID của 5.000 ảnh Held-out Test.
6. Kiểm tra mọi split không giao nhau.
7. Kiểm tra tồn tại 256 đường dẫn ảnh đại diện.
8. Dựng Train caption index gồm 566.747 captions.
9. Dựng Validation caption index gồm 25.010 captions.
10. Chỉ đọc token IDs, không đọc caption text.
11. Dựng deterministic image-balanced sampler.
12. Lấy đúng một caption cho mỗi ảnh trong mỗi epoch.
13. Kiểm tra epoch 0 lặp lại chính xác.
14. Kiểm tra caption được luân phiên giữa epoch 0 và epoch 1.
15. Không mở Test Parquet hoặc đọc Test images.

In [ ]:
import re
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
from torch.utils.data import Sampler


CELL_STARTED_AT = time.perf_counter()

EXPECTED_TRAIN_CAPTION_ROWS = 566_747
EXPECTED_TRAIN_IMAGES = 113_287
EXPECTED_VALIDATION_CAPTION_ROWS = 25_010
EXPECTED_VALIDATION_IMAGES = 5_000
EXPECTED_HELD_OUT_TEST_IMAGES = 5_000
EXPECTED_IMAGE_MANIFEST_ROWS = 123_287
EXPECTED_VOCABULARY_SIZE = 10_080
EXPECTED_MAX_FULL_SEQUENCE = 32

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

EXPECTED_DEVELOPMENT_COLUMNS = [
    "annotation_id",
    "image_id",
    "karpathy_split",
    "source_partition",
    "caption",
    "token_ids",
    "original_lexical_length",
    "retained_lexical_length",
    "was_truncated",
]
EXPECTED_IMAGE_MANIFEST_COLUMNS = [
    "image_id",
    "karpathy_split",
    "experiment_split",
    "source_partition",
    "image_path",
]

required_runtime_names = [
    "STUDENT_ID",
    "STUDENT_NAME",
    "DEVELOPMENT_SEED",
    "RUN_CONFIG_DIR",
    "RUN_AUDIT_DIR",
    "PHASE1_DEVELOPMENT_ARTIFACT_PATHS",
    "PHASE1_TEST_PARQUET_PATH",
    "COCO_TRAIN_IMAGES_LEAF_DIR",
    "COCO_VAL_IMAGES_LEAF_DIR",
    "S2_INPUT_CONTRACT_PATH",
    "S2_INPUT_CONTRACT_SHA256",
    "write_json",
    "sha256_file",
]
missing_runtime_names = [
    name for name in required_runtime_names
    if name not in globals()
]
assert not missing_runtime_names, (
    "Thiếu biến từ S2 Cell 1–2: "
    f"{missing_runtime_names}. Hãy chạy lại Cell 1 và Cell 2."
)
assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026

TRAIN_PARQUET_PATH = Path(
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
        "train_parquet"
    ]
)
VALIDATION_PARQUET_PATH = Path(
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
        "validation_parquet"
    ]
)
IMAGE_MANIFEST_PARQUET_PATH = Path(
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS[
        "image_manifest_parquet"
    ]
)


def assert_exact_schema(path, expected_columns):
    schema = pq.read_schema(path)
    assert schema.names == expected_columns, (
        f"Schema không đúng cho {path}\n"
        f"Actual: {schema.names}\n"
        f"Expected: {expected_columns}"
    )
    return schema


def parquet_metadata_summary(path):
    metadata = pq.read_metadata(path)
    return {
        "path": str(path),
        "rows": int(metadata.num_rows),
        "row_groups": int(
            metadata.num_row_groups
        ),
        "columns": int(metadata.num_columns),
        "size_bytes": int(
            Path(path).stat().st_size
        ),
    }


print("=" * 72)
print("S2 DEVELOPMENT PARQUET SCHEMA AUDIT")
print("=" * 72)
print()

train_schema = assert_exact_schema(
    TRAIN_PARQUET_PATH,
    EXPECTED_DEVELOPMENT_COLUMNS,
)
validation_schema = assert_exact_schema(
    VALIDATION_PARQUET_PATH,
    EXPECTED_DEVELOPMENT_COLUMNS,
)
image_manifest_schema = assert_exact_schema(
    IMAGE_MANIFEST_PARQUET_PATH,
    EXPECTED_IMAGE_MANIFEST_COLUMNS,
)
for schema in [
    train_schema,
    validation_schema,
]:
    assert pa.types.is_int64(
        schema.field("annotation_id").type
    )
    assert pa.types.is_int64(
        schema.field("image_id").type
    )
    assert pa.types.is_list(
        schema.field("token_ids").type
    )
    assert pa.types.is_int32(
        schema.field(
            "token_ids"
        ).type.value_type
    )
    assert pa.types.is_boolean(
        schema.field(
            "was_truncated"
        ).type
    )

parquet_summaries = {
    "train": parquet_metadata_summary(
        TRAIN_PARQUET_PATH
    ),
    "validation": parquet_metadata_summary(
        VALIDATION_PARQUET_PATH
    ),
    "image_manifest": parquet_metadata_summary(
        IMAGE_MANIFEST_PARQUET_PATH
    ),
}
assert parquet_summaries[
    "train"
]["rows"] == EXPECTED_TRAIN_CAPTION_ROWS
assert parquet_summaries[
    "validation"
]["rows"] == EXPECTED_VALIDATION_CAPTION_ROWS
assert parquet_summaries[
    "image_manifest"
]["rows"] == EXPECTED_IMAGE_MANIFEST_ROWS
for name, summary in (
    parquet_summaries.items()
):
    print(
        f"{name:14s}: {summary['rows']:,} rows | "
        f"{summary['row_groups']} row groups | "
        f"{summary['columns']} columns"
    )


print()
print("=" * 72)
print("BUILD IMAGE PATH AND SPLIT MEMBERSHIP MAPS")
print("=" * 72)
print()

manifest_table = pq.read_table(
    IMAGE_MANIFEST_PARQUET_PATH,
    columns=EXPECTED_IMAGE_MANIFEST_COLUMNS,
)
manifest = manifest_table.to_pydict()

S2_TRAIN_IMAGE_PATH_BY_ID = {}
S2_VALIDATION_IMAGE_PATH_BY_ID = {}
S2_HELD_OUT_TEST_IMAGE_IDS = set()

image_ids_seen = set()
karpathy_split_counts = Counter()
experiment_split_counts = Counter()
source_partition_counts = Counter()
invalid_filename_examples = []

train_filename_pattern = re.compile(
    r"^COCO_train2014_\d{12}\.jpg$",
    flags=re.IGNORECASE,
)
val_filename_pattern = re.compile(
    r"^COCO_val2014_\d{12}\.jpg$",
    flags=re.IGNORECASE,
)

for (
    image_id,
    karpathy_split,
    experiment_split,
    source_partition,
    stored_image_path,
) in zip(
    manifest["image_id"],
    manifest["karpathy_split"],
    manifest["experiment_split"],
    manifest["source_partition"],
    manifest["image_path"],
):
    image_id = int(image_id)
    assert image_id not in image_ids_seen
    image_ids_seen.add(image_id)
    karpathy_split_counts[
        karpathy_split
    ] += 1
    experiment_split_counts[
        experiment_split
    ] += 1
    source_partition_counts[
        source_partition
    ] += 1

    filename = Path(
        stored_image_path
    ).name
    if source_partition == "train2014":
        current_path = (
            Path(
                COCO_TRAIN_IMAGES_LEAF_DIR
            )
            / filename
        )
        filename_valid = (
            train_filename_pattern.fullmatch(
                filename
            )
            is not None
        )
    elif source_partition == "val2014":
        current_path = (
            Path(
                COCO_VAL_IMAGES_LEAF_DIR
            )
            / filename
        )
        filename_valid = (
            val_filename_pattern.fullmatch(
                filename
            )
            is not None
        )
    else:
        current_path = None
        filename_valid = False

    if not filename_valid:
        if len(
            invalid_filename_examples
        ) < 10:
            invalid_filename_examples.append(
                {
                    "image_id": image_id,
                    "source_partition": (
                        source_partition
                    ),
                    "filename": filename,
                }
            )
        continue

    if experiment_split == "train":
        S2_TRAIN_IMAGE_PATH_BY_ID[
            image_id
        ] = current_path
    elif experiment_split == "validation":
        S2_VALIDATION_IMAGE_PATH_BY_ID[
            image_id
        ] = current_path
    elif experiment_split == "held_out_test":
        S2_HELD_OUT_TEST_IMAGE_IDS.add(
            image_id
        )
    else:
        raise AssertionError(
            "Unknown experiment split: "
            f"{experiment_split}"
        )

assert not invalid_filename_examples
assert len(image_ids_seen) == (
    EXPECTED_IMAGE_MANIFEST_ROWS
)
assert dict(karpathy_split_counts) == {
    "train": 82_783,
    "restval": 30_504,
    "val": 5_000,
    "test": 5_000,
}
assert dict(experiment_split_counts) == {
    "train": EXPECTED_TRAIN_IMAGES,
    "validation": (
        EXPECTED_VALIDATION_IMAGES
    ),
    "held_out_test": (
        EXPECTED_HELD_OUT_TEST_IMAGES
    ),
}
assert dict(source_partition_counts) == {
    "train2014": 82_783,
    "val2014": 40_504,
}
assert len(
    S2_TRAIN_IMAGE_PATH_BY_ID
) == EXPECTED_TRAIN_IMAGES
assert len(
    S2_VALIDATION_IMAGE_PATH_BY_ID
) == EXPECTED_VALIDATION_IMAGES
assert len(
    S2_HELD_OUT_TEST_IMAGE_IDS
) == EXPECTED_HELD_OUT_TEST_IMAGES

train_image_ids = set(
    S2_TRAIN_IMAGE_PATH_BY_ID
)
validation_image_ids = set(
    S2_VALIDATION_IMAGE_PATH_BY_ID
)
assert not (
    train_image_ids
    & validation_image_ids
)
assert not (
    train_image_ids
    & S2_HELD_OUT_TEST_IMAGE_IDS
)
assert not (
    validation_image_ids
    & S2_HELD_OUT_TEST_IMAGE_IDS
)

train_paths = list(
    S2_TRAIN_IMAGE_PATH_BY_ID.values()
)
validation_paths = list(
    S2_VALIDATION_IMAGE_PATH_BY_ID.values()
)
sample_paths = (
    train_paths[:64]
    + train_paths[-64:]
    + validation_paths[:64]
    + validation_paths[-64:]
)
missing_sample_paths = [
    str(path)
    for path in sample_paths
    if not path.is_file()
]
assert not missing_sample_paths

print(
    "Karpathy splits       :",
    dict(karpathy_split_counts),
)
print(
    "Experiment splits     :",
    dict(experiment_split_counts),
)
print(
    "Source partitions     :",
    dict(source_partition_counts),
)
print(
    "Train image path map  :",
    len(S2_TRAIN_IMAGE_PATH_BY_ID),
)
print(
    "Validation path map   :",
    len(S2_VALIDATION_IMAGE_PATH_BY_ID),
)
print(
    "Held-out Test IDs     :",
    len(S2_HELD_OUT_TEST_IMAGE_IDS),
)
print(
    "Sample paths checked  :",
    len(sample_paths),
)
print(
    "Sample paths missing  :",
    len(missing_sample_paths),
)

del manifest_table
del manifest
del train_paths
del validation_paths


class DevelopmentCaptionIndex:
    REQUIRED_COLUMNS = [
        "annotation_id",
        "image_id",
        "karpathy_split",
        "token_ids",
        "was_truncated",
    ]

    def __init__(
        self,
        parquet_path,
        expected_rows,
        expected_images,
        allowed_splits,
        allowed_image_ids,
        held_out_test_ids,
    ):
        self.parquet_path = Path(
            parquet_path
        )
        self.allowed_splits = set(
            allowed_splits
        )
        self.allowed_image_ids = set(
            allowed_image_ids
        )
        self.held_out_test_ids = set(
            held_out_test_ids
        )
        table = pq.read_table(
            self.parquet_path,
            columns=self.REQUIRED_COLUMNS,
        )
        assert table.num_rows == expected_rows

        self.annotation_ids = (
            table["annotation_id"]
            .combine_chunks()
            .to_numpy(
                zero_copy_only=False
            )
        )
        self.image_ids = (
            table["image_id"]
            .combine_chunks()
            .to_numpy(
                zero_copy_only=False
            )
        )
        self.karpathy_splits = (
            table["karpathy_split"]
            .combine_chunks()
        )
        self.token_ids = (
            table["token_ids"]
            .combine_chunks()
        )
        self.was_truncated = (
            table["was_truncated"]
            .combine_chunks()
            .to_numpy(
                zero_copy_only=False
            )
        )

        split_counts = Counter(
            self.karpathy_splits.to_pylist()
        )
        unique_image_ids = {
            int(image_id)
            for image_id in self.image_ids
        }
        assert set(
            split_counts
        ).issubset(self.allowed_splits)
        assert len(
            unique_image_ids
        ) == expected_images
        assert unique_image_ids == (
            self.allowed_image_ids
        )
        assert not (
            unique_image_ids
            & self.held_out_test_ids
        )
        assert len({
            int(annotation_id)
            for annotation_id
            in self.annotation_ids
        }) == expected_rows

        token_lengths = (
            pc.list_value_length(
                self.token_ids
            )
        )
        token_length_min_max = pc.min_max(
            token_lengths
        ).as_py()
        token_value_min_max = pc.min_max(
            self.token_ids.values
        ).as_py()
        assert token_length_min_max[
            "min"
        ] >= 2
        assert token_length_min_max[
            "max"
        ] <= EXPECTED_MAX_FULL_SEQUENCE
        assert token_value_min_max[
            "min"
        ] >= BOS_ID
        assert token_value_min_max[
            "max"
        ] < EXPECTED_VOCABULARY_SIZE

        self.row_count = int(
            table.num_rows
        )
        self.image_count = len(
            unique_image_ids
        )
        self.split_counts = dict(
            split_counts
        )
        self.truncated_count = int(
            np.asarray(
                self.was_truncated
            ).sum()
        )
        self.token_length_minimum = int(
            token_length_min_max["min"]
        )
        self.token_length_maximum = int(
            token_length_min_max["max"]
        )
        del table

    def __len__(self):
        return self.row_count

    def record(self, row_index):
        row_index = int(row_index)
        token_ids = self.token_ids[
            row_index
        ].as_py()
        assert token_ids[0] == BOS_ID
        assert token_ids[-1] == EOS_ID
        assert PAD_ID not in token_ids
        return {
            "row_index": row_index,
            "annotation_id": int(
                self.annotation_ids[
                    row_index
                ]
            ),
            "image_id": int(
                self.image_ids[row_index]
            ),
            "karpathy_split": (
                self.karpathy_splits[
                    row_index
                ].as_py()
            ),
            "token_ids": token_ids,
            "was_truncated": bool(
                self.was_truncated[
                    row_index
                ]
            ),
        }


print()
print("=" * 72)
print("BUILD DEVELOPMENT CAPTION INDEXES")
print("=" * 72)
print()

S2_TRAIN_CAPTION_INDEX = (
    DevelopmentCaptionIndex(
        parquet_path=TRAIN_PARQUET_PATH,
        expected_rows=(
            EXPECTED_TRAIN_CAPTION_ROWS
        ),
        expected_images=(
            EXPECTED_TRAIN_IMAGES
        ),
        allowed_splits={
            "train",
            "restval",
        },
        allowed_image_ids=(
            train_image_ids
        ),
        held_out_test_ids=(
            S2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )
)
S2_VALIDATION_CAPTION_INDEX = (
    DevelopmentCaptionIndex(
        parquet_path=(
            VALIDATION_PARQUET_PATH
        ),
        expected_rows=(
            EXPECTED_VALIDATION_CAPTION_ROWS
        ),
        expected_images=(
            EXPECTED_VALIDATION_IMAGES
        ),
        allowed_splits={"val"},
        allowed_image_ids=(
            validation_image_ids
        ),
        held_out_test_ids=(
            S2_HELD_OUT_TEST_IMAGE_IDS
        ),
    )
)
annotation_overlap = {
    int(value)
    for value
    in S2_TRAIN_CAPTION_INDEX.annotation_ids
} & {
    int(value)
    for value
    in S2_VALIDATION_CAPTION_INDEX.annotation_ids
}
assert not annotation_overlap

print(
    "Training index       :",
    f"{len(S2_TRAIN_CAPTION_INDEX):,} captions / "
    f"{S2_TRAIN_CAPTION_INDEX.image_count:,} images",
)
print(
    "Training splits      :",
    S2_TRAIN_CAPTION_INDEX.split_counts,
)
print(
    "Validation index     :",
    f"{len(S2_VALIDATION_CAPTION_INDEX):,} captions / "
    f"{S2_VALIDATION_CAPTION_INDEX.image_count:,} images",
)
print(
    "Validation splits    :",
    S2_VALIDATION_CAPTION_INDEX.split_counts,
)
print(
    "Annotation overlap   :",
    len(annotation_overlap),
)


class DeterministicImageBalancedSampler(
    Sampler
):
    def __init__(
        self,
        caption_index,
        base_seed,
        epoch=0,
    ):
        self.caption_index = caption_index
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)
        rows_by_image_id = defaultdict(
            list
        )
        for row_index, image_id in enumerate(
            caption_index.image_ids
        ):
            rows_by_image_id[
                int(image_id)
            ].append(int(row_index))
        self.rows_by_image_id = {
            image_id: tuple(row_indices)
            for image_id, row_indices
            in rows_by_image_id.items()
        }
        self.sorted_image_ids = tuple(
            sorted(self.rows_by_image_id)
        )
        assert len(
            self.sorted_image_ids
        ) == EXPECTED_TRAIN_IMAGES
        assert not (
            set(self.sorted_image_ids)
            & S2_HELD_OUT_TEST_IMAGE_IDS
        )

    def __len__(self):
        return len(self.sorted_image_ids)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def stable_caption_offset(
        self,
        image_id,
        caption_count,
    ):
        mixed = (
            int(image_id) * 1_000_003
            + self.base_seed * 97_409
        )
        return mixed % caption_count

    def selected_rows_before_shuffle(
        self,
        epoch=None,
    ):
        selected_epoch = (
            self.epoch
            if epoch is None
            else int(epoch)
        )
        selected_rows = []
        for image_id in (
            self.sorted_image_ids
        ):
            image_rows = (
                self.rows_by_image_id[
                    image_id
                ]
            )
            offset = (
                self.stable_caption_offset(
                    image_id,
                    len(image_rows),
                )
            )
            selected_rows.append(
                image_rows[
                    (
                        offset
                        + selected_epoch
                    )
                    % len(image_rows)
                ]
            )
        return selected_rows

    def indices_for_epoch(
        self,
        epoch=None,
    ):
        selected_epoch = (
            self.epoch
            if epoch is None
            else int(epoch)
        )
        selected_rows = (
            self.selected_rows_before_shuffle(
                selected_epoch
            )
        )
        generator = torch.Generator()
        generator.manual_seed(
            self.base_seed
            + selected_epoch
        )
        permutation = torch.randperm(
            len(selected_rows),
            generator=generator,
        ).tolist()
        return [
            selected_rows[position]
            for position in permutation
        ]

    def __iter__(self):
        return iter(
            self.indices_for_epoch(
                self.epoch
            )
        )

    def state_dict(self):
        return {
            "base_seed": self.base_seed,
            "epoch": self.epoch,
            "image_count": len(
                self.sorted_image_ids
            ),
        }

    def load_state_dict(self, state):
        assert int(
            state["base_seed"]
        ) == self.base_seed
        assert int(
            state["image_count"]
        ) == len(self.sorted_image_ids)
        self.epoch = int(state["epoch"])


S2_IMAGE_BALANCED_SAMPLER_CLASS = (
    DeterministicImageBalancedSampler
)
S2_TRAIN_SAMPLER = (
    S2_IMAGE_BALANCED_SAMPLER_CLASS(
        caption_index=(
            S2_TRAIN_CAPTION_INDEX
        ),
        base_seed=DEVELOPMENT_SEED,
    )
)

epoch0_first = (
    S2_TRAIN_SAMPLER.indices_for_epoch(
        0
    )
)
epoch0_second = (
    S2_TRAIN_SAMPLER.indices_for_epoch(
        0
    )
)
epoch1 = (
    S2_TRAIN_SAMPLER.indices_for_epoch(
        1
    )
)
assert epoch0_first == epoch0_second
assert len(epoch0_first) == (
    EXPECTED_TRAIN_IMAGES
)
assert len(set(epoch0_first)) == (
    EXPECTED_TRAIN_IMAGES
)
assert len(epoch1) == EXPECTED_TRAIN_IMAGES

epoch0_image_to_row = {
    int(
        S2_TRAIN_CAPTION_INDEX.image_ids[
            row_index
        ]
    ): row_index
    for row_index in epoch0_first
}
epoch1_image_to_row = {
    int(
        S2_TRAIN_CAPTION_INDEX.image_ids[
            row_index
        ]
    ): row_index
    for row_index in epoch1
}
assert set(epoch0_image_to_row) == (
    train_image_ids
)
assert set(epoch1_image_to_row) == (
    train_image_ids
)
multi_caption_images = sum(
    len(rows) > 1
    for rows
    in S2_TRAIN_SAMPLER.rows_by_image_id.values()
)
changed_caption_images = sum(
    epoch0_image_to_row[image_id]
    != epoch1_image_to_row[image_id]
    for image_id in train_image_ids
)
assert changed_caption_images == (
    multi_caption_images
)
caption_count_distribution = dict(
    sorted(
        Counter(
            len(rows)
            for rows
            in S2_TRAIN_SAMPLER.rows_by_image_id.values()
        ).items()
    )
)

print()
print("=" * 72)
print("IMAGE-BALANCED SAMPLER AUDIT")
print("=" * 72)
print(
    "Rows per epoch          :",
    len(epoch0_first),
)
print(
    "Unique images per epoch :",
    len(epoch0_image_to_row),
)
print("Epoch 0 repeatable      : True")
print("Epoch 0/1 membership    : IDENTICAL")
print(
    "Multi-caption images    :",
    multi_caption_images,
)
print(
    "Captions changed        :",
    changed_caption_images,
)
print(
    "Caption count/image     :",
    caption_count_distribution,
)
print("Test overlap            : 0")


S2_CELL03_INDEX_CONTRACT_PATH = (
    RUN_CONFIG_DIR
    / "s2_development_index_and_sampler_contract.json"
)
S2_CELL03_AUDIT_PATH = (
    RUN_AUDIT_DIR
    / "s2_cell03_parquet_indexes_and_sampler_audit.json"
)
cell_elapsed_seconds = (
    time.perf_counter() - CELL_STARTED_AT
)

index_contract = {
    "contract_version": (
        "phase2_s2_development_index_sampler_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "locked",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "inputs": {
        "s2_input_contract": str(
            S2_INPUT_CONTRACT_PATH
        ),
        "s2_input_contract_sha256": (
            S2_INPUT_CONTRACT_SHA256
        ),
        "train_parquet": str(
            TRAIN_PARQUET_PATH
        ),
        "validation_parquet": str(
            VALIDATION_PARQUET_PATH
        ),
        "image_manifest_parquet": str(
            IMAGE_MANIFEST_PARQUET_PATH
        ),
    },
    "indexes": {
        "train_caption_rows": (
            EXPECTED_TRAIN_CAPTION_ROWS
        ),
        "train_images": (
            EXPECTED_TRAIN_IMAGES
        ),
        "validation_caption_rows": (
            EXPECTED_VALIDATION_CAPTION_ROWS
        ),
        "validation_images": (
            EXPECTED_VALIDATION_IMAGES
        ),
        "held_out_test_ids": (
            EXPECTED_HELD_OUT_TEST_IMAGES
        ),
        "annotation_overlap": 0,
        "image_membership_overlap": 0,
    },
    "sampler": {
        "class_name": (
            "DeterministicImageBalancedSampler"
        ),
        "sampling_unit": "image",
        "captions_per_image_per_epoch": 1,
        "rows_per_epoch": (
            EXPECTED_TRAIN_IMAGES
        ),
        "caption_selection": (
            "(stable_image_offset + epoch) modulo caption_count"
        ),
        "shuffle_seed": (
            "base_seed + epoch"
        ),
        "base_seed": DEVELOPMENT_SEED,
        "caption_count_per_image_distribution": (
            caption_count_distribution
        ),
    },
    "policy": {
        "caption_column_read": False,
        "token_ids_read": True,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "training_started": False,
    },
}
write_json(
    S2_CELL03_INDEX_CONTRACT_PATH,
    index_contract,
)
cell_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "student_id": STUDENT_ID,
    "schemas_passed": 3,
    "train_caption_rows": len(
        S2_TRAIN_CAPTION_INDEX
    ),
    "train_images": (
        S2_TRAIN_CAPTION_INDEX.image_count
    ),
    "validation_caption_rows": len(
        S2_VALIDATION_CAPTION_INDEX
    ),
    "validation_images": (
        S2_VALIDATION_CAPTION_INDEX.image_count
    ),
    "held_out_test_ids": len(
        S2_HELD_OUT_TEST_IMAGE_IDS
    ),
    "membership_overlaps": 0,
    "annotation_overlap": 0,
    "sample_image_paths_checked": len(
        sample_paths
    ),
    "sample_image_paths_missing": 0,
    "sampler_rows_per_epoch": len(
        epoch0_first
    ),
    "sampler_repeatable": True,
    "captions_changed_epoch0_to_epoch1": (
        changed_caption_images
    ),
    "caption_column_read": False,
    "token_ids_read": True,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "training_started": False,
    "elapsed_seconds": (
        cell_elapsed_seconds
    ),
}
write_json(
    S2_CELL03_AUDIT_PATH,
    cell_audit,
)
S2_CELL03_INDEX_CONTRACT_SHA256 = (
    sha256_file(
        S2_CELL03_INDEX_CONTRACT_PATH
    )
)
S2_CELL03_AUDIT_SHA256 = sha256_file(
    S2_CELL03_AUDIT_PATH
)


print()
print("=" * 72)
print("PHASE 2 S2 CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Development schemas       : 3 / 3 PASSED")
print("Training caption rows     : 566747")
print("Training unique images    : 113287")
print("Validation caption rows   : 25010")
print("Validation unique images  : 5000")
print("Held-out Test IDs         : 5000")
print("Membership overlaps       : 0")
print("Annotation overlap        : 0")
print("Image path sample         : 256 / 256 PASSED")
print("Image-balanced rows/epoch : 113287")
print("Captions/image/epoch      : 1")
print("Sampler repeatable        : True")
print("Epoch caption rotation    : PASSED")
print("Caption column read       : False")
print("Token IDs read            : True")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print(
    "Index contract            :",
    S2_CELL03_INDEX_CONTRACT_PATH,
)
print(
    "Index contract SHA-256    :",
    S2_CELL03_INDEX_CONTRACT_SHA256,
)
print(
    "Audit                     :",
    S2_CELL03_AUDIT_PATH,
)
print(
    "Audit SHA-256             :",
    S2_CELL03_AUDIT_SHA256,
)
print(
    "Elapsed                   :",
    round(cell_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : rebuild locked S2 preprocessing "
    "and construct deterministic train/validation DataLoaders"
)

## Cell 4 — MobileViT-S Preprocessing và Deterministic DataLoaders

Cell này thực hiện:

1. Đọc locked preprocessing contract của MobileViT-S.
2. Tiền xử lý ảnh RGB kích thước 224×224.
3. Train dùng RandomResizedCrop scale [0.8, 1.0].
4. Train dùng horizontal flip probability 0.5.
5. Validation dùng resize shorter 256 và center crop 224.
6. Sử dụng bicubic interpolation.
7. Giữ tensor trong [0,1] với mean 0 và std 1.
8. Áp dụng stateless augmentation theo sample/epoch.
9. Dựng dynamic right-padding caption collator.
10. Dựng resume-aware Train DataLoader.
11. Dựng deterministic Validation DataLoader.
12. So sánh hai Train loader và hai Validation loader.
13. Yêu cầu image/token/annotation sequences lặp lại chính xác.
14. Không load model hoặc tạo optimizer.
15. Không mở Held-out Test.

In [ ]:
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID", "STUDENT_NAME", "DEVELOPMENT_SEED",
    "RUN_CONFIG_DIR", "RUN_AUDIT_DIR", "SHARED_BUNDLE_ROOT",
    "S2_TRAIN_CAPTION_INDEX", "S2_VALIDATION_CAPTION_INDEX",
    "S2_TRAIN_IMAGE_PATH_BY_ID", "S2_VALIDATION_IMAGE_PATH_BY_ID",
    "S2_HELD_OUT_TEST_IMAGE_IDS", "S2_TRAIN_SAMPLER",
    "S2_CELL03_INDEX_CONTRACT_PATH",
    "S2_CELL03_INDEX_CONTRACT_SHA256",
    "PAD_ID", "BOS_ID", "EOS_ID", "EXPECTED_VOCABULARY_SIZE",
    "write_json", "read_json", "sha256_file", "resolve_inside",
]
missing_runtime_names = [
    name for name in required_runtime_names if name not in globals()
]
assert not missing_runtime_names, (
    f"Thiếu biến từ S2 Cell 1–3: {missing_runtime_names}"
)
assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026

S2_PREPROCESSING_CONTRACT_PATH = resolve_inside(
    SHARED_BUNDLE_ROOT,
    "configs/preprocessing/s2_preprocessing_contract.json",
)
SHARED_TRAINING_CONTRACT_PATH = resolve_inside(
    SHARED_BUNDLE_ROOT,
    "configs/phase2_shared_baseline_training_contract.json",
)
STATELESS_AUGMENTATION_CONTRACT_PATH = resolve_inside(
    SHARED_BUNDLE_ROOT,
    "configs/phase2_stateless_augmentation_contract.json",
)
S2_PREPROCESSING_CONTRACT = read_json(
    S2_PREPROCESSING_CONTRACT_PATH
)
SHARED_TRAINING_CONTRACT = read_json(
    SHARED_TRAINING_CONTRACT_PATH
)
STATELESS_AUGMENTATION_CONTRACT = read_json(
    STATELESS_AUGMENTATION_CONTRACT_PATH
)

assert S2_PREPROCESSING_CONTRACT["student_id"] == "S2"
assert S2_PREPROCESSING_CONTRACT["student_name"] == "MobileViT-S"
assert S2_PREPROCESSING_CONTRACT["library"] == "timm"
assert S2_PREPROCESSING_CONTRACT["model_name"] == (
    "mobilevit_s.cvnets_in1k"
)
assert S2_PREPROCESSING_CONTRACT["input_size"] == 224
assert S2_PREPROCESSING_CONTRACT["policy"][
    "validation_deterministic"
] is True
assert S2_PREPROCESSING_CONTRACT["policy"][
    "test_images_read"
] is False
assert SHARED_TRAINING_CONTRACT["status"] == "locked"
assert SHARED_TRAINING_CONTRACT["scope"]["student_models"][
    "S2"
] == "MobileViT-S"
assert SHARED_TRAINING_CONTRACT["scope"][
    "test_set_status"
] == "sealed"
assert SHARED_TRAINING_CONTRACT["training_data"][
    "images_per_epoch"
] == 113_287
assert SHARED_TRAINING_CONTRACT["training_data"][
    "captions_per_image_per_epoch"
] == 1
assert STATELESS_AUGMENTATION_CONTRACT["status"] == "locked"
assert STATELESS_AUGMENTATION_CONTRACT[
    "independent_of_worker_count"
] is True
assert STATELESS_AUGMENTATION_CONTRACT[
    "supports_mid_epoch_resume"
] is True

geometry = S2_PREPROCESSING_CONTRACT["shared_geometry"]
normalization = S2_PREPROCESSING_CONTRACT["normalization"]
batching = SHARED_TRAINING_CONTRACT["batching"]
S2_INPUT_SIZE = int(S2_PREPROCESSING_CONTRACT["input_size"])
S2_MEAN = tuple(float(value) for value in normalization["mean"])
S2_STD = tuple(float(value) for value in normalization["std"])

assert S2_MEAN == (0.0, 0.0, 0.0)
assert S2_STD == (1.0, 1.0, 1.0)
assert geometry["color_mode"] == "RGB"
assert geometry["interpolation"] == "bicubic"
assert tuple(geometry["train"]["random_resized_crop_scale"]) == (
    0.8, 1.0
)
assert float(geometry["train"]["horizontal_flip_probability"]) == 0.5
assert geometry["validation"]["resize_shorter"] == 256
assert geometry["validation"]["center_crop"] == 224


def create_s2_train_transform():
    train_geometry = geometry["train"]
    return transforms.Compose([
        transforms.RandomResizedCrop(
            size=S2_INPUT_SIZE,
            scale=tuple(
                float(value) for value in
                train_geometry["random_resized_crop_scale"]
            ),
            ratio=tuple(
                float(value) for value in
                train_geometry["random_resized_crop_ratio"]
            ),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        ),
        transforms.RandomHorizontalFlip(
            p=float(train_geometry["horizontal_flip_probability"])
        ),
        transforms.PILToTensor(),
        transforms.ConvertImageDtype(torch.float32),
        transforms.Normalize(mean=S2_MEAN, std=S2_STD),
    ])


def create_s2_validation_transform():
    validation_geometry = geometry["validation"]
    assert validation_geometry["random_operation"] is False
    return transforms.Compose([
        transforms.Resize(
            size=int(validation_geometry["resize_shorter"]),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        ),
        transforms.CenterCrop(
            size=int(validation_geometry["center_crop"])
        ),
        transforms.PILToTensor(),
        transforms.ConvertImageDtype(torch.float32),
        transforms.Normalize(mean=S2_MEAN, std=S2_STD),
    ])


S2_TRAIN_TRANSFORM = create_s2_train_transform()
S2_VALIDATION_TRANSFORM = create_s2_validation_transform()


class ImageCaptionDataset(Dataset):
    def __init__(
        self,
        caption_index,
        image_path_by_id,
        image_transform,
        held_out_test_ids,
    ):
        self.caption_index = caption_index
        self.image_path_by_id = dict(image_path_by_id)
        self.image_transform = image_transform
        self.held_out_test_ids = set(held_out_test_ids)
        assert not (
            set(self.image_path_by_id) & self.held_out_test_ids
        )

    def __len__(self):
        return len(self.caption_index)

    def __getitem__(self, row_index):
        row_index = int(row_index)
        record = self.caption_index.record(row_index)
        image_id = record["image_id"]
        assert image_id not in self.held_out_test_ids
        image_path = self.image_path_by_id.get(image_id)
        if image_path is None:
            raise KeyError(f"Không có image path cho image_id={image_id}")
        with Image.open(image_path) as source_image:
            image_tensor = self.image_transform(
                source_image.convert("RGB")
            )
        token_ids = torch.tensor(record["token_ids"], dtype=torch.long)
        assert image_tensor.dtype == torch.float32
        assert tuple(image_tensor.shape) == (
            3, S2_INPUT_SIZE, S2_INPUT_SIZE
        )
        assert torch.isfinite(image_tensor).all()
        # With mean=0/std=1, tensors must remain in the [0,1] range.
        assert float(image_tensor.min()) >= 0.0
        assert float(image_tensor.max()) <= 1.0
        assert token_ids[0].item() == BOS_ID
        assert token_ids[-1].item() == EOS_ID
        return {
            "row_index": row_index,
            "image": image_tensor,
            "token_ids": token_ids,
            "image_id": image_id,
            "annotation_id": record["annotation_id"],
            "karpathy_split": record["karpathy_split"],
            "was_truncated": record["was_truncated"],
        }


class StatelessEpochAugmentedDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        caption_index,
        base_seed,
        epoch,
    ):
        self.base_dataset = base_dataset
        self.caption_index = caption_index
        self.base_seed = int(base_seed)
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.base_dataset)

    def sample_seed(self, row_index):
        image_id = int(
            self.caption_index.image_ids[int(row_index)]
        )
        mixed = (
            self.base_seed * 1_000_003
            + self.epoch * 97_409
            + image_id * 65_537
            + int(row_index) * 257
        )
        return mixed % (2 ** 63 - 1)

    def __getitem__(self, row_index):
        row_index = int(row_index)
        sample_seed = self.sample_seed(row_index)
        python_state = random.getstate()
        numpy_state = np.random.get_state()
        try:
            random.seed(sample_seed)
            np.random.seed(sample_seed % (2 ** 32))
            with torch.random.fork_rng(devices=[], enabled=True):
                torch.manual_seed(sample_seed)
                sample = self.base_dataset[row_index]
        finally:
            random.setstate(python_state)
            np.random.set_state(numpy_state)
        return sample


class EpochCursorSampler(Sampler):
    def __init__(self, epoch_indices, start_position=0):
        self.epoch_indices = tuple(int(index) for index in epoch_indices)
        self.start_position = int(start_position)
        assert 0 <= self.start_position < len(self.epoch_indices)

    def __len__(self):
        return len(self.epoch_indices) - self.start_position

    def __iter__(self):
        return iter(self.epoch_indices[self.start_position:])


def dynamic_caption_collator(samples):
    assert samples
    images = torch.stack([sample["image"] for sample in samples], dim=0)
    decoder_inputs = [sample["token_ids"][:-1] for sample in samples]
    label_sequences = [sample["token_ids"][1:] for sample in samples]
    decoder_input_ids = pad_sequence(
        decoder_inputs, batch_first=True, padding_value=PAD_ID
    )
    labels = pad_sequence(
        label_sequences, batch_first=True, padding_value=PAD_ID
    )
    decoder_attention_mask = decoder_input_ids != PAD_ID
    sequence_lengths = torch.tensor(
        [sequence.numel() for sequence in decoder_inputs],
        dtype=torch.long,
    )
    assert decoder_input_ids.shape == labels.shape
    assert decoder_input_ids.shape[1] <= 31
    assert torch.all(decoder_input_ids[:, 0] == BOS_ID)
    for row_index, length in enumerate(sequence_lengths.tolist()):
        assert labels[row_index, length - 1].item() == EOS_ID
    return {
        "images": images,
        "decoder_input_ids": decoder_input_ids,
        "decoder_attention_mask": decoder_attention_mask,
        "labels": labels,
        "sequence_lengths": sequence_lengths,
        "row_indices": torch.tensor(
            [sample["row_index"] for sample in samples],
            dtype=torch.long,
        ),
        "image_ids": torch.tensor(
            [sample["image_id"] for sample in samples],
            dtype=torch.long,
        ),
        "annotation_ids": torch.tensor(
            [sample["annotation_id"] for sample in samples],
            dtype=torch.long,
        ),
        "karpathy_splits": [
            sample["karpathy_split"] for sample in samples
        ],
        "was_truncated": torch.tensor(
            [sample["was_truncated"] for sample in samples],
            dtype=torch.bool,
        ),
    }


def seed_dataloader_worker(worker_id):
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


S2_BASE_TRAIN_DATASET = ImageCaptionDataset(
    caption_index=S2_TRAIN_CAPTION_INDEX,
    image_path_by_id=S2_TRAIN_IMAGE_PATH_BY_ID,
    image_transform=S2_TRAIN_TRANSFORM,
    held_out_test_ids=S2_HELD_OUT_TEST_IMAGE_IDS,
)
S2_VALIDATION_DATASET = ImageCaptionDataset(
    caption_index=S2_VALIDATION_CAPTION_INDEX,
    image_path_by_id=S2_VALIDATION_IMAGE_PATH_BY_ID,
    image_transform=S2_VALIDATION_TRANSFORM,
    held_out_test_ids=S2_HELD_OUT_TEST_IMAGE_IDS,
)


def build_s2_train_dataloader(
    epoch,
    start_position=0,
    persistent_workers=None,
):
    epoch = int(epoch)
    start_position = int(start_position)
    physical_batch_size = int(batching["physical_batch_size"])
    accumulation_steps = int(
        batching["gradient_accumulation_steps"]
    )
    optimizer_boundary = physical_batch_size * accumulation_steps
    assert start_position >= 0
    assert start_position < 113_287
    assert (
        start_position == 0
        or start_position % optimizer_boundary == 0
    ), "Resume cursor phải ở optimizer-step boundary."

    epoch_indices = S2_TRAIN_SAMPLER.indices_for_epoch(epoch)
    cursor_sampler = EpochCursorSampler(
        epoch_indices=epoch_indices,
        start_position=start_position,
    )
    stateless_dataset = StatelessEpochAugmentedDataset(
        base_dataset=S2_BASE_TRAIN_DATASET,
        caption_index=S2_TRAIN_CAPTION_INDEX,
        base_seed=DEVELOPMENT_SEED,
        epoch=epoch,
    )
    num_workers = int(batching["num_workers"])
    use_persistent_workers = bool(
        (
            batching["persistent_workers"]
            if persistent_workers is None
            else persistent_workers
        )
        and num_workers > 0
    )
    worker_generator = torch.Generator()
    worker_generator.manual_seed(
        DEVELOPMENT_SEED + epoch + start_position
    )
    loader = DataLoader(
        dataset=stateless_dataset,
        batch_size=physical_batch_size,
        sampler=cursor_sampler,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=bool(batching["pin_memory"]),
        drop_last=bool(batching["drop_last"]),
        collate_fn=dynamic_caption_collator,
        worker_init_fn=seed_dataloader_worker,
        generator=worker_generator,
        persistent_workers=use_persistent_workers,
        prefetch_factor=(2 if num_workers > 0 else None),
    )
    return loader, {
        "epoch": epoch,
        "start_position": start_position,
        "remaining_samples": len(cursor_sampler),
        "physical_batch_size": physical_batch_size,
        "optimizer_boundary_samples": optimizer_boundary,
        "persistent_workers": use_persistent_workers,
    }


def build_s2_validation_dataloader(persistent_workers=None):
    num_workers = int(batching["num_workers"])
    use_persistent_workers = bool(
        (
            batching["persistent_workers"]
            if persistent_workers is None
            else persistent_workers
        )
        and num_workers > 0
    )
    worker_generator = torch.Generator()
    worker_generator.manual_seed(DEVELOPMENT_SEED + 10_000)
    return DataLoader(
        dataset=S2_VALIDATION_DATASET,
        batch_size=int(batching["physical_batch_size"]),
        shuffle=False,
        num_workers=num_workers,
        pin_memory=bool(batching["pin_memory"]),
        drop_last=False,
        collate_fn=dynamic_caption_collator,
        worker_init_fn=seed_dataloader_worker,
        generator=worker_generator,
        persistent_workers=use_persistent_workers,
        prefetch_factor=(2 if num_workers > 0 else None),
    )


def assert_valid_batch(batch, expected_batch_size):
    assert tuple(batch["images"].shape) == (
        expected_batch_size, 3, 224, 224
    )
    assert batch["images"].dtype == torch.float32
    assert torch.isfinite(batch["images"]).all()
    assert float(batch["images"].min()) >= 0.0
    assert float(batch["images"].max()) <= 1.0
    assert batch["decoder_input_ids"].shape == batch["labels"].shape
    assert batch["decoder_attention_mask"].shape == batch["labels"].shape
    assert batch["decoder_input_ids"].dtype == torch.long
    assert batch["labels"].dtype == torch.long
    assert int(batch["decoder_input_ids"].min()) >= PAD_ID
    assert int(batch["decoder_input_ids"].max()) < EXPECTED_VOCABULARY_SIZE
    assert not (
        set(batch["image_ids"].tolist())
        & S2_HELD_OUT_TEST_IMAGE_IDS
    )


def batch_maximum_error(left, right):
    return float(
        (left["images"] - right["images"]).abs().max().item()
    )


print("=" * 72)
print("S2 PREPROCESSING AND DATALOADER SMOKE TEST")
print("=" * 72)
print()

train_loader_a, train_loader_a_info = build_s2_train_dataloader(
    epoch=0, start_position=0, persistent_workers=False
)
train_batch_a = next(iter(train_loader_a))
del train_loader_a
train_loader_b, train_loader_b_info = build_s2_train_dataloader(
    epoch=0, start_position=0, persistent_workers=False
)
train_batch_b = next(iter(train_loader_b))
del train_loader_b
validation_loader_a = build_s2_validation_dataloader(
    persistent_workers=False
)
validation_batch_a = next(iter(validation_loader_a))
del validation_loader_a
validation_loader_b = build_s2_validation_dataloader(
    persistent_workers=False
)
validation_batch_b = next(iter(validation_loader_b))
del validation_loader_b

physical_batch_size = int(batching["physical_batch_size"])
for batch in [
    train_batch_a,
    train_batch_b,
    validation_batch_a,
    validation_batch_b,
]:
    assert_valid_batch(batch, physical_batch_size)

for key in [
    "row_indices",
    "image_ids",
    "annotation_ids",
    "decoder_input_ids",
    "labels",
]:
    assert torch.equal(train_batch_a[key], train_batch_b[key])
for key in [
    "image_ids",
    "annotation_ids",
    "decoder_input_ids",
    "labels",
]:
    assert torch.equal(validation_batch_a[key], validation_batch_b[key])

train_repeat_error = batch_maximum_error(
    train_batch_a, train_batch_b
)
validation_repeat_error = batch_maximum_error(
    validation_batch_a, validation_batch_b
)
assert train_repeat_error == 0.0
assert validation_repeat_error == 0.0

first_row_index = int(train_batch_a["row_indices"][0])
stateless_epoch0 = StatelessEpochAugmentedDataset(
    S2_BASE_TRAIN_DATASET,
    S2_TRAIN_CAPTION_INDEX,
    DEVELOPMENT_SEED,
    0,
)
stateless_epoch1 = StatelessEpochAugmentedDataset(
    S2_BASE_TRAIN_DATASET,
    S2_TRAIN_CAPTION_INDEX,
    DEVELOPMENT_SEED,
    1,
)
assert (
    stateless_epoch0.sample_seed(first_row_index)
    != stateless_epoch1.sample_seed(first_row_index)
)
assert len(set(train_batch_a["image_ids"].tolist())) == physical_batch_size

print("Train image shape       :", list(train_batch_a["images"].shape))
print("Train decoder shape     :", list(
    train_batch_a["decoder_input_ids"].shape
))
print("Train unique images     :", physical_batch_size)
print("Train tensor range      :", (
    float(train_batch_a["images"].min()),
    float(train_batch_a["images"].max()),
))
print("Train repeat error      :", train_repeat_error)
print("Validation image shape  :", list(
    validation_batch_a["images"].shape
))
print("Validation decoder shape:", list(
    validation_batch_a["decoder_input_ids"].shape
))
print("Validation tensor range :", (
    float(validation_batch_a["images"].min()),
    float(validation_batch_a["images"].max()),
))
print("Validation repeat error :", validation_repeat_error)


S2_CELL04_DATALOADER_CONTRACT_PATH = (
    RUN_CONFIG_DIR / "s2_preprocessing_and_dataloader_contract.json"
)
S2_CELL04_AUDIT_PATH = (
    RUN_AUDIT_DIR / "s2_cell04_preprocessing_and_dataloader_audit.json"
)
cell_elapsed_seconds = time.perf_counter() - CELL_STARTED_AT

dataloader_contract = {
    "contract_version": "phase2_s2_preprocessing_dataloader_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "locked",
    "student_id": STUDENT_ID,
    "inputs": {
        "index_contract": str(S2_CELL03_INDEX_CONTRACT_PATH),
        "index_contract_sha256": S2_CELL03_INDEX_CONTRACT_SHA256,
        "preprocessing_contract": str(S2_PREPROCESSING_CONTRACT_PATH),
        "preprocessing_contract_sha256": sha256_file(
            S2_PREPROCESSING_CONTRACT_PATH
        ),
        "training_contract": str(SHARED_TRAINING_CONTRACT_PATH),
        "training_contract_sha256": sha256_file(
            SHARED_TRAINING_CONTRACT_PATH
        ),
        "stateless_contract": str(
            STATELESS_AUGMENTATION_CONTRACT_PATH
        ),
        "stateless_contract_sha256": sha256_file(
            STATELESS_AUGMENTATION_CONTRACT_PATH
        ),
    },
    "preprocessing": {
        "input_shape": [3, 224, 224],
        "color_mode": "RGB",
        "interpolation": "bicubic",
        "mean": list(S2_MEAN),
        "std": list(S2_STD),
        "tensor_range": [0.0, 1.0],
        "train_scale": list(
            geometry["train"]["random_resized_crop_scale"]
        ),
        "train_ratio": list(
            geometry["train"]["random_resized_crop_ratio"]
        ),
        "train_flip_probability": geometry["train"][
            "horizontal_flip_probability"
        ],
        "validation_resize_shorter": geometry["validation"][
            "resize_shorter"
        ],
        "validation_center_crop": geometry["validation"][
            "center_crop"
        ],
    },
    "dataloader": {
        "physical_batch_size": physical_batch_size,
        "gradient_accumulation_steps": int(
            batching["gradient_accumulation_steps"]
        ),
        "effective_batch_size": int(batching["effective_batch_size"]),
        "num_workers": int(batching["num_workers"]),
        "pin_memory": bool(batching["pin_memory"]),
        "production_persistent_workers": bool(
            batching["persistent_workers"]
        ),
        "dynamic_right_padding": True,
        "resume_cursor_unit": (
            "sample position in shuffled epoch index list"
        ),
        "resume_boundary_samples": train_loader_a_info[
            "optimizer_boundary_samples"
        ],
    },
    "policy": {
        "stateless_augmentation": True,
        "caption_column_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "model_loaded": False,
        "optimizer_step_performed": False,
        "training_started": False,
    },
}
write_json(S2_CELL04_DATALOADER_CONTRACT_PATH, dataloader_contract)
cell_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "student_id": STUDENT_ID,
    "train_batch_shape": list(train_batch_a["images"].shape),
    "validation_batch_shape": list(validation_batch_a["images"].shape),
    "train_tensor_minimum": float(train_batch_a["images"].min()),
    "train_tensor_maximum": float(train_batch_a["images"].max()),
    "validation_tensor_minimum": float(
        validation_batch_a["images"].min()
    ),
    "validation_tensor_maximum": float(
        validation_batch_a["images"].max()
    ),
    "train_repeat_error": train_repeat_error,
    "validation_repeat_error": validation_repeat_error,
    "row_sequence_exact": True,
    "annotation_sequence_exact": True,
    "decoder_tensors_exact": True,
    "stateless_epoch_seed_changes": True,
    "caption_column_read": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "model_loaded": False,
    "optimizer_step_performed": False,
    "training_started": False,
    "elapsed_seconds": cell_elapsed_seconds,
}
write_json(S2_CELL04_AUDIT_PATH, cell_audit)
S2_CELL04_DATALOADER_CONTRACT_SHA256 = sha256_file(
    S2_CELL04_DATALOADER_CONTRACT_PATH
)
S2_CELL04_AUDIT_SHA256 = sha256_file(S2_CELL04_AUDIT_PATH)

print()
print("=" * 72)
print("PHASE 2 S2 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Preprocessing contract    : VERIFIED")
print("Input tensor              : RGB [3,224,224] float32")
print("Normalization             : mean 0 / std 1")
print("Tensor range              : [0,1]")
print("Train stateless augment   : True")
print("Validation deterministic  : True")
print("Physical/effective batch  : 32 / 64")
print("Gradient accumulation     : 2")
print("DataLoader workers        : 2")
print("Dynamic right padding     : True")
print("Train batch               : PASSED")
print("Validation batch          : PASSED")
print("Train two-loader exact    : True")
print("Validation two-loader exact: True")
print("Train repeat error        :", train_repeat_error)
print("Validation repeat error   :", validation_repeat_error)
print("Resume cursor supported   : True")
print("Caption column read       : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Model loaded              : False")
print("Optimizer step performed  : False")
print("Training started          : False")
print("Contract                  :", S2_CELL04_DATALOADER_CONTRACT_PATH)
print("Contract SHA-256          :", S2_CELL04_DATALOADER_CONTRACT_SHA256)
print("Audit                     :", S2_CELL04_AUDIT_PATH)
print("Audit SHA-256             :", S2_CELL04_AUDIT_SHA256)
print("Elapsed                   :", round(cell_elapsed_seconds, 2), "giây")
print(
    "Next                      : reconstruct S2 encoder-adapter and "
    "shared decoder, then strict-load published initial checkpoints"
)

## Cell 5 — Dựng MobileViT-S và tải Initial Checkpoints

Cell này thực hiện:

1. Đọc locked S2 encoder và shared decoder contracts.
2. Xác minh checksum của hai initial checkpoints.
3. Dựng MobileViT-S bằng timm với pretrained=False.
4. Không tải model từ Internet.
5. Loại classification head giống cấu trúc tại Phase 2A.
6. Dùng forward_features để lấy feature [B,640,7,7].
7. Dựng adapter 640→256.
8. Thêm 49 learned visual position embeddings và LayerNorm.
9. Dựng shared Transformer decoder giống S1.
10. Strict-load encoder và decoder checkpoints.
11. Kiểm tra parameter counts.
12. Chạy hai deterministic eval forward.
13. Kiểm tra visual memory [2,49,256].
14. Kiểm tra decoder logits [2,T,10080].
15. Đóng băng running statistics của 32 BatchNorm modules.
16. Không tạo optimizer hoặc cập nhật model.
17. Không mở Held-out Test.

In [ ]:
import math
import time
from datetime import datetime, timezone

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID", "STUDENT_NAME", "DEVELOPMENT_SEED", "device",
    "RUN_CONFIG_DIR", "RUN_AUDIT_DIR", "SHARED_BUNDLE_ROOT",
    "required_artifact_paths", "validation_batch_a",
    "S2_CELL04_DATALOADER_CONTRACT_PATH",
    "S2_CELL04_DATALOADER_CONTRACT_SHA256",
    "read_json", "write_json", "sha256_file", "resolve_inside",
]
missing_runtime_names = [
    name for name in required_runtime_names if name not in globals()
]
assert not missing_runtime_names, (
    f"Thiếu biến từ S2 Cell 1–4: {missing_runtime_names}"
)
assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026
assert device.type == "cuda"

S2_ENCODER_CONTRACT_PATH = resolve_inside(
    SHARED_BUNDLE_ROOT,
    "configs/encoders/s2_encoder_adapter_contract.json",
)
SHARED_DECODER_CONTRACT_PATH = resolve_inside(
    SHARED_BUNDLE_ROOT,
    "configs/decoder/shared_transformer_decoder_contract.json",
)
SHARED_TRAINING_CONTRACT_PATH = resolve_inside(
    SHARED_BUNDLE_ROOT,
    "configs/phase2_shared_baseline_training_contract.json",
)
S2_ENCODER_INITIAL_CHECKPOINT_PATH = required_artifact_paths[
    "s2_encoder_checkpoint"
]
SHARED_DECODER_INITIAL_CHECKPOINT_PATH = required_artifact_paths[
    "shared_decoder_checkpoint"
]

S2_ENCODER_CONTRACT = read_json(S2_ENCODER_CONTRACT_PATH)
SHARED_DECODER_CONTRACT = read_json(SHARED_DECODER_CONTRACT_PATH)
SHARED_TRAINING_CONTRACT = read_json(SHARED_TRAINING_CONTRACT_PATH)

assert S2_ENCODER_CONTRACT["contract_version"] == (
    "phase2_encoder_adapter_v1"
)
assert S2_ENCODER_CONTRACT["student_id"] == "S2"
assert S2_ENCODER_CONTRACT["student_name"] == "MobileViT-S"
assert S2_ENCODER_CONTRACT["library"] == "timm"
assert S2_ENCODER_CONTRACT["model_name"] == (
    "mobilevit_s.cvnets_in1k"
)
assert S2_ENCODER_CONTRACT["feature_extraction"]["method"] == (
    "timm_forward_features"
)
assert S2_ENCODER_CONTRACT["feature_extraction"]["raw_layout"] == "BCHW"
assert S2_ENCODER_CONTRACT["feature_extraction"][
    "source_feature_dimension"
] == 640
assert S2_ENCODER_CONTRACT["feature_extraction"]["raw_probe_shape"] == [
    1, 640, 7, 7
]
assert S2_ENCODER_CONTRACT["adapter"]["projection"] == "640->256"
assert S2_ENCODER_CONTRACT["adapter"]["removed_classification_heads"] == [
    "head"
]
assert S2_ENCODER_CONTRACT["output_contract"]["shape"] == [
    "B", 49, 256
]
assert S2_ENCODER_CONTRACT["parameters"]["total"] == 5_114_784

decoder_config = SHARED_DECODER_CONTRACT["architecture"]
assert decoder_config["vocabulary_size"] == 10_080
assert decoder_config["d_model"] == 256
assert decoder_config["nhead"] == 8
assert decoder_config["num_layers"] == 3
assert decoder_config["dim_feedforward"] == 1_024
assert decoder_config["maximum_decoder_steps"] == 31
assert decoder_config["tie_input_output_embeddings"] is True
assert SHARED_DECODER_CONTRACT["parameter_budget"][
    "actual_total_parameters"
] == 5_759_328

training_initialization = SHARED_TRAINING_CONTRACT["initialization"]
assert training_initialization["strict_state_dict_load"] is True
assert training_initialization["one_batch_overfit_weights_reused"] is False
expected_encoder_checkpoint_sha256 = training_initialization[
    "encoder_checkpoints"
]["S2"]["sha256"]
expected_decoder_checkpoint_sha256 = training_initialization[
    "shared_decoder_checkpoint"
]["sha256"]
actual_encoder_checkpoint_sha256 = sha256_file(
    S2_ENCODER_INITIAL_CHECKPOINT_PATH
)
actual_decoder_checkpoint_sha256 = sha256_file(
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH
)
assert actual_encoder_checkpoint_sha256 == (
    expected_encoder_checkpoint_sha256
)
assert actual_decoder_checkpoint_sha256 == (
    expected_decoder_checkpoint_sha256
)


def select_tensor_from_feature_output(feature_output):
    if torch.is_tensor(feature_output):
        return feature_output
    if isinstance(feature_output, dict):
        tensors = [
            value for value in feature_output.values()
            if torch.is_tensor(value)
        ]
        assert tensors
        return tensors[-1]
    if isinstance(feature_output, (tuple, list)):
        tensors = [
            value for value in feature_output if torch.is_tensor(value)
        ]
        assert tensors
        return tensors[-1]
    raise TypeError(
        f"Unsupported feature output: {type(feature_output).__name__}"
    )


def feature_tensor_to_tokens(feature_tensor, layout):
    if layout == "BCHW":
        tokens = feature_tensor.flatten(2).transpose(1, 2)
    elif layout == "BHWC":
        tokens = feature_tensor.flatten(1, 2)
    elif layout == "BNC":
        tokens = feature_tensor
    elif layout == "BCN":
        tokens = feature_tensor.transpose(1, 2)
    elif layout == "BC":
        tokens = feature_tensor.unsqueeze(1)
    else:
        raise ValueError(f"Feature layout không hợp lệ: {layout}")
    assert tokens.ndim == 3
    return tokens.contiguous()


class StudentVisualEncoderAdapter(nn.Module):
    def __init__(
        self,
        backbone,
        source_feature_dimension=640,
        visual_token_count=49,
        visual_memory_dimension=256,
    ):
        super().__init__()
        self.student_id = "S2"
        self.student_name = "MobileViT-S"
        self.backbone = backbone
        self.extraction_method = "timm_forward_features"
        self.feature_layout = "BCHW"
        self.source_feature_dimension = int(source_feature_dimension)
        self.visual_token_count = int(visual_token_count)
        self.visual_memory_dimension = int(visual_memory_dimension)
        self.projection = nn.Linear(
            self.source_feature_dimension,
            self.visual_memory_dimension,
        )
        self.visual_position_embedding = nn.Parameter(
            torch.empty(
                1,
                self.visual_token_count,
                self.visual_memory_dimension,
            )
        )
        self.output_norm = nn.LayerNorm(self.visual_memory_dimension)
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)
        nn.init.normal_(
            self.visual_position_embedding,
            mean=0.0,
            std=0.02,
        )

    def forward(self, images):
        feature_tensor = select_tensor_from_feature_output(
            self.backbone.forward_features(images)
        )
        assert tuple(feature_tensor.shape[1:]) == (640, 7, 7)
        visual_tokens = feature_tensor_to_tokens(
            feature_tensor,
            self.feature_layout,
        )
        assert visual_tokens.shape[1:] == (
            self.visual_token_count,
            self.source_feature_dimension,
        )
        visual_memory = self.projection(visual_tokens)
        visual_memory = visual_memory + self.visual_position_embedding
        visual_memory = self.output_norm(visual_memory)
        assert visual_memory.shape[1:] == (49, 256)
        return visual_memory


class SharedCaptionTransformerDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.vocabulary_size = int(config["vocabulary_size"])
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(
            config["maximum_decoder_steps"]
        )
        assert self.d_model % int(config["nhead"]) == 0
        assert self.d_model // int(config["nhead"]) == int(
            config["head_dimension"]
        )
        assert int(config["memory_dimension"]) == self.d_model

        self.token_embedding = nn.Embedding(
            self.vocabulary_size,
            self.d_model,
            padding_idx=self.pad_id,
        )
        self.position_embedding = nn.Embedding(
            self.maximum_decoder_steps,
            self.d_model,
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation=config["activation"],
            batch_first=bool(config["batch_first"]),
            norm_first=bool(config["norm_first"]),
            bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer=layer,
            num_layers=int(config["num_layers"]),
            norm=None,
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocabulary_size))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=self.d_model ** -0.5,
        )
        nn.init.normal_(
            self.position_embedding.weight,
            mean=0.0,
            std=0.02,
        )
        with torch.no_grad():
            self.token_embedding.weight[self.pad_id].zero_()
            self.output_bias.zero_()

    @staticmethod
    def build_causal_mask(sequence_length, target_device):
        return torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=target_device,
            ),
            diagonal=1,
        )

    def forward(
        self,
        decoder_input_ids,
        visual_memory,
        decoder_attention_mask=None,
        memory_attention_mask=None,
    ):
        assert decoder_input_ids.ndim == 2
        assert visual_memory.ndim == 3
        assert decoder_input_ids.shape[0] == visual_memory.shape[0]
        assert decoder_input_ids.shape[1] <= self.maximum_decoder_steps
        assert visual_memory.shape[-1] == self.d_model
        batch_size, sequence_length = decoder_input_ids.shape
        position_ids = torch.arange(
            sequence_length,
            device=decoder_input_ids.device,
            dtype=torch.long,
        ).unsqueeze(0).expand(batch_size, -1)
        target_embeddings = (
            self.token_embedding(decoder_input_ids)
            * math.sqrt(self.d_model)
            + self.position_embedding(position_ids)
        )
        target_embeddings = self.embedding_dropout(target_embeddings)
        target_key_padding_mask = (
            None
            if decoder_attention_mask is None
            else ~decoder_attention_mask.bool()
        )
        memory_key_padding_mask = (
            None
            if memory_attention_mask is None
            else ~memory_attention_mask.bool()
        )
        hidden_states = self.transformer_decoder(
            tgt=target_embeddings,
            memory=visual_memory,
            tgt_mask=self.build_causal_mask(
                sequence_length,
                decoder_input_ids.device,
            ),
            tgt_key_padding_mask=target_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )
        hidden_states = self.final_norm(hidden_states)
        logits = F.linear(
            hidden_states,
            self.token_embedding.weight,
            self.output_bias,
        )
        assert logits.shape == (
            batch_size,
            sequence_length,
            self.vocabulary_size,
        )
        return logits


def count_unique_parameters(model):
    seen = set()
    total = 0
    trainable = 0
    for parameter in model.parameters():
        identity = id(parameter)
        if identity in seen:
            continue
        seen.add(identity)
        total += parameter.numel()
        if parameter.requires_grad:
            trainable += parameter.numel()
    return int(total), int(trainable)


def set_s2_training_mode(encoder_model, decoder_model):
    encoder_model.train()
    decoder_model.train()
    frozen_batchnorm_count = 0
    for module in encoder_model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            frozen_batchnorm_count += 1
    return frozen_batchnorm_count


print("=" * 72)
print("RECONSTRUCT S2 MODEL ARCHITECTURE")
print("=" * 72)
print()

# No download: published checkpoint already contains pretrained backbone.
mobilevit_backbone = timm.create_model(
    "mobilevit_s.cvnets_in1k",
    pretrained=False,
)
if hasattr(mobilevit_backbone, "reset_classifier"):
    mobilevit_backbone.reset_classifier(
        num_classes=0,
        global_pool="",
    )
removed_head_modules = []
for head_name in ["head", "head_dist", "classifier", "fc"]:
    head_module = getattr(mobilevit_backbone, head_name, None)
    if (
        isinstance(head_module, nn.Module)
        and not isinstance(head_module, nn.Identity)
    ):
        setattr(mobilevit_backbone, head_name, nn.Identity())
        removed_head_modules.append(head_name)
assert removed_head_modules == ["head"]

S2_ENCODER_ADAPTER_MODEL = StudentVisualEncoderAdapter(
    backbone=mobilevit_backbone,
    source_feature_dimension=640,
    visual_token_count=49,
    visual_memory_dimension=256,
)
S2_SHARED_DECODER_MODEL = SharedCaptionTransformerDecoder(decoder_config)

encoder_total, encoder_trainable = count_unique_parameters(
    S2_ENCODER_ADAPTER_MODEL
)
decoder_total, decoder_trainable = count_unique_parameters(
    S2_SHARED_DECODER_MODEL
)
assert encoder_total == 5_114_784
assert encoder_trainable == 5_114_784
assert decoder_total == 5_759_328
assert decoder_trainable == 5_759_328
assert encoder_total + decoder_total == 10_874_112

print("Encoder checkpoint :", S2_ENCODER_INITIAL_CHECKPOINT_PATH)
print("Decoder checkpoint :", SHARED_DECODER_INITIAL_CHECKPOINT_PATH)

with torch.serialization.safe_globals([TorchVersion]):
    encoder_checkpoint = torch.load(
        S2_ENCODER_INITIAL_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )
    decoder_checkpoint = torch.load(
        SHARED_DECODER_INITIAL_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

assert encoder_checkpoint["student_id"] == "S2"
assert encoder_checkpoint["student_name"] == "MobileViT-S"
assert int(encoder_checkpoint["seed"]) == 42
assert int(decoder_checkpoint["seed"]) == 42
assert decoder_checkpoint["config"] == decoder_config

encoder_load_result = S2_ENCODER_ADAPTER_MODEL.load_state_dict(
    encoder_checkpoint["state_dict"],
    strict=True,
)
decoder_load_result = S2_SHARED_DECODER_MODEL.load_state_dict(
    decoder_checkpoint["state_dict"],
    strict=True,
)
assert not encoder_load_result.missing_keys
assert not encoder_load_result.unexpected_keys
assert not decoder_load_result.missing_keys
assert not decoder_load_result.unexpected_keys
del encoder_checkpoint
del decoder_checkpoint

assert not hasattr(S2_SHARED_DECODER_MODEL, "output_projection")
assert "token_embedding.weight" in S2_SHARED_DECODER_MODEL.state_dict()

S2_ENCODER_ADAPTER_MODEL = S2_ENCODER_ADAPTER_MODEL.to(device)
S2_SHARED_DECODER_MODEL = S2_SHARED_DECODER_MODEL.to(device)

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

S2_ENCODER_ADAPTER_MODEL.eval()
S2_SHARED_DECODER_MODEL.eval()
smoke_images = validation_batch_a["images"][:2].to(
    device,
    non_blocking=True,
)
smoke_decoder_input_ids = validation_batch_a[
    "decoder_input_ids"
][:2].to(device, non_blocking=True)
smoke_decoder_attention_mask = validation_batch_a[
    "decoder_attention_mask"
][:2].to(device, non_blocking=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)
with torch.inference_mode():
    visual_memory_first = S2_ENCODER_ADAPTER_MODEL(smoke_images)
    logits_first = S2_SHARED_DECODER_MODEL(
        decoder_input_ids=smoke_decoder_input_ids,
        visual_memory=visual_memory_first,
        decoder_attention_mask=smoke_decoder_attention_mask,
    )
    visual_memory_second = S2_ENCODER_ADAPTER_MODEL(smoke_images)
    logits_second = S2_SHARED_DECODER_MODEL(
        decoder_input_ids=smoke_decoder_input_ids,
        visual_memory=visual_memory_second,
        decoder_attention_mask=smoke_decoder_attention_mask,
    )
torch.cuda.synchronize(device)
peak_gpu_memory_bytes = int(torch.cuda.max_memory_allocated(device))

assert tuple(visual_memory_first.shape) == (2, 49, 256)
assert tuple(logits_first.shape) == (
    2,
    smoke_decoder_input_ids.shape[1],
    10_080,
)
assert torch.isfinite(visual_memory_first).all()
assert torch.isfinite(logits_first).all()
memory_repeat_error = float(
    (visual_memory_first - visual_memory_second).abs().max().item()
)
logits_repeat_error = float(
    (logits_first - logits_second).abs().max().item()
)
assert memory_repeat_error == 0.0
assert logits_repeat_error == 0.0

frozen_batchnorm_count = set_s2_training_mode(
    S2_ENCODER_ADAPTER_MODEL,
    S2_SHARED_DECODER_MODEL,
)
assert frozen_batchnorm_count == 32
assert S2_SHARED_DECODER_MODEL.training is True
assert all(
    not module.training
    for module in S2_ENCODER_ADAPTER_MODEL.modules()
    if isinstance(module, nn.modules.batchnorm._BatchNorm)
)
S2_ENCODER_ADAPTER_MODEL.eval()
S2_SHARED_DECODER_MODEL.eval()

print()
print("=" * 72)
print("S2 MODEL AND CHECKPOINT LOAD")
print("=" * 72)
print()
print("Encoder                  : MobileViT-S forward_features")
print("Adapter                  : 640 -> 256 + position + LN")
print("Decoder                  : Shared Transformer Decoder")
print("Removed heads            :", removed_head_modules)
print("Encoder parameters       :", f"{encoder_total:,}")
print("Decoder parameters       :", f"{decoder_total:,}")
print("End-to-end parameters    :", f"{encoder_total + decoder_total:,}")
print("Encoder strict load      : PASSED")
print("Decoder strict load      : PASSED")
print("Network download         : False")
print("Visual memory shape      :", list(visual_memory_first.shape))
print("Logits shape             :", list(logits_first.shape))
print("Memory repeat error      :", memory_repeat_error)
print("Logits repeat error      :", logits_repeat_error)
print("Frozen BatchNorm modules :", frozen_batchnorm_count)
print(
    "Peak GPU memory          :",
    round(peak_gpu_memory_bytes / (1024 ** 2), 2),
    "MiB",
)

S2_CELL05_MODEL_CONTRACT_PATH = (
    RUN_CONFIG_DIR / "s2_loaded_model_contract.json"
)
S2_CELL05_AUDIT_PATH = (
    RUN_AUDIT_DIR / "s2_cell05_model_checkpoint_load_audit.json"
)
cell_elapsed_seconds = time.perf_counter() - CELL_STARTED_AT

model_contract = {
    "contract_version": "phase2_s2_loaded_model_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "locked",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "encoder": {
        "model_name": "mobilevit_s.cvnets_in1k",
        "library": "timm",
        "extraction_method": "timm_forward_features",
        "removed_heads": removed_head_modules,
        "raw_feature_shape": [2, 640, 7, 7],
        "projection": "640->256",
        "visual_memory_shape": [2, 49, 256],
        "parameters": encoder_total,
        "checkpoint": str(S2_ENCODER_INITIAL_CHECKPOINT_PATH),
        "checkpoint_sha256": actual_encoder_checkpoint_sha256,
        "strict_load": True,
    },
    "decoder": {
        "parameters": decoder_total,
        "checkpoint": str(SHARED_DECODER_INITIAL_CHECKPOINT_PATH),
        "checkpoint_sha256": actual_decoder_checkpoint_sha256,
        "strict_load": True,
        "tied_input_output_embeddings": True,
    },
    "end_to_end_parameters": encoder_total + decoder_total,
    "batchnorm_running_statistics_frozen": True,
    "frozen_batchnorm_modules": frozen_batchnorm_count,
    "policy": {
        "pretrained_download": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "optimizer_created": False,
        "optimizer_step_performed": False,
        "training_started": False,
    },
}
write_json(S2_CELL05_MODEL_CONTRACT_PATH, model_contract)
cell_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "student_id": STUDENT_ID,
    "encoder_checkpoint_sha_verified": True,
    "decoder_checkpoint_sha_verified": True,
    "encoder_strict_load": True,
    "decoder_strict_load": True,
    "missing_keys": 0,
    "unexpected_keys": 0,
    "encoder_parameters": encoder_total,
    "decoder_parameters": decoder_total,
    "end_to_end_parameters": encoder_total + decoder_total,
    "visual_memory_shape": list(visual_memory_first.shape),
    "logits_shape": list(logits_first.shape),
    "memory_repeat_error": memory_repeat_error,
    "logits_repeat_error": logits_repeat_error,
    "frozen_batchnorm_modules": frozen_batchnorm_count,
    "peak_gpu_memory_bytes": peak_gpu_memory_bytes,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "optimizer_created": False,
    "optimizer_step_performed": False,
    "training_started": False,
    "elapsed_seconds": cell_elapsed_seconds,
}
write_json(S2_CELL05_AUDIT_PATH, cell_audit)
S2_CELL05_MODEL_CONTRACT_SHA256 = sha256_file(
    S2_CELL05_MODEL_CONTRACT_PATH
)
S2_CELL05_AUDIT_SHA256 = sha256_file(S2_CELL05_AUDIT_PATH)

print()
print("=" * 72)
print("PHASE 2 S2 CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Encoder architecture      : VERIFIED")
print("Decoder architecture      : VERIFIED")
print("Encoder checkpoint SHA    : VERIFIED")
print("Decoder checkpoint SHA    : VERIFIED")
print("Encoder strict load       : PASSED")
print("Decoder strict load       : PASSED")
print("Missing/unexpected keys   : 0 / 0")
print("Raw feature map           : [B,640,7,7]")
print("Visual memory             : [2,49,256]")
print("Decoder logits            : [2,T,10080]")
print("Forward tensors finite    : True")
print("Eval repeatability        : PASSED")
print("Memory repeat error       :", memory_repeat_error)
print("Logits repeat error       :", logits_repeat_error)
print("BatchNorm freeze policy   : 32 modules VERIFIED")
print("Pretrained download       : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Optimizer created         : False")
print("Optimizer step performed  : False")
print("Training started          : False")
print("Model contract            :", S2_CELL05_MODEL_CONTRACT_PATH)
print("Model contract SHA-256    :", S2_CELL05_MODEL_CONTRACT_SHA256)
print("Audit                     :", S2_CELL05_AUDIT_PATH)
print("Audit SHA-256             :", S2_CELL05_AUDIT_SHA256)
print("Elapsed                   :", round(cell_elapsed_seconds, 2), "giây")
print(
    "Next                      : build AdamW parameter groups, "
    "warmup-cosine scheduler and FP16 GradScaler, without stepping"
)

## Cell 6 — Khởi tạo Optimizer, Scheduler và FP16 GradScaler cho S2

Cell này thực hiện:

- Chia tham số thành ba nhóm không trùng lặp:
  - MobileViT-S backbone;
  - visual adapter;
  - shared Transformer decoder.
- Khởi tạo AdamW với learning rate riêng cho từng nhóm.
- Khởi tạo linear warmup 5% kết hợp cosine decay.
- Khởi tạo FP16 GradScaler.
- Kiểm tra toàn bộ trainable parameters được optimizer quản lý.
- Kiểm tra optimizer chưa có state huấn luyện.
- Ghi contract và audit kèm SHA-256.
- Không chạy forward, backward hoặc optimizer step.
- Không truy cập Held-out Test Set.

Cell chỉ được xem là đạt khi:

- Có đúng 3 parameter groups.
- Không có parameter trùng lặp hoặc bị bỏ sót.
- Tổng parameter coverage là 10.874.112.
- Scheduler có 35.420 bước, trong đó 1.771 bước warmup.
- GradScaler được bật với initial scale 1024.
- Test Set vẫn ở trạng thái SEALED.

In [ ]:
import math
import time
from datetime import datetime, timezone

import torch
from torch.optim.lr_scheduler import LambdaLR


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID",
    "STUDENT_NAME",
    "DEVELOPMENT_SEED",
    "RUN_CONFIG_DIR",
    "RUN_AUDIT_DIR",
    "SHARED_TRAINING_CONTRACT",
    "SHARED_TRAINING_CONTRACT_PATH",
    "S2_ENCODER_ADAPTER_MODEL",
    "S2_SHARED_DECODER_MODEL",
    "S2_CELL05_MODEL_CONTRACT_PATH",
    "S2_CELL05_MODEL_CONTRACT_SHA256",
    "write_json",
    "sha256_file",
]
missing_runtime_names = [
    name for name in required_runtime_names if name not in globals()
]
assert not missing_runtime_names, (
    "Thiếu biến từ S2 Cell 1–5: "
    f"{missing_runtime_names}. Hãy chạy lại các cell trước."
)

assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026
assert SHARED_TRAINING_CONTRACT["status"] == "locked"

optimization_contract = SHARED_TRAINING_CONTRACT["optimization"]
batching_contract = SHARED_TRAINING_CONTRACT["batching"]
duration_contract = SHARED_TRAINING_CONTRACT["training_duration"]
mixed_precision_contract = optimization_contract["mixed_precision"]
scheduler_contract = optimization_contract["scheduler"]

assert optimization_contract["optimizer"] == "AdamW"
assert batching_contract["physical_batch_size"] == 32
assert batching_contract["gradient_accumulation_steps"] == 2
assert batching_contract["effective_batch_size"] == 64
assert batching_contract["estimated_optimizer_steps_per_epoch"] == 1_771
assert duration_contract["maximum_epochs"] == 20
assert mixed_precision_contract["enabled"] is True
assert mixed_precision_contract["dtype"] == "float16"
assert mixed_precision_contract["gradient_scaler"] is True


def build_s2_optimizer(encoder, decoder):
    """Create three disjoint parameter groups from the locked contract."""
    group_contracts = optimization_contract["parameter_groups"]

    backbone_parameters = [
        parameter
        for parameter in encoder.backbone.parameters()
        if parameter.requires_grad
    ]
    adapter_parameters = [
        parameter
        for module in [encoder.projection, encoder.output_norm]
        for parameter in module.parameters()
        if parameter.requires_grad
    ]
    if encoder.visual_position_embedding.requires_grad:
        adapter_parameters.append(encoder.visual_position_embedding)
    decoder_parameters = [
        parameter
        for parameter in decoder.parameters()
        if parameter.requires_grad
    ]

    named_groups = [
        ("backbone", backbone_parameters),
        ("adapter", adapter_parameters),
        ("decoder", decoder_parameters),
    ]
    grouped_parameter_ids = [
        id(parameter)
        for _, parameters in named_groups
        for parameter in parameters
    ]
    expected_parameter_ids = {
        id(parameter)
        for parameter in list(encoder.parameters()) + list(decoder.parameters())
        if parameter.requires_grad
    }

    assert len(grouped_parameter_ids) == len(set(grouped_parameter_ids)), (
        "Một parameter xuất hiện trong nhiều optimizer groups."
    )
    assert set(grouped_parameter_ids) == expected_parameter_ids, (
        "Optimizer groups không phủ chính xác trainable parameters."
    )

    optimizer_groups = [
        {
            "name": group_name,
            "params": parameters,
            "lr": float(group_contracts[group_name]["learning_rate"]),
        }
        for group_name, parameters in named_groups
    ]
    optimizer = torch.optim.AdamW(
        optimizer_groups,
        betas=tuple(float(value) for value in optimization_contract["betas"]),
        eps=float(optimization_contract["epsilon"]),
        weight_decay=float(optimization_contract["weight_decay"]),
    )
    group_audit = {
        group_name: {
            "parameter_tensors": len(parameters),
            "parameter_elements": sum(
                parameter.numel() for parameter in parameters
            ),
            "base_learning_rate": float(
                group_contracts[group_name]["learning_rate"]
            ),
        }
        for group_name, parameters in named_groups
    }
    return optimizer, group_audit


TOTAL_OPTIMIZER_STEPS = (
    int(batching_contract["estimated_optimizer_steps_per_epoch"])
    * int(duration_contract["maximum_epochs"])
)
WARMUP_OPTIMIZER_STEPS = max(
    1,
    int(
        round(
            TOTAL_OPTIMIZER_STEPS
            * float(scheduler_contract["warmup_fraction_of_optimizer_steps"])
        )
    ),
)
MINIMUM_LR_FACTOR = float(
    scheduler_contract["minimum_learning_rate_factor"]
)

assert TOTAL_OPTIMIZER_STEPS == 35_420
assert WARMUP_OPTIMIZER_STEPS == 1_771
assert MINIMUM_LR_FACTOR == 0.05


def learning_rate_multiplier(step):
    """Linear warmup followed by cosine decay to the locked floor."""
    step = int(step)
    if step < WARMUP_OPTIMIZER_STEPS:
        return max(
            (step + 1) / WARMUP_OPTIMIZER_STEPS,
            1.0 / WARMUP_OPTIMIZER_STEPS,
        )

    decay_denominator = max(
        1, TOTAL_OPTIMIZER_STEPS - WARMUP_OPTIMIZER_STEPS
    )
    progress = min(
        1.0,
        (step - WARMUP_OPTIMIZER_STEPS) / decay_denominator,
    )
    cosine_factor = 0.5 * (1.0 + math.cos(math.pi * progress))
    return (
        MINIMUM_LR_FACTOR
        + (1.0 - MINIMUM_LR_FACTOR) * cosine_factor
    )


def build_s2_scheduler(optimizer):
    return LambdaLR(optimizer, lr_lambda=learning_rate_multiplier)


def build_s2_grad_scaler():
    return torch.amp.GradScaler(
        device="cuda",
        init_scale=float(mixed_precision_contract["initial_scale"]),
        growth_factor=float(mixed_precision_contract["growth_factor"]),
        backoff_factor=float(mixed_precision_contract["backoff_factor"]),
        growth_interval=int(mixed_precision_contract["growth_interval"]),
        enabled=True,
    )


print("=" * 72)
print("S2 OPTIMIZER PARAMETER-GROUP AUDIT")
print("=" * 72)
print()

all_trainable_parameters = (
    list(S2_ENCODER_ADAPTER_MODEL.parameters())
    + list(S2_SHARED_DECODER_MODEL.parameters())
)
assert all(parameter.grad is None for parameter in all_trainable_parameters)

S2_OPTIMIZER, S2_PARAMETER_GROUP_AUDIT = build_s2_optimizer(
    S2_ENCODER_ADAPTER_MODEL,
    S2_SHARED_DECODER_MODEL,
)
assert len(S2_OPTIMIZER.param_groups) == 3
assert len(S2_OPTIMIZER.state) == 0

expected_group_elements = {
    "backbone": 4_937_632,
    "adapter": 177_152,
    "decoder": 5_759_328,
}
expected_base_lrs = {
    "backbone": 0.0001,
    "adapter": 0.0003,
    "decoder": 0.0003,
}
for group_name in ["backbone", "adapter", "decoder"]:
    group_audit = S2_PARAMETER_GROUP_AUDIT[group_name]
    assert (
        group_audit["parameter_elements"]
        == expected_group_elements[group_name]
    )
    assert math.isclose(
        group_audit["base_learning_rate"],
        expected_base_lrs[group_name],
        rel_tol=0.0,
        abs_tol=1e-15,
    )

assert sum(
    group["parameter_elements"]
    for group in S2_PARAMETER_GROUP_AUDIT.values()
) == 10_874_112
for name, group in zip(
    ["backbone", "adapter", "decoder"], S2_OPTIMIZER.param_groups
):
    assert group["name"] == name

for label, group_name in [
    ("Backbone", "backbone"),
    ("Adapter ", "adapter"),
    ("Decoder ", "decoder"),
]:
    print(
        f"{label} :",
        f"{S2_PARAMETER_GROUP_AUDIT[group_name]['parameter_elements']:,}",
        f"parameters | LR={expected_base_lrs[group_name]}",
    )
print("Duplicate parameters : 0")
print("Unassigned parameters: 0")
print("Optimizer state      : EMPTY")


S2_SCHEDULER = build_s2_scheduler(S2_OPTIMIZER)
S2_GRAD_SCALER = build_s2_grad_scaler()

schedule_probe_steps = [
    0,
    WARMUP_OPTIMIZER_STEPS - 1,
    WARMUP_OPTIMIZER_STEPS,
    TOTAL_OPTIMIZER_STEPS // 2,
    TOTAL_OPTIMIZER_STEPS,
]
S2_SCHEDULE_PROBES = {
    str(step): learning_rate_multiplier(step)
    for step in schedule_probe_steps
}
assert 0.0 < S2_SCHEDULE_PROBES["0"] < 1.0
assert math.isclose(
    S2_SCHEDULE_PROBES[str(WARMUP_OPTIMIZER_STEPS - 1)],
    1.0,
    rel_tol=0.0,
    abs_tol=1e-15,
)
assert math.isclose(
    S2_SCHEDULE_PROBES[str(WARMUP_OPTIMIZER_STEPS)],
    1.0,
    rel_tol=0.0,
    abs_tol=1e-15,
)
assert math.isclose(
    S2_SCHEDULE_PROBES[str(TOTAL_OPTIMIZER_STEPS)],
    MINIMUM_LR_FACTOR,
    rel_tol=0.0,
    abs_tol=1e-15,
)

initial_multiplier = learning_rate_multiplier(0)
for group_name, optimizer_group in zip(
    ["backbone", "adapter", "decoder"], S2_OPTIMIZER.param_groups
):
    assert math.isclose(
        float(optimizer_group["lr"]),
        expected_base_lrs[group_name] * initial_multiplier,
        rel_tol=1e-12,
        abs_tol=1e-18,
    )

assert S2_GRAD_SCALER.is_enabled()
assert math.isclose(
    float(S2_GRAD_SCALER.get_scale()),
    1_024.0,
    rel_tol=0.0,
    abs_tol=0.0,
)
assert len(S2_OPTIMIZER.state) == 0
assert all(parameter.grad is None for parameter in all_trainable_parameters)

# Initial progress state consumed by the atomic-resume cell.
S2_OPTIMIZER_STEP = 0
S2_MICROBATCH_IN_ACCUMULATION = 0
S2_GLOBAL_SAMPLE_CURSOR = 0
S2_CURRENT_EPOCH = 0
S2_OVERFLOW_RETRY_COUNT = 0
S2_SCHEDULER_INFO = {
    "total_optimizer_steps": TOTAL_OPTIMIZER_STEPS,
    "warmup_optimizer_steps": WARMUP_OPTIMIZER_STEPS,
    "minimum_learning_rate_factor": MINIMUM_LR_FACTOR,
    "step_unit": "optimizer_step",
}

S2_CELL06_OPTIMIZATION_CONTRACT_PATH = (
    RUN_CONFIG_DIR / "s2_optimizer_scheduler_scaler_contract.json"
)
S2_CELL06_AUDIT_PATH = (
    RUN_AUDIT_DIR / "s2_cell06_optimizer_scheduler_scaler_audit.json"
)
cell_elapsed_seconds = time.perf_counter() - CELL_STARTED_AT

optimization_payload = {
    "contract_version": "phase2_s2_optimizer_scheduler_scaler_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "locked",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "inputs": {
        "shared_training_contract": str(SHARED_TRAINING_CONTRACT_PATH),
        "shared_training_contract_sha256": sha256_file(
            SHARED_TRAINING_CONTRACT_PATH
        ),
        "loaded_model_contract": str(S2_CELL05_MODEL_CONTRACT_PATH),
        "loaded_model_contract_sha256": S2_CELL05_MODEL_CONTRACT_SHA256,
    },
    "optimizer": {
        "name": "AdamW",
        "parameter_groups": S2_PARAMETER_GROUP_AUDIT,
        "betas": list(optimization_contract["betas"]),
        "epsilon": float(optimization_contract["epsilon"]),
        "weight_decay": float(optimization_contract["weight_decay"]),
        "gradient_clip_norm": float(
            optimization_contract["gradient_clip_norm"]
        ),
        "state_initially_empty": True,
    },
    "scheduler": {
        **S2_SCHEDULER_INFO,
        "name": "linear warmup plus cosine decay",
        "schedule_probes": S2_SCHEDULE_PROBES,
        "initial_multiplier": initial_multiplier,
    },
    "mixed_precision": {
        "enabled": True,
        "dtype": "float16",
        "loss_computed_in_float32": True,
        "grad_scaler_initial_scale": float(S2_GRAD_SCALER.get_scale()),
        "growth_factor": float(
            mixed_precision_contract["growth_factor"]
        ),
        "backoff_factor": float(
            mixed_precision_contract["backoff_factor"]
        ),
        "growth_interval": int(
            mixed_precision_contract["growth_interval"]
        ),
        "maximum_overflow_retries": int(
            mixed_precision_contract["maximum_overflow_retries_per_run"]
        ),
    },
    "initial_progress": {
        "epoch": S2_CURRENT_EPOCH,
        "optimizer_step": S2_OPTIMIZER_STEP,
        "global_sample_cursor": S2_GLOBAL_SAMPLE_CURSOR,
        "microbatch_in_accumulation": S2_MICROBATCH_IN_ACCUMULATION,
        "overflow_retry_count": S2_OVERFLOW_RETRY_COUNT,
    },
    "policy": {
        "parameter_coverage_exact": True,
        "duplicate_parameters": 0,
        "unassigned_parameters": 0,
        "forward_performed": False,
        "backward_performed": False,
        "optimizer_step_performed": False,
        "scheduler_initialization_applied": True,
        "scheduler_advance_step_performed": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "training_started": False,
    },
}
write_json(S2_CELL06_OPTIMIZATION_CONTRACT_PATH, optimization_payload)

cell_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "student_id": STUDENT_ID,
    "optimizer": "AdamW",
    "optimizer_parameter_groups": 3,
    "optimizer_parameter_elements": 10_874_112,
    "duplicate_parameters": 0,
    "unassigned_parameters": 0,
    "optimizer_state_entries": len(S2_OPTIMIZER.state),
    "optimizer_steps_per_epoch": 1_771,
    "maximum_epochs": 20,
    "total_optimizer_steps": TOTAL_OPTIMIZER_STEPS,
    "warmup_optimizer_steps": WARMUP_OPTIMIZER_STEPS,
    "minimum_lr_factor": MINIMUM_LR_FACTOR,
    "grad_scaler_enabled": S2_GRAD_SCALER.is_enabled(),
    "grad_scaler_initial_scale": float(S2_GRAD_SCALER.get_scale()),
    "forward_performed": False,
    "backward_performed": False,
    "optimizer_step_performed": False,
    "scheduler_initialization_applied": True,
    "scheduler_advance_step_performed": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "training_started": False,
    "elapsed_seconds": cell_elapsed_seconds,
}
write_json(S2_CELL06_AUDIT_PATH, cell_audit)

S2_CELL06_OPTIMIZATION_CONTRACT_SHA256 = sha256_file(
    S2_CELL06_OPTIMIZATION_CONTRACT_PATH
)
S2_CELL06_AUDIT_SHA256 = sha256_file(S2_CELL06_AUDIT_PATH)

print()
print("=" * 72)
print("S2 SCHEDULER AND GRADSCALER AUDIT")
print("=" * 72)
print("Optimizer steps/epoch   : 1771")
print("Maximum epochs          : 20")
print("Total optimizer steps   :", TOTAL_OPTIMIZER_STEPS)
print("Warmup optimizer steps  :", WARMUP_OPTIMIZER_STEPS)
print("Minimum LR factor       :", MINIMUM_LR_FACTOR)
print("Schedule probes         :", S2_SCHEDULE_PROBES)
print("GradScaler enabled      :", S2_GRAD_SCALER.is_enabled())
print("GradScaler initial scale:", S2_GRAD_SCALER.get_scale())

print()
print("=" * 72)
print("PHASE 2 S2 CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Optimizer                 : AdamW")
print("Parameter groups          : 3 / 3 PASSED")
print("Backbone LR               : 0.0001")
print("Adapter LR                : 0.0003")
print("Decoder LR                : 0.0003")
print("Parameter coverage        : 10874112 / 10874112")
print("Duplicate parameters      : 0")
print("Unassigned parameters     : 0")
print("Betas / epsilon           : [0.9,0.999] / 1e-08")
print("Weight decay              : 0.0001")
print("Gradient clip norm        : 1.0")
print("Optimizer steps/epoch     : 1771")
print("Total optimizer steps     : 35420")
print("Warmup optimizer steps    : 1771")
print("Minimum LR factor         : 0.05")
print("Mixed precision           : FP16")
print("GradScaler initial scale  : 1024.0")
print("Optimizer state           : EMPTY")
print("Forward performed         : False")
print("Backward performed        : False")
print("Optimizer step performed  : False")
print("Scheduler initialized     : True")
print("Scheduler advanced        : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Training started          : False")
print("Contract                  :", S2_CELL06_OPTIMIZATION_CONTRACT_PATH)
print("Contract SHA-256          :", S2_CELL06_OPTIMIZATION_CONTRACT_SHA256)
print("Audit                     :", S2_CELL06_AUDIT_PATH)
print("Audit SHA-256             :", S2_CELL06_AUDIT_SHA256)
print("Elapsed                   :", round(cell_elapsed_seconds, 2), "giây")
print(
    "Next                      : implement atomic checkpoint/resume "
    "and run a one-optimizer-step preflight that is discarded"
)

## Cell 7 — Atomic checkpoint/restore và optimizer-step preflight

Cell này thực hiện một bước kiểm tra huấn luyện có kiểm soát:

- Chụp trạng thái ban đầu của encoder, decoder, optimizer, scheduler,
  GradScaler và các bộ sinh số ngẫu nhiên.
- Chạy 2 microbatch, tương ứng 64 ảnh và 1 optimizer step.
- Kiểm tra loss và gradient hữu hạn.
- Kiểm tra encoder và decoder thực sự được cập nhật.
- Lưu checkpoint tạm thời bằng cơ chế atomic.
- Tải lại checkpoint và xác minh khả năng resume chính xác.
- Khôi phục toàn bộ trạng thái ban đầu sau preflight.
- Xóa checkpoint tạm thời.
- Kiểm tra checksum của initial checkpoints không thay đổi.
- Tạo marker cho phép chuyển sang các cell production.
- Không truy cập Held-out Test Set.

Đây chỉ là preflight bị loại bỏ, không phải full confirmation training.

In [ ]:
import copy
import math
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID",
    "STUDENT_NAME",
    "DEVELOPMENT_SEED",
    "device",
    "RUN_CHECKPOINT_DIR",
    "RUN_CONFIG_DIR",
    "RUN_AUDIT_DIR",
    "S2_ENCODER_ADAPTER_MODEL",
    "S2_SHARED_DECODER_MODEL",
    "S2_OPTIMIZER",
    "S2_SCHEDULER",
    "S2_GRAD_SCALER",
    "S2_PARAMETER_GROUP_AUDIT",
    "S2_SCHEDULER_INFO",
    "S2_CURRENT_EPOCH",
    "S2_OPTIMIZER_STEP",
    "S2_GLOBAL_SAMPLE_CURSOR",
    "S2_MICROBATCH_IN_ACCUMULATION",
    "S2_OVERFLOW_RETRY_COUNT",
    "S2_ENCODER_INITIAL_CHECKPOINT_PATH",
    "SHARED_DECODER_INITIAL_CHECKPOINT_PATH",
    "actual_encoder_checkpoint_sha256",
    "actual_decoder_checkpoint_sha256",
    "S2_CELL06_OPTIMIZATION_CONTRACT_PATH",
    "S2_CELL06_OPTIMIZATION_CONTRACT_SHA256",
    "SHARED_TRAINING_CONTRACT",
    "build_s2_train_dataloader",
    "set_s2_training_mode",
    "write_json",
    "sha256_file",
]
missing_runtime_names = [
    name for name in required_runtime_names if name not in globals()
]
assert not missing_runtime_names, (
    "Thiếu biến từ S2 Cell 1–6: "
    f"{missing_runtime_names}. Hãy chạy lại các cell trước."
)

assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026
assert S2_CURRENT_EPOCH == 0
assert S2_OPTIMIZER_STEP == 0
assert S2_GLOBAL_SAMPLE_CURSOR == 0
assert S2_MICROBATCH_IN_ACCUMULATION == 0
assert S2_OVERFLOW_RETRY_COUNT == 0
assert len(S2_OPTIMIZER.state) == 0

optimization_contract = SHARED_TRAINING_CONTRACT["optimization"]
batching_contract = SHARED_TRAINING_CONTRACT["batching"]
physical_batch_size = int(batching_contract["physical_batch_size"])
accumulation_steps = int(
    batching_contract["gradient_accumulation_steps"]
)
effective_batch_size = physical_batch_size * accumulation_steps
gradient_clip_norm = float(
    optimization_contract["gradient_clip_norm"]
)
maximum_overflow_retries = int(
    optimization_contract["mixed_precision"][
        "maximum_overflow_retries_per_run"
    ]
)

assert physical_batch_size == 32
assert accumulation_steps == 2
assert effective_batch_size == 64
assert gradient_clip_norm == 1.0
assert maximum_overflow_retries == 10


def cpu_clone_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {
            key: cpu_clone_tree(item) for key, item in value.items()
        }
    if isinstance(value, list):
        return [cpu_clone_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_clone_tree(item) for item in value)
    return copy.deepcopy(value)


def trees_exact(left, right):
    if torch.is_tensor(left):
        return (
            torch.is_tensor(right)
            and left.dtype == right.dtype
            and tuple(left.shape) == tuple(right.shape)
            and torch.equal(left.cpu(), right.cpu())
        )
    if isinstance(left, dict):
        return (
            isinstance(right, dict)
            and set(left) == set(right)
            and all(trees_exact(left[key], right[key]) for key in left)
        )
    if isinstance(left, (list, tuple)):
        return (
            isinstance(right, type(left))
            and len(left) == len(right)
            and all(trees_exact(a, b) for a, b in zip(left, right))
        )
    if isinstance(left, float):
        return (
            isinstance(right, float)
            and (left == right or (math.isnan(left) and math.isnan(right)))
        )
    return left == right


def capture_rng_state():
    numpy_state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": {
            "bit_generator": numpy_state[0],
            "state": numpy_state[1].tolist(),
            "position": int(numpy_state[2]),
            "has_gauss": int(numpy_state[3]),
            "cached_gaussian": float(numpy_state[4]),
        },
        "torch_cpu": torch.random.get_rng_state().cpu().clone(),
        "torch_cuda": [
            state.cpu().clone()
            for state in torch.cuda.get_rng_state_all()
        ],
    }


def restore_rng_state(state):
    random.setstate(state["python"])
    numpy_state = state["numpy"]
    np.random.set_state(
        (
            numpy_state["bit_generator"],
            np.asarray(numpy_state["state"], dtype=np.uint32),
            int(numpy_state["position"]),
            int(numpy_state["has_gauss"]),
            float(numpy_state["cached_gaussian"]),
        )
    )
    torch.random.set_rng_state(state["torch_cpu"].cpu())
    torch.cuda.set_rng_state_all(
        [tensor.cpu() for tensor in state["torch_cuda"]]
    )


def atomic_torch_save(payload, target_path):
    target_path = Path(target_path)
    temporary_path = target_path.with_suffix(
        target_path.suffix + ".tmp"
    )
    if temporary_path.exists():
        temporary_path.unlink()
    try:
        torch.save(payload, temporary_path)
        assert temporary_path.is_file()
        assert temporary_path.stat().st_size > 0
        os.replace(temporary_path, target_path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()
    assert target_path.is_file()
    assert target_path.stat().st_size > 0


def model_state_cpu(model):
    return cpu_clone_tree(model.state_dict())


def gradients_finite(parameters):
    missing = []
    nonfinite = []
    for name, parameter in parameters:
        if not parameter.requires_grad:
            continue
        if parameter.grad is None:
            missing.append(name)
        elif not torch.isfinite(parameter.grad).all():
            nonfinite.append(name)
    return missing, nonfinite


def move_batch_to_device(batch):
    return {
        "images": batch["images"].to(device, non_blocking=True),
        "decoder_input_ids": batch["decoder_input_ids"].to(
            device, non_blocking=True
        ),
        "decoder_attention_mask": batch["decoder_attention_mask"].to(
            device, non_blocking=True
        ),
        "labels": batch["labels"].to(device, non_blocking=True),
        "row_indices": batch["row_indices"].clone(),
        "image_ids": batch["image_ids"].clone(),
        "annotation_ids": batch["annotation_ids"].clone(),
    }


def forward_token_loss(batch):
    visual_memory = S2_ENCODER_ADAPTER_MODEL(batch["images"])
    assert tuple(visual_memory.shape[1:]) == (49, 256)
    assert torch.isfinite(visual_memory).all()
    memory_attention_mask = torch.ones(
        visual_memory.shape[:2],
        dtype=torch.bool,
        device=visual_memory.device,
    )
    logits = S2_SHARED_DECODER_MODEL(
        decoder_input_ids=batch["decoder_input_ids"],
        visual_memory=visual_memory,
        decoder_attention_mask=batch["decoder_attention_mask"],
        memory_attention_mask=memory_attention_mask,
    )
    assert logits.shape[-1] == 10_080
    assert torch.isfinite(logits).all()
    loss = F.cross_entropy(
        logits.reshape(-1, 10_080).float(),
        batch["labels"].reshape(-1),
        ignore_index=0,
    )
    assert torch.isfinite(loss)
    valid_tokens = int((batch["labels"] != 0).sum().item())
    return loss, valid_tokens


initial_encoder_state = model_state_cpu(S2_ENCODER_ADAPTER_MODEL)
initial_decoder_state = model_state_cpu(S2_SHARED_DECODER_MODEL)
initial_optimizer_state = cpu_clone_tree(S2_OPTIMIZER.state_dict())
initial_scheduler_state = cpu_clone_tree(S2_SCHEDULER.state_dict())
initial_scaler_state = cpu_clone_tree(S2_GRAD_SCALER.state_dict())
initial_rng_state = capture_rng_state()

TEMPORARY_PREFLIGHT_CHECKPOINT_PATH = (
    Path(RUN_CHECKPOINT_DIR)
    / "s2_temporary_atomic_resume_preflight.pt"
)
TEMPORARY_PREFLIGHT_TMP_PATH = (
    TEMPORARY_PREFLIGHT_CHECKPOINT_PATH.with_suffix(
        TEMPORARY_PREFLIGHT_CHECKPOINT_PATH.suffix + ".tmp"
    )
)
for stale_path in [
    TEMPORARY_PREFLIGHT_CHECKPOINT_PATH,
    TEMPORARY_PREFLIGHT_TMP_PATH,
]:
    if stale_path.exists():
        stale_path.unlink()

preflight_completed = False
resume_load_exact = False
temporary_checkpoint_sha256 = None
temporary_checkpoint_size = 0
microbatch_losses = []
valid_tokens_seen = 0
gradient_norm_before_clip = None
changed_encoder_tensors = 0
changed_decoder_tensors = 0
preflight_optimizer_state_entries = 0
preflight_scheduler_last_epoch = None
preflight_scaler_scale = None
preflight_sample_cursor = 0

print("=" * 72)
print("S2 ONE-OPTIMIZER-STEP PREFLIGHT")
print("=" * 72)
print()
print(
    "Physical/effective batch :",
    physical_batch_size,
    "/",
    effective_batch_size,
)
print("Gradient accumulation    :", accumulation_steps)
print("Checkpoint boundary      : optimizer-step boundary")
print("Temporary weights kept   : False")
print("Test data read           : False")

try:
    train_loader, loader_info = build_s2_train_dataloader(
        epoch=0,
        start_position=0,
        persistent_workers=False,
    )
    train_iterator = iter(train_loader)
    frozen_batchnorm_count = set_s2_training_mode(
        S2_ENCODER_ADAPTER_MODEL,
        S2_SHARED_DECODER_MODEL,
    )
    assert frozen_batchnorm_count == 32
    S2_OPTIMIZER.zero_grad(set_to_none=True)

    batch_sequence = []
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    for microbatch_index in range(accumulation_steps):
        batch = move_batch_to_device(next(train_iterator))
        batch_sequence.append(
            {
                "microbatch_index": microbatch_index,
                "row_indices": batch["row_indices"].tolist(),
                "image_ids": batch["image_ids"].tolist(),
                "annotation_ids": batch["annotation_ids"].tolist(),
            }
        )
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            loss, valid_tokens = forward_token_loss(batch)
            scaled_loss = loss / accumulation_steps
        S2_GRAD_SCALER.scale(scaled_loss).backward()
        microbatch_losses.append(float(loss.detach().item()))
        valid_tokens_seen += valid_tokens

    del train_iterator
    del train_loader

    S2_GRAD_SCALER.unscale_(S2_OPTIMIZER)
    all_named_parameters = [
        (f"encoder.{name}", parameter)
        for name, parameter in S2_ENCODER_ADAPTER_MODEL.named_parameters()
    ] + [
        (f"decoder.{name}", parameter)
        for name, parameter in S2_SHARED_DECODER_MODEL.named_parameters()
    ]
    missing_gradients, nonfinite_gradients = gradients_finite(
        all_named_parameters
    )
    assert not missing_gradients, (
        "Trainable parameters không nhận gradient: "
        f"{missing_gradients}"
    )
    assert not nonfinite_gradients, (
        "Gradient không hữu hạn: "
        f"{nonfinite_gradients}"
    )

    gradient_norm_tensor = torch.nn.utils.clip_grad_norm_(
        [
            parameter
            for _, parameter in all_named_parameters
            if parameter.requires_grad
        ],
        max_norm=gradient_clip_norm,
        error_if_nonfinite=True,
    )
    gradient_norm_before_clip = float(gradient_norm_tensor.item())
    assert math.isfinite(gradient_norm_before_clip)

    scale_before_step = float(S2_GRAD_SCALER.get_scale())
    S2_GRAD_SCALER.step(S2_OPTIMIZER)
    S2_GRAD_SCALER.update()
    scale_after_step = float(S2_GRAD_SCALER.get_scale())
    assert scale_before_step == 1_024.0
    assert scale_after_step == 1_024.0, "Preflight gặp FP16 overflow."

    S2_SCHEDULER.step()
    S2_OPTIMIZER.zero_grad(set_to_none=True)
    torch.cuda.synchronize(device)

    preflight_encoder_state = model_state_cpu(S2_ENCODER_ADAPTER_MODEL)
    preflight_decoder_state = model_state_cpu(S2_SHARED_DECODER_MODEL)
    preflight_optimizer_state = cpu_clone_tree(S2_OPTIMIZER.state_dict())
    preflight_scheduler_state = cpu_clone_tree(S2_SCHEDULER.state_dict())
    preflight_scaler_state = cpu_clone_tree(S2_GRAD_SCALER.state_dict())
    preflight_rng_state = capture_rng_state()

    changed_encoder_tensors = sum(
        not torch.equal(
            initial_encoder_state[name], preflight_encoder_state[name]
        )
        for name in initial_encoder_state
    )
    changed_decoder_tensors = sum(
        not torch.equal(
            initial_decoder_state[name], preflight_decoder_state[name]
        )
        for name in initial_decoder_state
    )
    assert changed_encoder_tensors > 0
    assert changed_decoder_tensors > 0
    preflight_optimizer_state_entries = len(S2_OPTIMIZER.state)
    assert preflight_optimizer_state_entries > 0
    preflight_scheduler_last_epoch = int(S2_SCHEDULER.last_epoch)
    preflight_scaler_scale = float(S2_GRAD_SCALER.get_scale())
    preflight_sample_cursor = effective_batch_size

    checkpoint_payload = {
        "schema_version": "phase2_s2_atomic_resume_v1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "student_id": STUDENT_ID,
        "student_name": STUDENT_NAME,
        "seed": DEVELOPMENT_SEED,
        "contract_sha256": {
            "optimization": S2_CELL06_OPTIMIZATION_CONTRACT_SHA256,
        },
        "progress": {
            "epoch": 0,
            "optimizer_step": 1,
            "global_sample_cursor": preflight_sample_cursor,
            "microbatch_in_accumulation": 0,
            "overflow_retry_count": 0,
            "checkpoint_boundary": "optimizer_step",
        },
        "encoder_state_dict": preflight_encoder_state,
        "decoder_state_dict": preflight_decoder_state,
        "optimizer_state_dict": preflight_optimizer_state,
        "scheduler_state_dict": preflight_scheduler_state,
        "grad_scaler_state_dict": preflight_scaler_state,
        "rng_state": preflight_rng_state,
        "batch_sequence": batch_sequence,
        "data_policy": {
            "test_parquet_opened": False,
            "test_images_read": False,
            "test_set_status": "sealed",
            "test_set_used": False,
            "flickr_used": False,
        },
    }
    atomic_torch_save(
        checkpoint_payload, TEMPORARY_PREFLIGHT_CHECKPOINT_PATH
    )
    temporary_checkpoint_size = int(
        TEMPORARY_PREFLIGHT_CHECKPOINT_PATH.stat().st_size
    )
    temporary_checkpoint_sha256 = sha256_file(
        TEMPORARY_PREFLIGHT_CHECKPOINT_PATH
    )

    # Prove that a saved optimizer-step boundary can be restored exactly.
    S2_ENCODER_ADAPTER_MODEL.load_state_dict(
        initial_encoder_state, strict=True
    )
    S2_SHARED_DECODER_MODEL.load_state_dict(
        initial_decoder_state, strict=True
    )
    S2_OPTIMIZER.load_state_dict(initial_optimizer_state)
    S2_SCHEDULER.load_state_dict(initial_scheduler_state)
    S2_GRAD_SCALER.load_state_dict(initial_scaler_state)

    loaded_checkpoint = torch.load(
        TEMPORARY_PREFLIGHT_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )
    assert loaded_checkpoint["schema_version"] == (
        "phase2_s2_atomic_resume_v1"
    )
    assert loaded_checkpoint["student_id"] == "S2"
    assert loaded_checkpoint["progress"]["optimizer_step"] == 1
    assert loaded_checkpoint["progress"]["global_sample_cursor"] == 64
    assert (
        loaded_checkpoint["progress"]["microbatch_in_accumulation"] == 0
    )

    S2_ENCODER_ADAPTER_MODEL.load_state_dict(
        loaded_checkpoint["encoder_state_dict"], strict=True
    )
    S2_SHARED_DECODER_MODEL.load_state_dict(
        loaded_checkpoint["decoder_state_dict"], strict=True
    )
    S2_OPTIMIZER.load_state_dict(
        loaded_checkpoint["optimizer_state_dict"]
    )
    S2_SCHEDULER.load_state_dict(
        loaded_checkpoint["scheduler_state_dict"]
    )
    S2_GRAD_SCALER.load_state_dict(
        loaded_checkpoint["grad_scaler_state_dict"]
    )
    restore_rng_state(loaded_checkpoint["rng_state"])

    assert trees_exact(
        model_state_cpu(S2_ENCODER_ADAPTER_MODEL),
        preflight_encoder_state,
    )
    assert trees_exact(
        model_state_cpu(S2_SHARED_DECODER_MODEL),
        preflight_decoder_state,
    )
    assert trees_exact(
        cpu_clone_tree(S2_OPTIMIZER.state_dict()),
        preflight_optimizer_state,
    )
    assert trees_exact(
        cpu_clone_tree(S2_SCHEDULER.state_dict()),
        preflight_scheduler_state,
    )
    assert trees_exact(
        cpu_clone_tree(S2_GRAD_SCALER.state_dict()),
        preflight_scaler_state,
    )
    assert trees_exact(capture_rng_state(), preflight_rng_state)
    resume_load_exact = True
    preflight_completed = True
    del loaded_checkpoint

finally:
    # Always leave the notebook at the exact initial training state.
    S2_ENCODER_ADAPTER_MODEL.load_state_dict(
        initial_encoder_state, strict=True
    )
    S2_SHARED_DECODER_MODEL.load_state_dict(
        initial_decoder_state, strict=True
    )
    S2_OPTIMIZER.load_state_dict(initial_optimizer_state)
    S2_SCHEDULER.load_state_dict(initial_scheduler_state)
    S2_GRAD_SCALER.load_state_dict(initial_scaler_state)
    S2_OPTIMIZER.zero_grad(set_to_none=True)
    restore_rng_state(initial_rng_state)
    S2_ENCODER_ADAPTER_MODEL.eval()
    S2_SHARED_DECODER_MODEL.eval()

    S2_CURRENT_EPOCH = 0
    S2_OPTIMIZER_STEP = 0
    S2_GLOBAL_SAMPLE_CURSOR = 0
    S2_MICROBATCH_IN_ACCUMULATION = 0
    S2_OVERFLOW_RETRY_COUNT = 0
    for temporary_path in [
        TEMPORARY_PREFLIGHT_CHECKPOINT_PATH,
        TEMPORARY_PREFLIGHT_TMP_PATH,
    ]:
        if temporary_path.exists():
            temporary_path.unlink()


assert preflight_completed
assert resume_load_exact
assert not TEMPORARY_PREFLIGHT_CHECKPOINT_PATH.exists()
assert not TEMPORARY_PREFLIGHT_TMP_PATH.exists()
assert trees_exact(
    model_state_cpu(S2_ENCODER_ADAPTER_MODEL), initial_encoder_state
)
assert trees_exact(
    model_state_cpu(S2_SHARED_DECODER_MODEL), initial_decoder_state
)
assert trees_exact(
    cpu_clone_tree(S2_OPTIMIZER.state_dict()), initial_optimizer_state
)
assert trees_exact(
    cpu_clone_tree(S2_SCHEDULER.state_dict()), initial_scheduler_state
)
assert trees_exact(
    cpu_clone_tree(S2_GRAD_SCALER.state_dict()), initial_scaler_state
)
assert trees_exact(capture_rng_state(), initial_rng_state)
assert all(
    parameter.grad is None
    for parameter in (
        list(S2_ENCODER_ADAPTER_MODEL.parameters())
        + list(S2_SHARED_DECODER_MODEL.parameters())
    )
)
assert len(S2_OPTIMIZER.state) == 0
assert S2_CURRENT_EPOCH == 0
assert S2_OPTIMIZER_STEP == 0
assert S2_GLOBAL_SAMPLE_CURSOR == 0
assert S2_MICROBATCH_IN_ACCUMULATION == 0
assert (
    sha256_file(S2_ENCODER_INITIAL_CHECKPOINT_PATH)
    == actual_encoder_checkpoint_sha256
)
assert (
    sha256_file(SHARED_DECODER_INITIAL_CHECKPOINT_PATH)
    == actual_decoder_checkpoint_sha256
)

S2_CELL07_PREFLIGHT_CONTRACT_PATH = (
    RUN_CONFIG_DIR / "s2_atomic_resume_preflight_contract.json"
)
S2_CELL07_AUDIT_PATH = (
    RUN_AUDIT_DIR / "s2_cell07_atomic_resume_preflight_audit.json"
)
S2_FULL_TRAINING_READY_PATH = (
    Path(RUN_CONFIG_DIR).parent
    / "S2_SEED2026_FULL_TRAINING_READY.json"
)
cell_elapsed_seconds = time.perf_counter() - CELL_STARTED_AT

preflight_contract = {
    "contract_version": "phase2_s2_atomic_resume_preflight_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "passed",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "input_contract": {
        "path": str(S2_CELL06_OPTIMIZATION_CONTRACT_PATH),
        "sha256": S2_CELL06_OPTIMIZATION_CONTRACT_SHA256,
    },
    "preflight": {
        "optimizer_steps": 1,
        "microbatches": accumulation_steps,
        "images": effective_batch_size,
        "valid_tokens": valid_tokens_seen,
        "microbatch_losses": microbatch_losses,
        "mean_loss": float(
            sum(microbatch_losses) / len(microbatch_losses)
        ),
        "gradient_norm_before_clip": gradient_norm_before_clip,
        "changed_encoder_tensors": changed_encoder_tensors,
        "changed_decoder_tensors": changed_decoder_tensors,
        "optimizer_state_entries": preflight_optimizer_state_entries,
        "scheduler_last_epoch": preflight_scheduler_last_epoch,
        "grad_scaler_scale": preflight_scaler_scale,
        "sample_cursor": preflight_sample_cursor,
    },
    "checkpoint": {
        "atomic_save": True,
        "resume_load_exact": resume_load_exact,
        "temporary_checkpoint_sha256": temporary_checkpoint_sha256,
        "temporary_checkpoint_size": temporary_checkpoint_size,
        "temporary_checkpoint_deleted": True,
        "temporary_tmp_file_deleted": True,
    },
    "restoration": {
        "encoder_state_exact": True,
        "decoder_state_exact": True,
        "optimizer_state_exact": True,
        "scheduler_state_exact": True,
        "grad_scaler_state_exact": True,
        "rng_state_exact": True,
        "all_gradients_none": True,
        "initial_checkpoint_files_unchanged": True,
        "epoch": 0,
        "optimizer_step": 0,
        "global_sample_cursor": 0,
    },
    "policy": {
        "preflight_weights_saved": False,
        "full_training_started": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
    },
}
write_json(S2_CELL07_PREFLIGHT_CONTRACT_PATH, preflight_contract)

preflight_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "student_id": STUDENT_ID,
    "one_optimizer_step_passed": True,
    "loss_finite": True,
    "gradients_finite": True,
    "parameter_updates_observed": True,
    "fp16_overflow": False,
    "atomic_checkpoint_passed": True,
    "resume_load_exact": True,
    "initial_state_restored_exact": True,
    "temporary_checkpoints_deleted": True,
    "full_training_started": False,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "elapsed_seconds": cell_elapsed_seconds,
}
write_json(S2_CELL07_AUDIT_PATH, preflight_audit)

S2_CELL07_PREFLIGHT_CONTRACT_SHA256 = sha256_file(
    S2_CELL07_PREFLIGHT_CONTRACT_PATH
)
S2_CELL07_AUDIT_SHA256 = sha256_file(S2_CELL07_AUDIT_PATH)

s2_ready_marker = {
    "schema_version": "phase2_s2_seed2026_full_training_ready_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "ready",
    "student_id": STUDENT_ID,
    "student_name": STUDENT_NAME,
    "seed": DEVELOPMENT_SEED,
    "authorized_next_action": (
        "full S2 seed-2026 baseline training from published initial "
        "checkpoints with Validation-only checkpoint selection"
    ),
    "preflight_contract": {
        "path": str(S2_CELL07_PREFLIGHT_CONTRACT_PATH),
        "sha256": S2_CELL07_PREFLIGHT_CONTRACT_SHA256,
    },
    "preflight_audit": {
        "path": str(S2_CELL07_AUDIT_PATH),
        "sha256": S2_CELL07_AUDIT_SHA256,
    },
    "initial_state_restored_exact": True,
    "temporary_checkpoints_deleted": True,
    "full_training_authorized": True,
    "full_training_started": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
}
write_json(S2_FULL_TRAINING_READY_PATH, s2_ready_marker)
S2_FULL_TRAINING_READY_SHA256 = sha256_file(
    S2_FULL_TRAINING_READY_PATH
)

print()
print("=" * 72)
print("S2 ATOMIC CHECKPOINT / RESTORE AUDIT")
print("=" * 72)
print("Microbatch losses       :", microbatch_losses)
print("Valid tokens            :", valid_tokens_seen)
print("Gradient norm           :", gradient_norm_before_clip)
print("Encoder tensors changed :", changed_encoder_tensors)
print("Decoder tensors changed :", changed_decoder_tensors)
print("Optimizer state entries :", preflight_optimizer_state_entries)
print("Scheduler last epoch    :", preflight_scheduler_last_epoch)
print("GradScaler scale        :", preflight_scaler_scale)
print("Resume load exact       :", resume_load_exact)
print("Initial restore exact   : True")
print("Temporary checkpoint    : DELETED")

print()
print("=" * 72)
print("PHASE 2 S2 CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Preflight optimizer steps : 1")
print("Preflight microbatches    : 2")
print("Preflight images          : 64")
print("Loss finite               : True")
print("Gradients finite          : True")
print("Parameter update observed : True")
print("FP16 overflow             : False")
print("Atomic checkpoint save    : PASSED")
print("Resume load exact         : True")
print("Encoder restored exact    : True")
print("Decoder restored exact    : True")
print("Optimizer restored exact  : True")
print("Scheduler restored exact  : True")
print("GradScaler restored exact : True")
print("RNG restored exact        : True")
print("Progress restored         : epoch 0 / step 0 / cursor 0")
print("Initial checkpoints intact: True")
print("Temporary checkpoints     : DELETED")
print("Full training authorized  : True")
print("Full training started     : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Preflight contract        :", S2_CELL07_PREFLIGHT_CONTRACT_PATH)
print("Contract SHA-256          :", S2_CELL07_PREFLIGHT_CONTRACT_SHA256)
print("Audit                     :", S2_CELL07_AUDIT_PATH)
print("Audit SHA-256             :", S2_CELL07_AUDIT_SHA256)
print("READY marker              :", S2_FULL_TRAINING_READY_PATH)
print("READY SHA-256             :", S2_FULL_TRAINING_READY_SHA256)
print("Elapsed                   :", round(cell_elapsed_seconds, 2), "giây")
print(
    "Next                      : build and audit the official "
    "Validation metric backend without opening Held-out Test"
)

## Cell 8 — Backend đánh giá chính thức trên Validation

Cell này thực hiện:

- Xác minh marker sẵn sàng từ Cell 7.
- Đọc caption của Karpathy Validation.
- Xây dựng references cho 5.000 ảnh Validation.
- Kiểm tra đủ 25.010 reference captions.
- Kiểm tra caption rỗng và rò rỉ sang Held-out Test.
- Khóa pycocoevalcap phiên bản 1.2.
- Kiểm tra Java và COCO PTBTokenizer.
- Kết nối năm metric:
  - BLEU-1;
  - BLEU-4;
  - METEOR;
  - ROUGE-L;
  - CIDEr-D.
- Smoke-test backend trên 32 ảnh Validation.
- Ghi references, metric contract, audit và SHA-256.
- Không mở Test Parquet, Test caption hoặc Test image.

In [ ]:
%pip install -q --disable-pip-version-check pycocoevalcap==1.2

In [ ]:
import importlib.metadata
import importlib.util
import math
import subprocess
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import pyarrow.parquet as pq


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID",
    "STUDENT_NAME",
    "DEVELOPMENT_SEED",
    "RUN_ROOT",
    "RUN_CONFIG_DIR",
    "RUN_AUDIT_DIR",
    "S2_VALIDATION_IMAGE_PATH_BY_ID",
    "S2_HELD_OUT_TEST_IMAGE_IDS",
    "PHASE1_DEVELOPMENT_ARTIFACT_PATHS",
    "S2_FULL_TRAINING_READY_PATH",
    "S2_FULL_TRAINING_READY_SHA256",
    "read_json",
    "write_json",
    "sha256_file",
]
missing_runtime_names = [
    name for name in required_runtime_names if name not in globals()
]
assert not missing_runtime_names, (
    "Thiếu biến từ S2 Cell 1–7: "
    f"{missing_runtime_names}. Hãy chạy lại các cell trước."
)

assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026

s2_ready = read_json(S2_FULL_TRAINING_READY_PATH)
assert s2_ready["status"] == "ready"
assert s2_ready["full_training_authorized"] is True
assert s2_ready["initial_state_restored_exact"] is True
assert s2_ready["temporary_checkpoints_deleted"] is True
assert s2_ready["test_set_status"] == "sealed"
assert s2_ready["test_set_used"] is False
assert (
    sha256_file(S2_FULL_TRAINING_READY_PATH)
    == S2_FULL_TRAINING_READY_SHA256
)


PACKAGE_NAME = "pycocoevalcap"
REQUIRED_PACKAGE_VERSION = "1.2"

assert importlib.util.find_spec(PACKAGE_NAME) is not None, (
    "Thiếu pycocoevalcap. Thêm một Kaggle Code Cell ngay trước Cell 8:\n"
    "%pip install -q --disable-pip-version-check pycocoevalcap==1.2\n"
    "Sau khi cài xong, chạy lại Cell 8; không restart runtime."
)
installed_metric_version = importlib.metadata.version(PACKAGE_NAME)
assert installed_metric_version == REQUIRED_PACKAGE_VERSION, (
    "pycocoevalcap phải là phiên bản 1.2, "
    f"hiện tại={installed_metric_version}"
)

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


java_probe = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True,
    timeout=30,
    check=False,
)
assert java_probe.returncode == 0, (
    "Java runtime không khả dụng; COCO PTBTokenizer/METEOR không thể chạy."
)
java_version_lines = (
    java_probe.stderr.strip().splitlines()
    or java_probe.stdout.strip().splitlines()
)
assert java_version_lines
java_version_line = java_version_lines[0]


print("=" * 72)
print("BUILD KARPATHY VALIDATION REFERENCES")
print("=" * 72)
print()

VALIDATION_PARQUET_PATH = Path(
    PHASE1_DEVELOPMENT_ARTIFACT_PATHS["validation_parquet"]
)
validation_table = pq.read_table(
    VALIDATION_PARQUET_PATH,
    columns=[
        "annotation_id",
        "image_id",
        "karpathy_split",
        "caption",
    ],
)
assert validation_table.num_rows == 25_010

validation_columns = validation_table.to_pydict()
S2_VALIDATION_REFERENCES = defaultdict(list)
annotation_ids = set()
empty_caption_count = 0

for annotation_id, image_id, karpathy_split, caption in zip(
    validation_columns["annotation_id"],
    validation_columns["image_id"],
    validation_columns["karpathy_split"],
    validation_columns["caption"],
):
    annotation_id = int(annotation_id)
    image_id = int(image_id)
    caption = str(caption).strip()

    assert karpathy_split == "val"
    assert annotation_id not in annotation_ids
    assert image_id not in S2_HELD_OUT_TEST_IMAGE_IDS
    assert image_id in S2_VALIDATION_IMAGE_PATH_BY_ID

    annotation_ids.add(annotation_id)
    if not caption:
        empty_caption_count += 1
    S2_VALIDATION_REFERENCES[image_id].append(caption)

assert empty_caption_count == 0
assert len(annotation_ids) == 25_010
assert len(S2_VALIDATION_REFERENCES) == 5_000
assert set(S2_VALIDATION_REFERENCES) == set(
    S2_VALIDATION_IMAGE_PATH_BY_ID
)
assert not (
    set(S2_VALIDATION_REFERENCES) & S2_HELD_OUT_TEST_IMAGE_IDS
)

reference_count_distribution = dict(
    sorted(
        Counter(
            len(captions)
            for captions in S2_VALIDATION_REFERENCES.values()
        ).items()
    )
)
assert sum(
    reference_count * image_count
    for reference_count, image_count in reference_count_distribution.items()
) == 25_010
assert reference_count_distribution == {5: 4_990, 6: 10}

S2_VALIDATION_REFERENCE_DIR = Path(RUN_ROOT) / "validation"
S2_VALIDATION_REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
S2_VALIDATION_REFERENCES_PATH = (
    S2_VALIDATION_REFERENCE_DIR
    / "karpathy_validation_references.json"
)
validation_reference_payload = {
    "schema_version": "phase2_s2_validation_references_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "MS COCO 2014",
    "split": "Karpathy validation",
    "image_count": 5_000,
    "caption_count": 25_010,
    "reference_count_distribution": reference_count_distribution,
    "references": {
        str(image_id): captions
        for image_id, captions in sorted(
            S2_VALIDATION_REFERENCES.items()
        )
    },
    "data_policy": {
        "validation_caption_text_read": True,
        "test_parquet_opened": False,
        "test_caption_text_read": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
}
write_json(
    S2_VALIDATION_REFERENCES_PATH,
    validation_reference_payload,
)
S2_VALIDATION_REFERENCES_SHA256 = sha256_file(
    S2_VALIDATION_REFERENCES_PATH
)

del validation_table
del validation_columns

print("Validation images       :", len(S2_VALIDATION_REFERENCES))
print("Validation references   :", len(annotation_ids))
print("References/image        :", reference_count_distribution)
print("Empty captions          :", empty_caption_count)
print("Held-out Test overlap   : 0")


def coco_metric_payload(references_by_image, hypotheses_by_image):
    image_ids = sorted(references_by_image)
    assert image_ids
    assert set(image_ids) == set(hypotheses_by_image)
    ground_truth = {
        image_id: [
            {"caption": caption}
            for caption in references_by_image[image_id]
        ]
        for image_id in image_ids
    }
    results = {
        image_id: [
            {"caption": str(hypotheses_by_image[image_id])}
        ]
        for image_id in image_ids
    }
    return ground_truth, results


def compute_coco_caption_metrics(
    references_by_image,
    hypotheses_by_image,
):
    """Compute the five locked validation metrics with COCO tokenization."""
    ground_truth, results = coco_metric_payload(
        references_by_image,
        hypotheses_by_image,
    )
    tokenizer = PTBTokenizer()
    tokenized_ground_truth = tokenizer.tokenize(ground_truth)
    tokenized_results = tokenizer.tokenize(results)

    bleu_score, _ = Bleu(4).compute_score(
        tokenized_ground_truth,
        tokenized_results,
    )
    meteor_scorer = Meteor()
    try:
        meteor_score, _ = meteor_scorer.compute_score(
            tokenized_ground_truth,
            tokenized_results,
        )
    finally:
        close_method = getattr(meteor_scorer, "close", None)
        if callable(close_method):
            close_method()

    rouge_score, _ = Rouge().compute_score(
        tokenized_ground_truth,
        tokenized_results,
    )
    cider_score, _ = Cider().compute_score(
        tokenized_ground_truth,
        tokenized_results,
    )
    metrics = {
        "BLEU-1": float(bleu_score[0]),
        "BLEU-4": float(bleu_score[3]),
        "METEOR": float(meteor_score),
        "ROUGE-L": float(rouge_score),
        "CIDEr-D": float(cider_score),
    }
    assert all(math.isfinite(value) for value in metrics.values())
    assert all(value >= 0.0 for value in metrics.values())
    return metrics


print()
print("=" * 72)
print("OFFICIAL COCO METRIC BACKEND SMOKE TEST")
print("=" * 72)
print()

metric_smoke_image_ids = sorted(S2_VALIDATION_REFERENCES)[:32]
metric_smoke_references = {
    image_id: S2_VALIDATION_REFERENCES[image_id]
    for image_id in metric_smoke_image_ids
}
metric_smoke_hypotheses = {
    image_id: S2_VALIDATION_REFERENCES[image_id][0]
    for image_id in metric_smoke_image_ids
}
S2_METRIC_BACKEND_SMOKE_SCORES = compute_coco_caption_metrics(
    metric_smoke_references,
    metric_smoke_hypotheses,
)
assert set(S2_METRIC_BACKEND_SMOKE_SCORES) == {
    "BLEU-1",
    "BLEU-4",
    "METEOR",
    "ROUGE-L",
    "CIDEr-D",
}

print("Backend package         :", PACKAGE_NAME)
print("Backend version         :", installed_metric_version)
print("Java                    :", java_version_line)
print("PTBTokenizer            : PASSED")
print("Metric images           :", len(metric_smoke_image_ids))
print("Metrics                 :", S2_METRIC_BACKEND_SMOKE_SCORES)
print("All metrics finite      : True")


S2_CELL08_METRIC_CONTRACT_PATH = (
    RUN_CONFIG_DIR / "s2_validation_metric_backend_contract.json"
)
S2_CELL08_AUDIT_PATH = (
    RUN_AUDIT_DIR / "s2_cell08_validation_metric_backend_audit.json"
)
cell_elapsed_seconds = time.perf_counter() - CELL_STARTED_AT

metric_contract = {
    "contract_version": "phase2_s2_validation_metric_backend_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "locked",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "full_training_ready": {
        "path": str(S2_FULL_TRAINING_READY_PATH),
        "sha256": S2_FULL_TRAINING_READY_SHA256,
    },
    "backend": {
        "package": PACKAGE_NAME,
        "version": installed_metric_version,
        "tokenizer": "COCO PTBTokenizer",
        "java": java_version_line,
        "metrics": [
            "BLEU-1",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
            "CIDEr-D",
        ],
        "smoke_image_count": len(metric_smoke_image_ids),
        "smoke_scores": S2_METRIC_BACKEND_SMOKE_SCORES,
    },
    "validation_references": {
        "path": str(S2_VALIDATION_REFERENCES_PATH),
        "sha256": S2_VALIDATION_REFERENCES_SHA256,
        "image_count": 5_000,
        "caption_count": 25_010,
        "reference_count_distribution": reference_count_distribution,
    },
    "policy": {
        "validation_caption_text_read": True,
        "training_caption_text_read": False,
        "test_parquet_opened": False,
        "test_caption_text_read": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "full_training_started": False,
    },
}
write_json(S2_CELL08_METRIC_CONTRACT_PATH, metric_contract)

cell_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "student_id": STUDENT_ID,
    "metric_package_version": installed_metric_version,
    "java_available": True,
    "ptbtokenizer_passed": True,
    "metric_smoke_passed": True,
    "metric_smoke_scores": S2_METRIC_BACKEND_SMOKE_SCORES,
    "validation_images": 5_000,
    "validation_references": 25_010,
    "empty_validation_captions": 0,
    "validation_test_overlap": 0,
    "validation_caption_text_read": True,
    "training_caption_text_read": False,
    "test_parquet_opened": False,
    "test_caption_text_read": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "full_training_started": False,
    "elapsed_seconds": cell_elapsed_seconds,
}
write_json(S2_CELL08_AUDIT_PATH, cell_audit)

S2_CELL08_METRIC_CONTRACT_SHA256 = sha256_file(
    S2_CELL08_METRIC_CONTRACT_PATH
)
S2_CELL08_AUDIT_SHA256 = sha256_file(S2_CELL08_AUDIT_PATH)

print()
print("=" * 72)
print("PHASE 2 S2 CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Metric backend            : pycocoevalcap 1.2")
print("COCO PTBTokenizer         : PASSED")
print("Java runtime              : AVAILABLE")
print("Metrics                   : 5 / 5 PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("Empty validation captions : 0")
print("Validation/Test overlap   : 0")
print("Validation caption read   : True")
print("Training caption read     : False")
print("Test Parquet opened       : False")
print("Test caption read         : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Full training started     : False")
print("References                :", S2_VALIDATION_REFERENCES_PATH)
print("References SHA-256        :", S2_VALIDATION_REFERENCES_SHA256)
print("Metric contract           :", S2_CELL08_METRIC_CONTRACT_PATH)
print("Contract SHA-256          :", S2_CELL08_METRIC_CONTRACT_SHA256)
print("Audit                     :", S2_CELL08_AUDIT_PATH)
print("Audit SHA-256             :", S2_CELL08_AUDIT_SHA256)
print("Elapsed                   :", round(cell_elapsed_seconds, 2), "giây")
print(
    "Next                      : define and audit the production "
    "teacher-forced Validation and Beam-3 generation engine"
)

## Cell 9 — Production Validation và Beam-3 Generation Engine

Cell này thực hiện:

- Xây dựng dataset gồm đúng 5.000 ảnh Validation duy nhất.
- Định nghĩa teacher-forced Validation trên 25.010 captions.
- Định nghĩa Beam Search với:
  - beam size = 3;
  - length penalty = 0.7;
  - tối đa 31 decoder steps;
  - tối thiểu 2 lexical tokens trước EOS;
  - cấm sinh PAD, BOS và UNK.
- Kết nối generation với năm COCO-caption metrics.
- Khóa quy tắc chọn checkpoint bằng Validation:
  1. CIDEr-D cao hơn;
  2. BLEU-4 cao hơn;
  3. Validation Cross-Entropy thấp hơn;
  4. Epoch sớm hơn.
- Smoke-test teacher-forced Validation.
- Chạy Beam-3 hai lần để kiểm tra khả năng lặp lại.
- Xác minh model và toàn bộ training state không thay đổi.
- Không optimizer step, không full Validation và không truy cập Test Set.

In [ ]:
import json
import math
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID",
    "STUDENT_NAME",
    "DEVELOPMENT_SEED",
    "RUN_ROOT",
    "RUN_CONFIG_DIR",
    "RUN_AUDIT_DIR",
    "device",
    "training_contract",
    "validation_policy",
    "vocabulary",
    "PAD_ID",
    "BOS_ID",
    "EOS_ID",
    "UNK_ID",
    "S2_VALIDATION_IMAGE_PATH_BY_ID",
    "S2_HELD_OUT_TEST_IMAGE_IDS",
    "S2_VALIDATION_TRANSFORM",
    "S2_VALIDATION_DATASET",
    "build_s2_validation_dataloader",
    "seed_dataloader_worker",
    "dynamic_caption_collator",
    "S2_ENCODER_ADAPTER_MODEL",
    "S2_SHARED_DECODER_MODEL",
    "S2_OPTIMIZER",
    "S2_SCHEDULER",
    "S2_GRAD_SCALER",
    "set_s2_training_mode",
    "move_batch_to_device",
    "model_state_cpu",
    "cpu_clone_tree",
    "trees_exact",
    "compute_coco_caption_metrics",
    "S2_VALIDATION_REFERENCES",
    "S2_VALIDATION_REFERENCES_PATH",
    "S2_VALIDATION_REFERENCES_SHA256",
    "S2_CELL08_METRIC_CONTRACT_PATH",
    "S2_CELL08_METRIC_CONTRACT_SHA256",
    "S2_CELL08_AUDIT_PATH",
    "S2_CELL08_AUDIT_SHA256",
    "read_json",
    "write_json",
    "sha256_file",
]
missing_runtime_names = [
    name for name in required_runtime_names if name not in globals()
]
assert not missing_runtime_names, (
    "Thiếu biến từ S2 Cell 1–8: "
    f"{missing_runtime_names}. Hãy chạy lại các cell trước."
)

assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026
assert PAD_ID == 0
assert BOS_ID == 1
assert EOS_ID == 2
assert UNK_ID == 3
assert len(S2_VALIDATION_IMAGE_PATH_BY_ID) == 5_000
assert len(S2_VALIDATION_REFERENCES) == 5_000
assert not (
    set(S2_VALIDATION_IMAGE_PATH_BY_ID)
    & set(S2_HELD_OUT_TEST_IMAGE_IDS)
)

cell08_metric_contract = read_json(
    S2_CELL08_METRIC_CONTRACT_PATH
)
cell08_metric_audit = read_json(
    S2_CELL08_AUDIT_PATH
)
assert cell08_metric_contract["status"] == "locked"
assert cell08_metric_audit["status"] == "passed"
assert cell08_metric_audit["metric_smoke_passed"] is True
assert cell08_metric_audit["test_set_status"] == "sealed"
assert cell08_metric_audit["test_set_used"] is False
assert (
    sha256_file(S2_CELL08_METRIC_CONTRACT_PATH)
    == S2_CELL08_METRIC_CONTRACT_SHA256
)
assert (
    sha256_file(S2_CELL08_AUDIT_PATH)
    == S2_CELL08_AUDIT_SHA256
)
assert (
    sha256_file(S2_VALIDATION_REFERENCES_PATH)
    == S2_VALIDATION_REFERENCES_SHA256
)


generation_policy = validation_policy["generation_validation"]
decoding_policy = generation_policy["decoding"]
selection_policy = validation_policy["checkpoint_selection"]
teacher_forced_policy = validation_policy[
    "teacher_forced_validation"
]

BEAM_SIZE = int(decoding_policy["beam_size"])
LENGTH_PENALTY = float(decoding_policy["length_penalty"])
MAXIMUM_DECODER_STEPS = int(
    decoding_policy["maximum_decoder_steps"]
)
MINIMUM_LEXICAL_TOKENS = int(
    decoding_policy["minimum_lexical_tokens_before_eos"]
)
FORBIDDEN_GENERATION_TOKEN_IDS = tuple(
    int(token_id)
    for token_id in decoding_policy[
        "forbidden_generation_token_ids"
    ]
)
VOCABULARY_SIZE = int(vocabulary["vocabulary_size"])
ID_TO_TOKEN = tuple(vocabulary["id_to_token"])

assert BEAM_SIZE == 3
assert LENGTH_PENALTY == 0.7
assert MAXIMUM_DECODER_STEPS == 31
assert MINIMUM_LEXICAL_TOKENS == 2
assert FORBIDDEN_GENERATION_TOKEN_IDS == (
    PAD_ID,
    BOS_ID,
    UNK_ID,
)
assert int(decoding_policy["eos_id"]) == EOS_ID
assert VOCABULARY_SIZE == 10_080
assert len(ID_TO_TOKEN) == VOCABULARY_SIZE
assert selection_policy["primary_metric"] == "CIDEr-D"
assert selection_policy["primary_mode"] == "max"
assert selection_policy["test_metrics_forbidden"] is True
assert selection_policy["test_predictions_forbidden"] is True
assert teacher_forced_policy["corpus"] == (
    "all 25,010 validation captions"
)


class S2UniqueValidationImageDataset(Dataset):
    """One deterministic record per Karpathy Validation image."""

    def __init__(
        self,
        image_path_by_id,
        image_transform,
        held_out_test_ids,
    ):
        self.image_path_by_id = {
            int(image_id): Path(image_path)
            for image_id, image_path
            in image_path_by_id.items()
        }
        self.image_ids = tuple(sorted(self.image_path_by_id))
        self.image_transform = image_transform
        self.held_out_test_ids = set(
            int(value) for value in held_out_test_ids
        )

        assert len(self.image_ids) == 5_000
        assert not (
            set(self.image_ids) & self.held_out_test_ids
        )

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[int(index)]
        assert image_id not in self.held_out_test_ids
        with Image.open(
            self.image_path_by_id[image_id]
        ) as source_image:
            image_tensor = self.image_transform(
                source_image.convert("RGB")
            )
        assert tuple(image_tensor.shape) == (3, 224, 224)
        assert image_tensor.dtype == torch.float32
        assert torch.isfinite(image_tensor).all()
        return {
            "image_id": image_id,
            "image": image_tensor,
        }


def unique_validation_image_collator(samples):
    assert samples
    return {
        "image_ids": torch.tensor(
            [sample["image_id"] for sample in samples],
            dtype=torch.long,
        ),
        "images": torch.stack(
            [sample["image"] for sample in samples],
            dim=0,
        ),
    }


S2_UNIQUE_VALIDATION_IMAGE_DATASET = (
    S2UniqueValidationImageDataset(
        image_path_by_id=S2_VALIDATION_IMAGE_PATH_BY_ID,
        image_transform=S2_VALIDATION_TRANSFORM,
        held_out_test_ids=S2_HELD_OUT_TEST_IMAGE_IDS,
    )
)


def build_s2_unique_validation_image_dataloader(
    batch_size=32,
    num_workers=2,
    persistent_workers=True,
):
    batch_size = int(batch_size)
    num_workers = int(num_workers)
    assert batch_size > 0
    assert num_workers >= 0

    worker_generator = torch.Generator()
    worker_generator.manual_seed(
        DEVELOPMENT_SEED + 20_000
    )
    return DataLoader(
        dataset=S2_UNIQUE_VALIDATION_IMAGE_DATASET,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
        collate_fn=unique_validation_image_collator,
        worker_init_fn=seed_dataloader_worker,
        generator=worker_generator,
        persistent_workers=bool(
            persistent_workers and num_workers > 0
        ),
        prefetch_factor=(
            2 if num_workers > 0 else None
        ),
    )


def decode_caption_token_ids(token_ids):
    lexical_tokens = []
    saw_eos = False
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == BOS_ID:
            continue
        if token_id == EOS_ID:
            saw_eos = True
            break
        if token_id in {
            PAD_ID,
            UNK_ID,
        }:
            continue
        lexical_tokens.append(ID_TO_TOKEN[token_id])

    caption = " ".join(lexical_tokens).strip()
    return caption, saw_eos, len(lexical_tokens)


@torch.inference_mode()
def s2_beam_search_from_visual_memory(
    visual_memory,
    beam_size=BEAM_SIZE,
    length_penalty=LENGTH_PENALTY,
    maximum_decoder_steps=MAXIMUM_DECODER_STEPS,
    minimum_lexical_tokens=MINIMUM_LEXICAL_TOKENS,
):
    """
    Batched Beam Search.

    Ranking rule:
        normalized_score = raw_log_probability /
                           generated_length ** length_penalty

    generated_length includes EOS when EOS is emitted. This matches the
    previously locked Phase-8 mobile decoder convention.
    """
    assert visual_memory.ndim == 3
    batch_size, memory_length, memory_dimension = (
        visual_memory.shape
    )
    assert memory_length == 49
    assert memory_dimension == 256
    assert beam_size == 3
    assert maximum_decoder_steps == 31

    model_device = visual_memory.device
    negative_infinity = torch.tensor(
        float("-inf"),
        device=model_device,
        dtype=torch.float32,
    )

    beam_tokens = torch.full(
        (batch_size, beam_size, 1),
        fill_value=BOS_ID,
        device=model_device,
        dtype=torch.long,
    )
    beam_scores = torch.full(
        (batch_size, beam_size),
        fill_value=float("-inf"),
        device=model_device,
        dtype=torch.float32,
    )
    beam_scores[:, 0] = 0.0
    beam_finished = torch.zeros(
        (batch_size, beam_size),
        device=model_device,
        dtype=torch.bool,
    )
    beam_lengths = torch.zeros(
        (batch_size, beam_size),
        device=model_device,
        dtype=torch.long,
    )

    expanded_memory = (
        visual_memory.unsqueeze(1)
        .expand(-1, beam_size, -1, -1)
        .reshape(
            batch_size * beam_size,
            memory_length,
            memory_dimension,
        )
        .contiguous()
    )
    memory_attention_mask = torch.ones(
        (batch_size * beam_size, memory_length),
        device=model_device,
        dtype=torch.bool,
    )

    for generation_step in range(
        maximum_decoder_steps
    ):
        flat_tokens = beam_tokens.reshape(
            batch_size * beam_size,
            -1,
        )
        token_attention_mask = (
            flat_tokens != PAD_ID
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = S2_SHARED_DECODER_MODEL(
                decoder_input_ids=flat_tokens,
                visual_memory=expanded_memory,
                decoder_attention_mask=(
                    token_attention_mask
                ),
                memory_attention_mask=(
                    memory_attention_mask
                ),
            )

        next_log_probabilities = F.log_softmax(
            logits[:, -1, :].float(),
            dim=-1,
        ).reshape(
            batch_size,
            beam_size,
            VOCABULARY_SIZE,
        )
        assert torch.isfinite(
            next_log_probabilities[
                torch.isfinite(next_log_probabilities)
            ]
        ).all()

        next_log_probabilities[
            :,
            :,
            list(FORBIDDEN_GENERATION_TOKEN_IDS),
        ] = negative_infinity
        if generation_step < minimum_lexical_tokens:
            next_log_probabilities[
                :,
                :,
                EOS_ID,
            ] = negative_infinity

        if beam_finished.any():
            next_log_probabilities = (
                next_log_probabilities.masked_fill(
                    beam_finished.unsqueeze(-1),
                    negative_infinity,
                )
            )
            finished_indices = (
                beam_finished.nonzero(
                    as_tuple=False
                )
            )
            next_log_probabilities[
                finished_indices[:, 0],
                finished_indices[:, 1],
                EOS_ID,
            ] = 0.0

        candidate_scores = (
            beam_scores.unsqueeze(-1)
            + next_log_probabilities
        )
        flat_candidate_scores = (
            candidate_scores.reshape(
                batch_size,
                beam_size * VOCABULARY_SIZE,
            )
        )
        next_scores, next_flat_indices = (
            torch.topk(
                flat_candidate_scores,
                k=beam_size,
                dim=-1,
            )
        )
        parent_beam_indices = (
            next_flat_indices
            // VOCABULARY_SIZE
        )
        next_token_ids = (
            next_flat_indices
            % VOCABULARY_SIZE
        )

        gather_token_indices = (
            parent_beam_indices.unsqueeze(-1)
            .expand(
                -1,
                -1,
                beam_tokens.shape[-1],
            )
        )
        parent_tokens = torch.gather(
            beam_tokens,
            dim=1,
            index=gather_token_indices,
        )
        parent_finished = torch.gather(
            beam_finished,
            dim=1,
            index=parent_beam_indices,
        )
        parent_lengths = torch.gather(
            beam_lengths,
            dim=1,
            index=parent_beam_indices,
        )

        beam_tokens = torch.cat(
            [
                parent_tokens,
                next_token_ids.unsqueeze(-1),
            ],
            dim=-1,
        )
        beam_scores = next_scores
        beam_lengths = (
            parent_lengths
            + (~parent_finished).long()
        )
        beam_finished = (
            parent_finished
            | (next_token_ids == EOS_ID)
        )

        if bool(beam_finished.all()):
            break

    normalized_scores = beam_scores / (
        beam_lengths.clamp_min(1).float()
        ** float(length_penalty)
    )
    best_beam_indices = torch.argmax(
        normalized_scores,
        dim=1,
    )
    batch_indices = torch.arange(
        batch_size,
        device=model_device,
    )

    best_tokens = beam_tokens[
        batch_indices,
        best_beam_indices,
    ]
    best_raw_scores = beam_scores[
        batch_indices,
        best_beam_indices,
    ]
    best_normalized_scores = normalized_scores[
        batch_indices,
        best_beam_indices,
    ]
    best_lengths = beam_lengths[
        batch_indices,
        best_beam_indices,
    ]

    assert torch.isfinite(best_raw_scores).all()
    assert torch.isfinite(
        best_normalized_scores
    ).all()
    return {
        "token_ids": best_tokens.detach().cpu(),
        "raw_scores": (
            best_raw_scores.detach().cpu()
        ),
        "normalized_scores": (
            best_normalized_scores.detach().cpu()
        ),
        "generated_lengths": (
            best_lengths.detach().cpu()
        ),
        "all_beams_finished": bool(
            beam_finished.all()
        ),
    }


@torch.inference_mode()
def run_s2_teacher_forced_validation(
    validation_loader=None,
):
    """Mean token Cross-Entropy over all 25,010 Validation captions."""
    if validation_loader is None:
        validation_loader = (
            build_s2_validation_dataloader()
        )

    S2_ENCODER_ADAPTER_MODEL.eval()
    S2_SHARED_DECODER_MODEL.eval()

    total_negative_log_likelihood = 0.0
    total_valid_tokens = 0
    total_correct_tokens = 0
    total_captions = 0

    for cpu_batch in validation_loader:
        batch = move_batch_to_device(cpu_batch)
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            visual_memory = (
                S2_ENCODER_ADAPTER_MODEL(
                    batch["images"]
                )
            )
            memory_attention_mask = torch.ones(
                visual_memory.shape[:2],
                dtype=torch.bool,
                device=visual_memory.device,
            )
            logits = S2_SHARED_DECODER_MODEL(
                decoder_input_ids=batch[
                    "decoder_input_ids"
                ],
                visual_memory=visual_memory,
                decoder_attention_mask=batch[
                    "decoder_attention_mask"
                ],
                memory_attention_mask=(
                    memory_attention_mask
                ),
            )

        flat_logits = logits.reshape(
            -1,
            VOCABULARY_SIZE,
        ).float()
        flat_labels = batch["labels"].reshape(-1)
        valid_mask = flat_labels != PAD_ID
        valid_token_count = int(
            valid_mask.sum().item()
        )
        loss_sum = F.cross_entropy(
            flat_logits,
            flat_labels,
            ignore_index=PAD_ID,
            reduction="sum",
        )

        predictions = logits.argmax(dim=-1)
        correct_count = int(
            (
                (predictions == batch["labels"])
                & (batch["labels"] != PAD_ID)
            ).sum().item()
        )
        assert torch.isfinite(loss_sum)

        total_negative_log_likelihood += float(
            loss_sum.item()
        )
        total_valid_tokens += valid_token_count
        total_correct_tokens += correct_count
        total_captions += int(
            batch["images"].shape[0]
        )

    assert total_captions == len(
        S2_VALIDATION_DATASET
    )
    assert total_captions == 25_010
    assert total_valid_tokens > 0

    return {
        "caption_count": total_captions,
        "valid_token_count": total_valid_tokens,
        "mean_token_cross_entropy": (
            total_negative_log_likelihood
            / total_valid_tokens
        ),
        "token_accuracy": (
            total_correct_tokens
            / total_valid_tokens
        ),
    }


@torch.inference_mode()
def generate_s2_validation_captions(
    image_loader=None,
    progress_every=25,
):
    """Generate exactly one Beam-3 hypothesis for each Validation image."""
    if image_loader is None:
        image_loader = (
            build_s2_unique_validation_image_dataloader()
        )

    S2_ENCODER_ADAPTER_MODEL.eval()
    S2_SHARED_DECODER_MODEL.eval()

    hypotheses = {}
    generation_records = {}
    started_at = time.perf_counter()

    for batch_index, cpu_batch in enumerate(
        image_loader
    ):
        image_ids = [
            int(value)
            for value
            in cpu_batch["image_ids"].tolist()
        ]
        assert not (
            set(image_ids)
            & set(S2_HELD_OUT_TEST_IMAGE_IDS)
        )
        images = cpu_batch["images"].to(
            device,
            non_blocking=True,
        )
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            visual_memory = (
                S2_ENCODER_ADAPTER_MODEL(images)
            )

        beam_output = (
            s2_beam_search_from_visual_memory(
                visual_memory
            )
        )
        for row_index, image_id in enumerate(
            image_ids
        ):
            token_ids = [
                int(value)
                for value
                in beam_output["token_ids"][
                    row_index
                ].tolist()
            ]
            caption, has_eos, word_count = (
                decode_caption_token_ids(
                    token_ids
                )
            )
            assert caption
            assert word_count >= MINIMUM_LEXICAL_TOKENS
            assert not any(
                token_id
                in FORBIDDEN_GENERATION_TOKEN_IDS
                for token_id in token_ids[1:]
            )
            assert image_id not in hypotheses

            hypotheses[image_id] = caption
            generation_records[image_id] = {
                "caption": caption,
                "token_ids": token_ids,
                "word_count": word_count,
                "has_eos": has_eos,
                "raw_score": float(
                    beam_output["raw_scores"][
                        row_index
                    ].item()
                ),
                "normalized_score": float(
                    beam_output[
                        "normalized_scores"
                    ][row_index].item()
                ),
            }

        if (
            progress_every
            and (batch_index + 1)
            % int(progress_every) == 0
        ):
            print(
                "Generated",
                f"{len(hypotheses):,}/5,000",
                "Validation captions",
            )

    assert len(hypotheses) == 5_000
    assert set(hypotheses) == set(
        S2_VALIDATION_REFERENCES
    )
    assert not (
        set(hypotheses)
        & set(S2_HELD_OUT_TEST_IMAGE_IDS)
    )
    return {
        "hypotheses": hypotheses,
        "records": generation_records,
        "elapsed_seconds": (
            time.perf_counter() - started_at
        ),
    }


def evaluate_s2_validation():
    """
    Production per-epoch Validation entry point.

    Runs teacher-forced CE on 25,010 captions and Beam-3 generation on 5,000
    unique images, then computes the five locked COCO caption metrics.
    """
    teacher_forced = (
        run_s2_teacher_forced_validation()
    )
    generation = (
        generate_s2_validation_captions()
    )
    metrics = compute_coco_caption_metrics(
        S2_VALIDATION_REFERENCES,
        generation["hypotheses"],
    )
    assert set(metrics) == {
        "BLEU-1",
        "BLEU-4",
        "METEOR",
        "ROUGE-L",
        "CIDEr-D",
    }
    return {
        "teacher_forced": teacher_forced,
        "generation": generation,
        "metrics": metrics,
    }


def is_better_s2_validation_result(
    candidate,
    incumbent,
):
    """
    Selection order:
      1. CIDEr-D higher
      2. BLEU-4 higher
      3. Validation CE lower
      4. Earlier epoch
    """
    if incumbent is None:
        return True

    candidate_key = (
        float(candidate["metrics"]["CIDEr-D"]),
        float(candidate["metrics"]["BLEU-4"]),
        -float(
            candidate["teacher_forced"][
                "mean_token_cross_entropy"
            ]
        ),
        -int(candidate["epoch"]),
    )
    incumbent_key = (
        float(incumbent["metrics"]["CIDEr-D"]),
        float(incumbent["metrics"]["BLEU-4"]),
        -float(
            incumbent["teacher_forced"][
                "mean_token_cross_entropy"
            ]
        ),
        -int(incumbent["epoch"]),
    )
    return candidate_key > incumbent_key


print("=" * 72)
print("S2 PRODUCTION VALIDATION / BEAM-3 ENGINE AUDIT")
print("=" * 72)
print()
print("Teacher-forced corpus   : 25,010 Validation captions")
print("Generation corpus       : 5,000 unique Validation images")
print("Beam size               :", BEAM_SIZE)
print("Length penalty          :", LENGTH_PENALTY)
print("Maximum decoder steps   :", MAXIMUM_DECODER_STEPS)
print("Minimum lexical tokens  :", MINIMUM_LEXICAL_TOKENS)
print("Forbidden token IDs     :", list(FORBIDDEN_GENERATION_TOKEN_IDS))
print("Selection primary       : CIDEr-D")
print("Test data read          : False")


# Snapshot every mutable training object before the read-only smoke test.
encoder_state_before = model_state_cpu(
    S2_ENCODER_ADAPTER_MODEL
)
decoder_state_before = model_state_cpu(
    S2_SHARED_DECODER_MODEL
)
optimizer_state_before = cpu_clone_tree(
    S2_OPTIMIZER.state_dict()
)
scheduler_state_before = cpu_clone_tree(
    S2_SCHEDULER.state_dict()
)
scaler_state_before = cpu_clone_tree(
    S2_GRAD_SCALER.state_dict()
)


# Teacher-forced smoke test on one deterministic caption batch.
teacher_forced_smoke_loader = (
    build_s2_validation_dataloader(
        persistent_workers=False
    )
)
teacher_forced_cpu_batch = next(
    iter(teacher_forced_smoke_loader)
)
teacher_forced_batch = move_batch_to_device(
    teacher_forced_cpu_batch
)

S2_ENCODER_ADAPTER_MODEL.eval()
S2_SHARED_DECODER_MODEL.eval()
with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=True,
    ):
        smoke_memory_first = (
            S2_ENCODER_ADAPTER_MODEL(
                teacher_forced_batch["images"]
            )
        )
        smoke_logits_first = (
            S2_SHARED_DECODER_MODEL(
                decoder_input_ids=(
                    teacher_forced_batch[
                        "decoder_input_ids"
                    ]
                ),
                visual_memory=smoke_memory_first,
                decoder_attention_mask=(
                    teacher_forced_batch[
                        "decoder_attention_mask"
                    ]
                ),
                memory_attention_mask=torch.ones(
                    smoke_memory_first.shape[:2],
                    dtype=torch.bool,
                    device=device,
                ),
            )
        )
        smoke_memory_second = (
            S2_ENCODER_ADAPTER_MODEL(
                teacher_forced_batch["images"]
            )
        )
        smoke_logits_second = (
            S2_SHARED_DECODER_MODEL(
                decoder_input_ids=(
                    teacher_forced_batch[
                        "decoder_input_ids"
                    ]
                ),
                visual_memory=smoke_memory_second,
                decoder_attention_mask=(
                    teacher_forced_batch[
                        "decoder_attention_mask"
                    ]
                ),
                memory_attention_mask=torch.ones(
                    smoke_memory_second.shape[:2],
                    dtype=torch.bool,
                    device=device,
                ),
            )
        )

teacher_forced_smoke_loss = F.cross_entropy(
    smoke_logits_first.reshape(
        -1,
        VOCABULARY_SIZE,
    ).float(),
    teacher_forced_batch["labels"].reshape(-1),
    ignore_index=PAD_ID,
)
teacher_forced_repeat_error = float(
    (
        smoke_logits_first.float()
        - smoke_logits_second.float()
    ).abs().max().item()
)
assert torch.isfinite(teacher_forced_smoke_loss)
assert teacher_forced_repeat_error == 0.0


# Beam smoke test on two unique images, repeated exactly.
beam_smoke_loader = (
    build_s2_unique_validation_image_dataloader(
        batch_size=2,
        num_workers=0,
        persistent_workers=False,
    )
)
beam_smoke_batch = next(iter(beam_smoke_loader))
beam_smoke_image_ids = [
    int(value)
    for value in beam_smoke_batch[
        "image_ids"
    ].tolist()
]
assert len(set(beam_smoke_image_ids)) == 2
assert not (
    set(beam_smoke_image_ids)
    & set(S2_HELD_OUT_TEST_IMAGE_IDS)
)

with torch.inference_mode():
    beam_smoke_images = beam_smoke_batch[
        "images"
    ].to(device)
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=True,
    ):
        beam_smoke_memory = (
            S2_ENCODER_ADAPTER_MODEL(
                beam_smoke_images
            )
        )
    beam_first = (
        s2_beam_search_from_visual_memory(
            beam_smoke_memory
        )
    )
    beam_second = (
        s2_beam_search_from_visual_memory(
            beam_smoke_memory
        )
    )

assert torch.equal(
    beam_first["token_ids"],
    beam_second["token_ids"],
)
assert torch.equal(
    beam_first["raw_scores"],
    beam_second["raw_scores"],
)
assert torch.equal(
    beam_first["normalized_scores"],
    beam_second["normalized_scores"],
)

beam_smoke_captions = {}
beam_smoke_records = []
for row_index, image_id in enumerate(
    beam_smoke_image_ids
):
    token_ids = [
        int(value)
        for value
        in beam_first["token_ids"][
            row_index
        ].tolist()
    ]
    caption, has_eos, word_count = (
        decode_caption_token_ids(token_ids)
    )
    assert caption
    assert word_count >= MINIMUM_LEXICAL_TOKENS
    assert not any(
        token_id
        in FORBIDDEN_GENERATION_TOKEN_IDS
        for token_id in token_ids[1:]
    )
    beam_smoke_captions[image_id] = caption
    beam_smoke_records.append(
        {
            "image_id": image_id,
            "caption": caption,
            "word_count": word_count,
            "has_eos": has_eos,
            "generated_length": int(
                beam_first[
                    "generated_lengths"
                ][row_index].item()
            ),
            "raw_score": float(
                beam_first["raw_scores"][
                    row_index
                ].item()
            ),
            "normalized_score": float(
                beam_first[
                    "normalized_scores"
                ][row_index].item()
            ),
        }
    )


# Explicitly audit the four-level selection order.
selection_base = {
    "epoch": 3,
    "metrics": {
        "CIDEr-D": 0.50,
        "BLEU-4": 0.20,
    },
    "teacher_forced": {
        "mean_token_cross_entropy": 3.00,
    },
}
selection_cider = {
    **selection_base,
    "epoch": 9,
    "metrics": {
        "CIDEr-D": 0.51,
        "BLEU-4": 0.01,
    },
}
selection_bleu = {
    **selection_base,
    "epoch": 9,
    "metrics": {
        "CIDEr-D": 0.50,
        "BLEU-4": 0.21,
    },
}
selection_ce = {
    **selection_base,
    "epoch": 9,
    "teacher_forced": {
        "mean_token_cross_entropy": 2.99,
    },
}
selection_later = {
    **selection_base,
    "epoch": 4,
}
assert is_better_s2_validation_result(
    selection_base,
    None,
)
assert is_better_s2_validation_result(
    selection_cider,
    selection_base,
)
assert is_better_s2_validation_result(
    selection_bleu,
    selection_base,
)
assert is_better_s2_validation_result(
    selection_ce,
    selection_base,
)
assert not is_better_s2_validation_result(
    selection_later,
    selection_base,
)


# Verify that read-only Validation changed no training state.
encoder_state_unchanged = trees_exact(
    encoder_state_before,
    S2_ENCODER_ADAPTER_MODEL.state_dict(),
)
decoder_state_unchanged = trees_exact(
    decoder_state_before,
    S2_SHARED_DECODER_MODEL.state_dict(),
)
optimizer_state_unchanged = trees_exact(
    optimizer_state_before,
    S2_OPTIMIZER.state_dict(),
)
scheduler_state_unchanged = trees_exact(
    scheduler_state_before,
    S2_SCHEDULER.state_dict(),
)
scaler_state_unchanged = trees_exact(
    scaler_state_before,
    S2_GRAD_SCALER.state_dict(),
)
assert encoder_state_unchanged
assert decoder_state_unchanged
assert optimizer_state_unchanged
assert scheduler_state_unchanged
assert scaler_state_unchanged

print()
print("Teacher-forced smoke")
print("Batch captions          :", int(
    teacher_forced_batch["images"].shape[0]
))
print("Valid tokens            :", int(
    (
        teacher_forced_batch["labels"]
        != PAD_ID
    ).sum().item()
))
print("Cross-entropy           :", float(
    teacher_forced_smoke_loss.item()
))
print("Repeat maximum error    :", teacher_forced_repeat_error)
print()
print("Beam-3 smoke")
print("Unique images           :", len(beam_smoke_image_ids))
print("Token IDs repeat exact  : True")
print("Scores repeat exact     : True")
for record in beam_smoke_records:
    print(
        f"image_id={record['image_id']} | "
        f"words={record['word_count']} | "
        f"EOS={record['has_eos']} | "
        f"score={record['normalized_score']:.6f} | "
        f"caption={record['caption']}"
    )
print()
print("Checkpoint selection    : 4-level order PASSED")
print("Training state unchanged: True")


S2_CELL09_ENGINE_CONTRACT_PATH = (
    RUN_CONFIG_DIR
    / "s2_production_validation_and_beam_engine_contract.json"
)
S2_CELL09_AUDIT_PATH = (
    RUN_AUDIT_DIR
    / "s2_cell09_production_validation_and_beam_engine_audit.json"
)
S2_PRODUCTION_ENGINE_READY_PATH = (
    Path(RUN_ROOT)
    / "S2_SEED2026_PRODUCTION_ENGINE_READY.json"
)

cell_elapsed_seconds = (
    time.perf_counter() - CELL_STARTED_AT
)

engine_contract = {
    "contract_version": (
        "phase2_s2_production_validation_beam_engine_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "locked",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "teacher_forced_validation": {
        "frequency": "every epoch",
        "caption_count": 25_010,
        "loss": "mean token Cross-Entropy",
        "augmentation": False,
        "entry_point": (
            "run_s2_teacher_forced_validation"
        ),
    },
    "generation_validation": {
        "frequency": "every epoch",
        "image_count": 5_000,
        "one_hypothesis_per_image": True,
        "entry_point": (
            "generate_s2_validation_captions"
        ),
        "beam_size": BEAM_SIZE,
        "length_penalty": LENGTH_PENALTY,
        "score_normalization": (
            "raw_log_probability / "
            "generated_length ** length_penalty"
        ),
        "generated_length_includes_eos": True,
        "maximum_decoder_steps": (
            MAXIMUM_DECODER_STEPS
        ),
        "minimum_lexical_tokens_before_eos": (
            MINIMUM_LEXICAL_TOKENS
        ),
        "forbidden_generation_token_ids": list(
            FORBIDDEN_GENERATION_TOKEN_IDS
        ),
        "eos_id": EOS_ID,
    },
    "metrics": {
        "entry_point": (
            "compute_coco_caption_metrics"
        ),
        "names": [
            "BLEU-1",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
            "CIDEr-D",
        ],
        "references_path": str(
            S2_VALIDATION_REFERENCES_PATH
        ),
        "references_sha256": (
            S2_VALIDATION_REFERENCES_SHA256
        ),
    },
    "selection": {
        "primary": "CIDEr-D higher",
        "tie_break_order": [
            "BLEU-4 higher",
            "validation Cross-Entropy lower",
            "earlier epoch",
        ],
        "entry_point": (
            "is_better_s2_validation_result"
        ),
    },
    "policy": {
        "validation_only_selection": True,
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_images_read": False,
        "test_predictions_written": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "optimizer_step_performed": False,
        "scheduler_advanced": False,
        "full_training_started": False,
    },
}
write_json(
    S2_CELL09_ENGINE_CONTRACT_PATH,
    engine_contract,
)

cell_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "student_id": STUDENT_ID,
    "teacher_forced_smoke": {
        "captions": int(
            teacher_forced_batch["images"].shape[0]
        ),
        "valid_tokens": int(
            (
                teacher_forced_batch["labels"]
                != PAD_ID
            ).sum().item()
        ),
        "cross_entropy": float(
            teacher_forced_smoke_loss.item()
        ),
        "repeat_maximum_error": (
            teacher_forced_repeat_error
        ),
    },
    "beam_smoke": {
        "images": len(beam_smoke_image_ids),
        "beam_size": BEAM_SIZE,
        "token_ids_repeat_exact": True,
        "scores_repeat_exact": True,
        "records": beam_smoke_records,
    },
    "selection_order_passed": True,
    "encoder_state_unchanged": (
        encoder_state_unchanged
    ),
    "decoder_state_unchanged": (
        decoder_state_unchanged
    ),
    "optimizer_state_unchanged": (
        optimizer_state_unchanged
    ),
    "scheduler_state_unchanged": (
        scheduler_state_unchanged
    ),
    "grad_scaler_state_unchanged": (
        scaler_state_unchanged
    ),
    "test_parquet_opened": False,
    "test_caption_read": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "optimizer_step_performed": False,
    "full_training_started": False,
    "elapsed_seconds": cell_elapsed_seconds,
}
write_json(
    S2_CELL09_AUDIT_PATH,
    cell_audit,
)

S2_CELL09_ENGINE_CONTRACT_SHA256 = (
    sha256_file(
        S2_CELL09_ENGINE_CONTRACT_PATH
    )
)
S2_CELL09_AUDIT_SHA256 = sha256_file(
    S2_CELL09_AUDIT_PATH
)

production_engine_ready = {
    "marker_version": (
        "phase2_s2_production_engine_ready_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "ready",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "teacher_forced_validation_ready": True,
    "beam3_generation_ready": True,
    "official_metrics_ready": True,
    "validation_only_selection_ready": True,
    "training_state_unchanged": True,
    "test_set_status": "sealed",
    "test_set_used": False,
    "full_training_started": False,
    "contract": {
        "path": str(
            S2_CELL09_ENGINE_CONTRACT_PATH
        ),
        "sha256": (
            S2_CELL09_ENGINE_CONTRACT_SHA256
        ),
    },
    "audit": {
        "path": str(S2_CELL09_AUDIT_PATH),
        "sha256": S2_CELL09_AUDIT_SHA256,
    },
}
write_json(
    S2_PRODUCTION_ENGINE_READY_PATH,
    production_engine_ready,
)
S2_PRODUCTION_ENGINE_READY_SHA256 = (
    sha256_file(
        S2_PRODUCTION_ENGINE_READY_PATH
    )
)


print()
print("=" * 72)
print("PHASE 2 S2 CELL 9 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Teacher-forced Validation : READY")
print("Validation captions       : 25010")
print("Unique-image generation   : READY")
print("Validation images         : 5000")
print("Beam size / penalty       : 3 / 0.7")
print("Maximum decoder steps     : 31")
print("Minimum lexical tokens    : 2")
print("Forbidden token IDs       : [0, 1, 3]")
print("COCO metrics              : 5 / 5 WIRED")
print("Checkpoint primary        : CIDEr-D")
print("Tie-break order           : BLEU-4 / CE / earlier epoch")
print("Teacher-forced smoke      : PASSED")
print("Beam repeatability        : PASSED")
print("Selection audit           : PASSED")
print("Encoder state unchanged   :", encoder_state_unchanged)
print("Decoder state unchanged   :", decoder_state_unchanged)
print("Optimizer state unchanged :", optimizer_state_unchanged)
print("Scheduler state unchanged :", scheduler_state_unchanged)
print("GradScaler state unchanged:", scaler_state_unchanged)
print("Optimizer step performed  : False")
print("Test Parquet opened       : False")
print("Test caption read         : False")
print("Test images read          : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Full training started     : False")
print("Contract                  :", S2_CELL09_ENGINE_CONTRACT_PATH)
print("Contract SHA-256          :", S2_CELL09_ENGINE_CONTRACT_SHA256)
print("Audit                     :", S2_CELL09_AUDIT_PATH)
print("Audit SHA-256             :", S2_CELL09_AUDIT_SHA256)
print("READY marker              :", S2_PRODUCTION_ENGINE_READY_PATH)
print("READY SHA-256             :", S2_PRODUCTION_ENGINE_READY_SHA256)
print(
    "Elapsed                   :",
    round(cell_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : define the production epoch trainer, "
    "atomic last/best checkpoints, CSV logs, early stopping, and run "
    "a discarded short production pilot"
)

## Cell 10 — Production Trainer và Discarded Pilot

Cell này thực hiện:

- Xây dựng production epoch trainer.
- Tích lũy gradient theo tổng valid-token count.
- Dùng FP16-safe backward divisor bằng 1024.
- Hỗ trợ atomic last/best checkpoint.
- Hỗ trợ resume tại optimizer-step boundary và giữa epoch.
- Ghi optimizer-step JSONL và epoch-history CSV.
- Lưu Validation predictions và metrics theo epoch.
- Khóa early stopping:
  - epoch tối thiểu = 5;
  - patience = 4;
  - CIDEr-D minimum delta = 0.001.
- Chạy pilot gồm 4 optimizer steps, 8 microbatch và 256 ảnh.
- Kiểm tra checkpoint/resume chính xác.
- Khôi phục trạng thái ban đầu và xóa toàn bộ pilot artifacts.
- Không chạy full Validation hoặc truy cập Held-out Test.

Pilot này bị loại bỏ và không được tính là confirmation training.

In [ ]:
import csv
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import torch


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID",
    "STUDENT_NAME",
    "DEVELOPMENT_SEED",
    "RUN_ROOT",
    "RUN_CONFIG_DIR",
    "RUN_AUDIT_DIR",
    "RUN_CHECKPOINT_DIR",
    "device",
    "training_contract",
    "validation_policy",
    "S2_ENCODER_ADAPTER_MODEL",
    "S2_SHARED_DECODER_MODEL",
    "S2_OPTIMIZER",
    "S2_SCHEDULER",
    "S2_GRAD_SCALER",
    "TOTAL_OPTIMIZER_STEPS",
    "WARMUP_OPTIMIZER_STEPS",
    "MINIMUM_LR_FACTOR",
    "build_s2_train_dataloader",
    "set_s2_training_mode",
    "move_batch_to_device",
    "forward_token_loss",
    "gradients_finite",
    "atomic_torch_save",
    "model_state_cpu",
    "cpu_clone_tree",
    "trees_exact",
    "capture_rng_state",
    "restore_rng_state",
    "evaluate_s2_validation",
    "is_better_s2_validation_result",
    "S2_PRODUCTION_ENGINE_READY_PATH",
    "S2_PRODUCTION_ENGINE_READY_SHA256",
    "S2_CELL09_ENGINE_CONTRACT_PATH",
    "S2_CELL09_ENGINE_CONTRACT_SHA256",
    "S2_CELL09_AUDIT_PATH",
    "S2_CELL09_AUDIT_SHA256",
    "S2_ENCODER_INITIAL_CHECKPOINT_PATH",
    "SHARED_DECODER_INITIAL_CHECKPOINT_PATH",
    "actual_encoder_checkpoint_sha256",
    "actual_decoder_checkpoint_sha256",
    "read_json",
    "write_json",
    "sha256_file",
]
missing_runtime_names = [
    name for name in required_runtime_names if name not in globals()
]
assert not missing_runtime_names, (
    "Thiếu biến từ S2 Cell 1–9: "
    f"{missing_runtime_names}. Hãy chạy lại các cell trước."
)

assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026

engine_ready = read_json(
    S2_PRODUCTION_ENGINE_READY_PATH
)
assert engine_ready["status"] == "ready"
assert engine_ready[
    "teacher_forced_validation_ready"
] is True
assert engine_ready[
    "beam3_generation_ready"
] is True
assert engine_ready[
    "official_metrics_ready"
] is True
assert engine_ready[
    "training_state_unchanged"
] is True
assert engine_ready["test_set_status"] == "sealed"
assert engine_ready["test_set_used"] is False
assert (
    sha256_file(S2_PRODUCTION_ENGINE_READY_PATH)
    == S2_PRODUCTION_ENGINE_READY_SHA256
)
assert (
    sha256_file(S2_CELL09_ENGINE_CONTRACT_PATH)
    == S2_CELL09_ENGINE_CONTRACT_SHA256
)
assert (
    sha256_file(S2_CELL09_AUDIT_PATH)
    == S2_CELL09_AUDIT_SHA256
)


batching_contract = training_contract["batching"]
optimization_contract = training_contract[
    "optimization"
]
duration_contract = training_contract[
    "training_duration"
]
early_stopping_contract = duration_contract[
    "early_stopping"
]
mixed_precision_contract = optimization_contract[
    "mixed_precision"
]

PHYSICAL_BATCH_SIZE = int(
    batching_contract["physical_batch_size"]
)
GRADIENT_ACCUMULATION_STEPS = int(
    batching_contract[
        "gradient_accumulation_steps"
    ]
)
EFFECTIVE_BATCH_SIZE = int(
    batching_contract["effective_batch_size"]
)
GRADIENT_CLIP_NORM = float(
    optimization_contract["gradient_clip_norm"]
)
MAXIMUM_OVERFLOW_RETRIES = int(
    mixed_precision_contract[
        "maximum_overflow_retries_per_run"
    ]
)
MAXIMUM_EPOCHS = int(
    duration_contract["maximum_epochs"]
)
MINIMUM_EPOCHS_BEFORE_EARLY_STOP = int(
    duration_contract[
        "minimum_epochs_before_early_stop"
    ]
)
EARLY_STOPPING_PATIENCE = int(
    early_stopping_contract[
        "patience_metric_evaluations"
    ]
)
EARLY_STOPPING_MINIMUM_DELTA = float(
    early_stopping_contract["minimum_delta"]
)
# Backpropagating the raw token-sum NLL and then applying an FP16 GradScaler
# of 1024 can overflow before gradients are unscaled. Divide by a fixed,
# checkpointed constant during backward, then multiply unscaled gradients by
# the same constant / valid-token-count. The final gradient is still exactly
# the gradient of mean token NLL:
#
#   grad(loss_sum / D) * (D / valid_tokens)
#       = grad(loss_sum) / valid_tokens
TOKEN_SUM_BACKWARD_DIVISOR = 1_024.0

assert PHYSICAL_BATCH_SIZE == 32
assert GRADIENT_ACCUMULATION_STEPS == 2
assert EFFECTIVE_BATCH_SIZE == 64
assert GRADIENT_CLIP_NORM == 1.0
assert MAXIMUM_OVERFLOW_RETRIES == 10
assert MAXIMUM_EPOCHS == 20
assert MINIMUM_EPOCHS_BEFORE_EARLY_STOP == 5
assert EARLY_STOPPING_PATIENCE == 4
assert EARLY_STOPPING_MINIMUM_DELTA == 0.001
assert TOKEN_SUM_BACKWARD_DIVISOR == 1_024.0
assert TOTAL_OPTIMIZER_STEPS == 35_420
assert WARMUP_OPTIMIZER_STEPS == 1_771
assert MINIMUM_LR_FACTOR == 0.05


S2_TRAINING_LOG_DIR = (
    Path(RUN_ROOT) / "logs" / "full_training"
)
S2_TRAINING_LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
S2_TRAINING_HISTORY_CSV_PATH = (
    S2_TRAINING_LOG_DIR
    / "s2_seed2026_epoch_history.csv"
)
S2_TRAINING_STEP_JSONL_PATH = (
    S2_TRAINING_LOG_DIR
    / "s2_seed2026_optimizer_steps.jsonl"
)
S2_VALIDATION_RESULT_DIR = (
    Path(RUN_ROOT) / "validation" / "epochs"
)
S2_VALIDATION_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
S2_LAST_CHECKPOINT_PATH = (
    Path(RUN_CHECKPOINT_DIR)
    / "s2_mobilevit_s_seed2026_last.pt"
)
S2_BEST_CHECKPOINT_PATH = (
    Path(RUN_CHECKPOINT_DIR)
    / "s2_mobilevit_s_seed2026_best.pt"
)


def atomic_write_text(target_path, text):
    target_path = Path(target_path)
    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    temporary_path = target_path.with_suffix(
        target_path.suffix + ".tmp"
    )
    if temporary_path.exists():
        temporary_path.unlink()
    try:
        with temporary_path.open(
            "w",
            encoding="utf-8",
            newline="",
        ) as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary_path, target_path)
    finally:
        if temporary_path.exists():
            temporary_path.unlink()
    assert target_path.is_file()


def append_jsonl_durable(target_path, record):
    target_path = Path(target_path)
    target_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    new_line = json.dumps(
        record,
        ensure_ascii=False,
        sort_keys=True,
    )
    with target_path.open(
        "a",
        encoding="utf-8",
        newline="",
    ) as handle:
        handle.write(new_line + "\n")
        handle.flush()
        os.fsync(handle.fileno())


EPOCH_HISTORY_FIELDNAMES = [
    "epoch",
    "optimizer_steps",
    "samples",
    "valid_tokens",
    "train_mean_token_ce",
    "validation_mean_token_ce",
    "validation_token_accuracy",
    "bleu_1",
    "bleu_4",
    "meteor",
    "rouge_l",
    "cider_d",
    "is_best",
    "epochs_without_significant_improvement",
    "epoch_elapsed_seconds",
]


def write_epoch_history_csv_atomic(
    history_rows,
    target_path=S2_TRAINING_HISTORY_CSV_PATH,
):
    from io import StringIO

    buffer = StringIO()
    writer = csv.DictWriter(
        buffer,
        fieldnames=EPOCH_HISTORY_FIELDNAMES,
        extrasaction="raise",
    )
    writer.writeheader()
    for row in history_rows:
        writer.writerow(row)
    atomic_write_text(
        target_path,
        buffer.getvalue(),
    )


def named_trainable_parameters():
    return [
        (f"encoder.{name}", parameter)
        for name, parameter
        in S2_ENCODER_ADAPTER_MODEL.named_parameters()
        if parameter.requires_grad
    ] + [
        (f"decoder.{name}", parameter)
        for name, parameter
        in S2_SHARED_DECODER_MODEL.named_parameters()
        if parameter.requires_grad
    ]


def current_learning_rates():
    return {
        str(group["name"]): float(
            group["lr"]
        )
        for group in S2_OPTIMIZER.param_groups
    }


def build_s2_checkpoint_payload(
    progress,
    best_validation_result,
    epoch_history,
):
    assert int(
        progress["microbatch_in_accumulation"]
    ) == 0
    assert progress["checkpoint_boundary"] == (
        "optimizer_step"
    )
    return {
        "schema_version": (
            "phase2_s2_full_training_checkpoint_v1"
        ),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "student_id": STUDENT_ID,
        "student_name": STUDENT_NAME,
        "seed": DEVELOPMENT_SEED,
        "progress": dict(progress),
        "encoder_state_dict": model_state_cpu(
            S2_ENCODER_ADAPTER_MODEL
        ),
        "decoder_state_dict": model_state_cpu(
            S2_SHARED_DECODER_MODEL
        ),
        "optimizer_state_dict": cpu_clone_tree(
            S2_OPTIMIZER.state_dict()
        ),
        "scheduler_state_dict": cpu_clone_tree(
            S2_SCHEDULER.state_dict()
        ),
        "grad_scaler_state_dict": cpu_clone_tree(
            S2_GRAD_SCALER.state_dict()
        ),
        "rng_state": capture_rng_state(),
        "best_validation_result": (
            best_validation_result
        ),
        "epoch_history": list(epoch_history),
        "contracts_and_checksums": {
            "production_engine": {
                "path": str(
                    S2_CELL09_ENGINE_CONTRACT_PATH
                ),
                "sha256": (
                    S2_CELL09_ENGINE_CONTRACT_SHA256
                ),
            },
            "encoder_initial": {
                "path": str(
                    S2_ENCODER_INITIAL_CHECKPOINT_PATH
                ),
                "sha256": (
                    actual_encoder_checkpoint_sha256
                ),
            },
            "decoder_initial": {
                "path": str(
                    SHARED_DECODER_INITIAL_CHECKPOINT_PATH
                ),
                "sha256": (
                    actual_decoder_checkpoint_sha256
                ),
            },
        },
        "data_policy": {
            "selection_split": (
                "Karpathy validation"
            ),
            "test_parquet_opened": False,
            "test_caption_read": False,
            "test_images_read": False,
            "test_metrics_computed": False,
            "test_set_status": "sealed",
            "test_set_used": False,
            "flickr_used": False,
        },
    }


def save_s2_training_checkpoint(
    target_path,
    progress,
    best_validation_result,
    epoch_history,
):
    payload = build_s2_checkpoint_payload(
        progress=progress,
        best_validation_result=(
            best_validation_result
        ),
        epoch_history=epoch_history,
    )
    atomic_torch_save(payload, target_path)
    return {
        "path": str(target_path),
        "size_bytes": int(
            Path(target_path).stat().st_size
        ),
        "sha256": sha256_file(target_path),
    }


def load_s2_training_checkpoint(
    checkpoint_path,
):
    checkpoint_path = Path(checkpoint_path)
    assert checkpoint_path.is_file()
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )
    assert checkpoint["schema_version"] == (
        "phase2_s2_full_training_checkpoint_v1"
    )
    assert checkpoint["student_id"] == "S2"
    assert int(checkpoint["seed"]) == DEVELOPMENT_SEED
    assert checkpoint["data_policy"][
        "test_set_status"
    ] == "sealed"
    assert checkpoint["data_policy"][
        "test_set_used"
    ] is False
    assert checkpoint["progress"][
        "checkpoint_boundary"
    ] == "optimizer_step"
    assert int(
        checkpoint["progress"][
            "microbatch_in_accumulation"
        ]
    ) == 0

    S2_ENCODER_ADAPTER_MODEL.load_state_dict(
        checkpoint["encoder_state_dict"],
        strict=True,
    )
    S2_SHARED_DECODER_MODEL.load_state_dict(
        checkpoint["decoder_state_dict"],
        strict=True,
    )
    S2_OPTIMIZER.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )
    S2_SCHEDULER.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )
    S2_GRAD_SCALER.load_state_dict(
        checkpoint["grad_scaler_state_dict"]
    )
    restore_rng_state(checkpoint["rng_state"])
    S2_OPTIMIZER.zero_grad(set_to_none=True)
    return checkpoint


def train_s2_epoch(
    epoch,
    start_position=0,
    starting_global_optimizer_step=0,
    starting_overflow_retry_count=0,
    maximum_successful_optimizer_steps=None,
    step_log_path=None,
    periodic_checkpoint_callback=None,
    checkpoint_every_successful_steps=250,
    progress_print_every=100,
):
    """
    Train one epoch, or a bounded prefix for a pilot.

    Gradient accumulation is token-balanced: each microbatch contributes its
    summed token NLL, then unscaled gradients are divided by the total number
    of valid target tokens in the accumulation window before clipping.
    """
    epoch = int(epoch)
    start_position = int(start_position)
    global_optimizer_step = int(
        starting_global_optimizer_step
    )
    overflow_retry_count = int(
        starting_overflow_retry_count
    )
    maximum_successful_optimizer_steps = (
        None
        if maximum_successful_optimizer_steps
        is None
        else int(maximum_successful_optimizer_steps)
    )

    assert 0 <= epoch < MAXIMUM_EPOCHS
    assert start_position >= 0
    assert (
        start_position == 0
        or start_position % EFFECTIVE_BATCH_SIZE
        == 0
    )
    if maximum_successful_optimizer_steps is not None:
        assert maximum_successful_optimizer_steps > 0
    checkpoint_every_successful_steps = int(
        checkpoint_every_successful_steps
    )
    assert checkpoint_every_successful_steps > 0

    train_loader, loader_info = (
        build_s2_train_dataloader(
            epoch=epoch,
            start_position=start_position,
        )
    )
    set_s2_training_mode(
        S2_ENCODER_ADAPTER_MODEL,
        S2_SHARED_DECODER_MODEL,
    )
    S2_OPTIMIZER.zero_grad(set_to_none=True)

    epoch_started_at = time.perf_counter()
    total_loss_sum = 0.0
    total_valid_tokens = 0
    total_samples = 0
    successful_steps_this_call = 0
    skipped_overflow_steps = 0
    accumulation_microbatches = 0
    accumulation_valid_tokens = 0
    accumulation_loss_sum = 0.0
    accumulation_samples = 0
    last_gradient_norm = None
    last_scale = float(
        S2_GRAD_SCALER.get_scale()
    )
    reached_requested_limit = False

    microbatch_count = len(train_loader)
    for microbatch_index, cpu_batch in enumerate(
        train_loader
    ):
        batch = move_batch_to_device(cpu_batch)
        batch_size = int(batch["images"].shape[0])

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            mean_loss, valid_tokens = (
                forward_token_loss(batch)
            )
            loss_sum = (
                mean_loss
                * int(valid_tokens)
            )
            backward_loss = (
                loss_sum
                / TOKEN_SUM_BACKWARD_DIVISOR
            )

        assert torch.isfinite(loss_sum)
        assert torch.isfinite(backward_loss)
        S2_GRAD_SCALER.scale(
            backward_loss
        ).backward()

        loss_sum_value = float(
            loss_sum.detach().item()
        )
        total_loss_sum += loss_sum_value
        total_valid_tokens += int(valid_tokens)
        total_samples += batch_size
        accumulation_loss_sum += loss_sum_value
        accumulation_valid_tokens += int(
            valid_tokens
        )
        accumulation_samples += batch_size
        accumulation_microbatches += 1

        is_last_microbatch = (
            microbatch_index + 1
            == microbatch_count
        )
        accumulation_boundary = (
            accumulation_microbatches
            == GRADIENT_ACCUMULATION_STEPS
            or is_last_microbatch
        )
        if not accumulation_boundary:
            continue

        assert accumulation_valid_tokens > 0
        S2_GRAD_SCALER.unscale_(S2_OPTIMIZER)

        named_parameters = (
            named_trainable_parameters()
        )
        missing_gradients, nonfinite_gradients = (
            gradients_finite(named_parameters)
        )
        assert not missing_gradients
        scale_before = float(
            S2_GRAD_SCALER.get_scale()
        )

        step_succeeded = not nonfinite_gradients
        if step_succeeded:
            token_mean_gradient_factor = (
                TOKEN_SUM_BACKWARD_DIVISOR
                / float(accumulation_valid_tokens)
            )
            for _, parameter in named_parameters:
                if parameter.grad is not None:
                    parameter.grad.mul_(
                        token_mean_gradient_factor
                    )

            gradient_norm_tensor = (
                torch.nn.utils.clip_grad_norm_(
                    [
                        parameter
                        for _, parameter
                        in named_parameters
                    ],
                    max_norm=GRADIENT_CLIP_NORM,
                    error_if_nonfinite=True,
                )
            )
            last_gradient_norm = float(
                gradient_norm_tensor.item()
            )
            assert math.isfinite(
                last_gradient_norm
            )

        S2_GRAD_SCALER.step(S2_OPTIMIZER)
        S2_GRAD_SCALER.update()
        scale_after = float(
            S2_GRAD_SCALER.get_scale()
        )
        last_scale = scale_after

        if scale_after < scale_before:
            step_succeeded = False

        if step_succeeded:
            S2_SCHEDULER.step()
            global_optimizer_step += 1
            successful_steps_this_call += 1
        else:
            overflow_retry_count += 1
            skipped_overflow_steps += 1
            assert (
                overflow_retry_count
                <= MAXIMUM_OVERFLOW_RETRIES
            ), (
                "Vượt quá giới hạn FP16 overflow retries."
            )

        S2_OPTIMIZER.zero_grad(set_to_none=True)

        step_record = {
            "created_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "epoch": epoch,
            "global_optimizer_step": (
                global_optimizer_step
            ),
            "successful_step": step_succeeded,
            "samples_in_window": (
                accumulation_samples
            ),
            "valid_tokens_in_window": (
                accumulation_valid_tokens
            ),
            "mean_token_cross_entropy": (
                accumulation_loss_sum
                / accumulation_valid_tokens
            ),
            "gradient_norm_before_clip": (
                last_gradient_norm
            ),
            "grad_scaler_scale_before": (
                scale_before
            ),
            "grad_scaler_scale_after": (
                scale_after
            ),
            "learning_rates": (
                current_learning_rates()
            ),
            "global_sample_cursor": (
                start_position + total_samples
            ),
            "test_set_used": False,
        }
        if step_log_path is not None:
            append_jsonl_durable(
                step_log_path,
                step_record,
            )

        if (
            step_succeeded
            and periodic_checkpoint_callback
            is not None
            and global_optimizer_step
            % checkpoint_every_successful_steps
            == 0
        ):
            periodic_checkpoint_callback(
                {
                    "epoch": epoch,
                    "optimizer_step": (
                        global_optimizer_step
                    ),
                    "global_sample_cursor": (
                        start_position
                        + total_samples
                    ),
                    "microbatch_in_accumulation": 0,
                    "overflow_retry_count": (
                        overflow_retry_count
                    ),
                    "checkpoint_boundary": (
                        "optimizer_step"
                    ),
                }
            )

        accumulation_microbatches = 0
        accumulation_valid_tokens = 0
        accumulation_loss_sum = 0.0
        accumulation_samples = 0

        if (
            progress_print_every
            and successful_steps_this_call > 0
            and successful_steps_this_call
            % int(progress_print_every) == 0
        ):
            print(
                f"epoch={epoch + 1:02d} | "
                f"step={global_optimizer_step:05d} | "
                f"cursor={start_position + total_samples:,} | "
                f"loss={step_record['mean_token_cross_entropy']:.6f} | "
                f"grad={last_gradient_norm:.4f}"
            )

        if (
            maximum_successful_optimizer_steps
            is not None
            and successful_steps_this_call
            >= maximum_successful_optimizer_steps
        ):
            reached_requested_limit = True
            break

    assert accumulation_microbatches == 0
    assert total_valid_tokens > 0
    assert total_samples > 0
    S2_OPTIMIZER.zero_grad(set_to_none=True)
    torch.cuda.synchronize(device)

    next_cursor = start_position + total_samples
    epoch_complete = (
        next_cursor >= 113_287
    )
    if epoch_complete:
        next_epoch = epoch + 1
        next_cursor = 0
    else:
        next_epoch = epoch
        assert (
            next_cursor % EFFECTIVE_BATCH_SIZE
            == 0
        )

    return {
        "epoch": epoch,
        "next_epoch": next_epoch,
        "next_global_sample_cursor": (
            next_cursor
        ),
        "global_optimizer_step": (
            global_optimizer_step
        ),
        "overflow_retry_count": (
            overflow_retry_count
        ),
        "optimizer_steps": (
            successful_steps_this_call
        ),
        "skipped_overflow_steps": (
            skipped_overflow_steps
        ),
        "samples": total_samples,
        "valid_tokens": total_valid_tokens,
        "mean_token_cross_entropy": (
            total_loss_sum
            / total_valid_tokens
        ),
        "last_gradient_norm_before_clip": (
            last_gradient_norm
        ),
        "grad_scaler_scale": last_scale,
        "epoch_complete": epoch_complete,
        "reached_requested_limit": (
            reached_requested_limit
        ),
        "elapsed_seconds": (
            time.perf_counter()
            - epoch_started_at
        ),
        "loader_info": loader_info,
    }


def compact_validation_result(
    epoch,
    validation_result,
):
    return {
        "epoch": int(epoch),
        "teacher_forced": dict(
            validation_result[
                "teacher_forced"
            ]
        ),
        "metrics": dict(
            validation_result["metrics"]
        ),
        "generation_elapsed_seconds": float(
            validation_result[
                "generation"
            ]["elapsed_seconds"]
        ),
    }


def persist_s2_validation_result(
    epoch,
    validation_result,
):
    compact_result = compact_validation_result(
        epoch,
        validation_result,
    )
    hypotheses = (
        validation_result["generation"][
            "hypotheses"
        ]
    )
    records = (
        validation_result["generation"][
            "records"
        ]
    )
    payload = {
        **compact_result,
        "hypotheses": {
            str(image_id): caption
            for image_id, caption
            in sorted(hypotheses.items())
        },
        "generation_records": {
            str(image_id): record
            for image_id, record
            in sorted(records.items())
        },
        "data_policy": {
            "split": "Karpathy validation",
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    }
    target_path = (
        S2_VALIDATION_RESULT_DIR
        / f"epoch_{int(epoch):02d}_validation.json"
    )
    write_json(target_path, payload)
    return {
        "path": str(target_path),
        "sha256": sha256_file(target_path),
    }


def run_s2_full_training(
    resume_checkpoint_path=None,
):
    """
    Production entry point for Cell 11.

    A Kaggle interruption can be resumed by passing the existing last
    checkpoint. Checkpoints are only written at optimizer-step boundaries.
    """
    epoch_history = []
    best_validation_result = None
    epochs_without_significant_improvement = 0
    current_epoch = 0
    global_optimizer_step = 0
    global_sample_cursor = 0
    overflow_retry_count = 0

    if resume_checkpoint_path is not None:
        checkpoint = (
            load_s2_training_checkpoint(
                resume_checkpoint_path
            )
        )
        progress = checkpoint["progress"]
        current_epoch = int(
            progress["epoch"]
        )
        global_optimizer_step = int(
            progress["optimizer_step"]
        )
        global_sample_cursor = int(
            progress["global_sample_cursor"]
        )
        overflow_retry_count = int(
            progress["overflow_retry_count"]
        )
        epoch_history = list(
            checkpoint["epoch_history"]
        )
        best_validation_result = checkpoint[
            "best_validation_result"
        ]
        epochs_without_significant_improvement = int(
            progress[
                "epochs_without_significant_improvement"
            ]
        )

    while current_epoch < MAXIMUM_EPOCHS:
        epoch_started_at = time.perf_counter()

        def save_mid_epoch_last_checkpoint(
            step_progress,
        ):
            periodic_progress = {
                **step_progress,
                "epochs_without_significant_improvement": (
                    epochs_without_significant_improvement
                ),
            }
            checkpoint_info = (
                save_s2_training_checkpoint(
                    target_path=(
                        S2_LAST_CHECKPOINT_PATH
                    ),
                    progress=periodic_progress,
                    best_validation_result=(
                        best_validation_result
                    ),
                    epoch_history=epoch_history,
                )
            )
            print(
                "Periodic last checkpoint:",
                f"step={step_progress['optimizer_step']} | "
                f"cursor={step_progress['global_sample_cursor']:,} | "
                f"sha={checkpoint_info['sha256'][:12]}",
            )

        train_result = train_s2_epoch(
            epoch=current_epoch,
            start_position=global_sample_cursor,
            starting_global_optimizer_step=(
                global_optimizer_step
            ),
            starting_overflow_retry_count=(
                overflow_retry_count
            ),
            step_log_path=(
                S2_TRAINING_STEP_JSONL_PATH
            ),
            periodic_checkpoint_callback=(
                save_mid_epoch_last_checkpoint
            ),
            checkpoint_every_successful_steps=250,
        )
        assert train_result["epoch_complete"]

        global_optimizer_step = int(
            train_result[
                "global_optimizer_step"
            ]
        )
        overflow_retry_count = int(
            train_result[
                "overflow_retry_count"
            ]
        )

        validation_result = (
            evaluate_s2_validation()
        )
        validation_artifact = (
            persist_s2_validation_result(
                current_epoch + 1,
                validation_result,
            )
        )
        candidate = compact_validation_result(
            current_epoch + 1,
            validation_result,
        )
        is_best = (
            is_better_s2_validation_result(
                candidate,
                best_validation_result,
            )
        )

        previous_best_cider = (
            None
            if best_validation_result is None
            else float(
                best_validation_result[
                    "metrics"
                ]["CIDEr-D"]
            )
        )
        candidate_cider = float(
            candidate["metrics"]["CIDEr-D"]
        )
        significant_improvement = (
            previous_best_cider is None
            or candidate_cider
            > previous_best_cider
            + EARLY_STOPPING_MINIMUM_DELTA
        )
        if significant_improvement:
            epochs_without_significant_improvement = 0
        else:
            epochs_without_significant_improvement += 1
        if is_best:
            best_validation_result = candidate

        epoch_elapsed_seconds = (
            time.perf_counter()
            - epoch_started_at
        )
        metrics = validation_result["metrics"]
        teacher_forced = validation_result[
            "teacher_forced"
        ]
        epoch_row = {
            "epoch": current_epoch + 1,
            "optimizer_steps": (
                train_result["optimizer_steps"]
            ),
            "samples": train_result["samples"],
            "valid_tokens": (
                train_result["valid_tokens"]
            ),
            "train_mean_token_ce": (
                train_result[
                    "mean_token_cross_entropy"
                ]
            ),
            "validation_mean_token_ce": (
                teacher_forced[
                    "mean_token_cross_entropy"
                ]
            ),
            "validation_token_accuracy": (
                teacher_forced["token_accuracy"]
            ),
            "bleu_1": metrics["BLEU-1"],
            "bleu_4": metrics["BLEU-4"],
            "meteor": metrics["METEOR"],
            "rouge_l": metrics["ROUGE-L"],
            "cider_d": metrics["CIDEr-D"],
            "is_best": is_best,
            "epochs_without_significant_improvement": (
                epochs_without_significant_improvement
            ),
            "epoch_elapsed_seconds": (
                epoch_elapsed_seconds
            ),
        }
        epoch_history.append(epoch_row)
        write_epoch_history_csv_atomic(
            epoch_history
        )

        next_epoch = current_epoch + 1
        progress = {
            "epoch": next_epoch,
            "optimizer_step": (
                global_optimizer_step
            ),
            "global_sample_cursor": 0,
            "microbatch_in_accumulation": 0,
            "overflow_retry_count": (
                overflow_retry_count
            ),
            "epochs_without_significant_improvement": (
                epochs_without_significant_improvement
            ),
            "checkpoint_boundary": (
                "optimizer_step"
            ),
            "validation_artifact": (
                validation_artifact
            ),
        }
        last_checkpoint = (
            save_s2_training_checkpoint(
                target_path=(
                    S2_LAST_CHECKPOINT_PATH
                ),
                progress=progress,
                best_validation_result=(
                    best_validation_result
                ),
                epoch_history=epoch_history,
            )
        )
        best_checkpoint = None
        if is_best:
            best_checkpoint = (
                save_s2_training_checkpoint(
                    target_path=(
                        S2_BEST_CHECKPOINT_PATH
                    ),
                    progress=progress,
                    best_validation_result=(
                        best_validation_result
                    ),
                    epoch_history=epoch_history,
                )
            )

        print(
            f"epoch={next_epoch:02d} | "
            f"train_CE={epoch_row['train_mean_token_ce']:.6f} | "
            f"val_CE={epoch_row['validation_mean_token_ce']:.6f} | "
            f"CIDEr-D={epoch_row['cider_d']:.6f} | "
            f"BLEU-4={epoch_row['bleu_4']:.6f} | "
            f"best={is_best} | "
            f"last={last_checkpoint['sha256'][:12]}"
        )
        if best_checkpoint is not None:
            print(
                "Best checkpoint updated:",
                best_checkpoint["sha256"],
            )

        should_early_stop = (
            next_epoch
            >= MINIMUM_EPOCHS_BEFORE_EARLY_STOP
            and epochs_without_significant_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        current_epoch = next_epoch
        global_sample_cursor = 0
        if should_early_stop:
            print(
                "Early stopping:",
                f"{epochs_without_significant_improvement} "
                "Validation evaluations without "
                f"CIDEr-D improvement > "
                f"{EARLY_STOPPING_MINIMUM_DELTA}.",
            )
            break

    assert best_validation_result is not None
    assert S2_LAST_CHECKPOINT_PATH.is_file()
    assert S2_BEST_CHECKPOINT_PATH.is_file()
    return {
        "completed_epochs": len(
            epoch_history
        ),
        "global_optimizer_step": (
            global_optimizer_step
        ),
        "overflow_retry_count": (
            overflow_retry_count
        ),
        "best_validation_result": (
            best_validation_result
        ),
        "epoch_history": epoch_history,
        "last_checkpoint": {
            "path": str(
                S2_LAST_CHECKPOINT_PATH
            ),
            "sha256": sha256_file(
                S2_LAST_CHECKPOINT_PATH
            ),
        },
        "best_checkpoint": {
            "path": str(
                S2_BEST_CHECKPOINT_PATH
            ),
            "sha256": sha256_file(
                S2_BEST_CHECKPOINT_PATH
            ),
        },
        "test_set_status": "sealed",
        "test_set_used": False,
    }


print("=" * 72)
print("S2 DISCARDED PRODUCTION TRAINER PILOT")
print("=" * 72)
print()
print("Pilot optimizer steps   : 4")
print("Expected pilot images   : 256")
print(
    "Backward divisor       :",
    TOKEN_SUM_BACKWARD_DIVISOR,
)
print("Final gradient rule     : grad(token-sum) / valid-token-count")
print("Gradient accumulation   :", GRADIENT_ACCUMULATION_STEPS)
print("Checkpoint boundary     : optimizer-step")
print("Pilot artifacts retained: False")
print("Full Validation run     : False")
print("Test data read          : False")


pilot_encoder_initial = model_state_cpu(
    S2_ENCODER_ADAPTER_MODEL
)
pilot_decoder_initial = model_state_cpu(
    S2_SHARED_DECODER_MODEL
)
pilot_optimizer_initial = cpu_clone_tree(
    S2_OPTIMIZER.state_dict()
)
pilot_scheduler_initial = cpu_clone_tree(
    S2_SCHEDULER.state_dict()
)
pilot_scaler_initial = cpu_clone_tree(
    S2_GRAD_SCALER.state_dict()
)
pilot_rng_initial = capture_rng_state()

TEMPORARY_PILOT_CHECKPOINT_PATH = (
    Path(RUN_CHECKPOINT_DIR)
    / "s2_temporary_production_trainer_pilot.pt"
)
TEMPORARY_PILOT_STEP_LOG_PATH = (
    S2_TRAINING_LOG_DIR
    / "s2_temporary_production_trainer_pilot.jsonl"
)
temporary_paths = [
    TEMPORARY_PILOT_CHECKPOINT_PATH,
    TEMPORARY_PILOT_CHECKPOINT_PATH.with_suffix(
        TEMPORARY_PILOT_CHECKPOINT_PATH.suffix
        + ".tmp"
    ),
    TEMPORARY_PILOT_STEP_LOG_PATH,
    TEMPORARY_PILOT_STEP_LOG_PATH.with_suffix(
        TEMPORARY_PILOT_STEP_LOG_PATH.suffix
        + ".tmp"
    ),
]
for stale_path in temporary_paths:
    if stale_path.exists():
        stale_path.unlink()

pilot_passed = False
pilot_resume_exact = False
pilot_checkpoint_size = 0
pilot_checkpoint_sha256 = None
pilot_result = None
pilot_step_records = []

try:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

    pilot_result = train_s2_epoch(
        epoch=0,
        start_position=0,
        starting_global_optimizer_step=0,
        starting_overflow_retry_count=0,
        maximum_successful_optimizer_steps=4,
        step_log_path=(
            TEMPORARY_PILOT_STEP_LOG_PATH
        ),
        progress_print_every=1,
    )
    assert pilot_result["optimizer_steps"] == 4
    assert pilot_result[
        "skipped_overflow_steps"
    ] == 0
    assert pilot_result["samples"] == 256
    assert pilot_result[
        "next_global_sample_cursor"
    ] == 256
    assert pilot_result[
        "global_optimizer_step"
    ] == 4
    assert pilot_result[
        "reached_requested_limit"
    ] is True
    assert pilot_result["epoch_complete"] is False
    assert math.isfinite(
        pilot_result[
            "mean_token_cross_entropy"
        ]
    )
    assert math.isfinite(
        pilot_result[
            "last_gradient_norm_before_clip"
        ]
    )

    pilot_step_records = [
        json.loads(line)
        for line in (
            TEMPORARY_PILOT_STEP_LOG_PATH
            .read_text(encoding="utf-8")
            .splitlines()
        )
        if line.strip()
    ]
    assert len(pilot_step_records) == 4
    assert all(
        record["successful_step"] is True
        for record in pilot_step_records
    )
    assert [
        int(record["global_optimizer_step"])
        for record in pilot_step_records
    ] == [1, 2, 3, 4]
    assert [
        int(record["global_sample_cursor"])
        for record in pilot_step_records
    ] == [64, 128, 192, 256]

    pilot_progress = {
        "epoch": 0,
        "optimizer_step": 4,
        "global_sample_cursor": 256,
        "microbatch_in_accumulation": 0,
        "overflow_retry_count": 0,
        "epochs_without_significant_improvement": 0,
        "checkpoint_boundary": (
            "optimizer_step"
        ),
    }
    pilot_checkpoint_info = (
        save_s2_training_checkpoint(
            target_path=(
                TEMPORARY_PILOT_CHECKPOINT_PATH
            ),
            progress=pilot_progress,
            best_validation_result=None,
            epoch_history=[],
        )
    )
    pilot_checkpoint_size = int(
        pilot_checkpoint_info["size_bytes"]
    )
    pilot_checkpoint_sha256 = (
        pilot_checkpoint_info["sha256"]
    )

    pilot_encoder_trained = model_state_cpu(
        S2_ENCODER_ADAPTER_MODEL
    )
    pilot_decoder_trained = model_state_cpu(
        S2_SHARED_DECODER_MODEL
    )
    pilot_optimizer_trained = cpu_clone_tree(
        S2_OPTIMIZER.state_dict()
    )
    pilot_scheduler_trained = cpu_clone_tree(
        S2_SCHEDULER.state_dict()
    )
    pilot_scaler_trained = cpu_clone_tree(
        S2_GRAD_SCALER.state_dict()
    )
    pilot_rng_trained = capture_rng_state()

    # Move away, then prove that the production loader restores exact state.
    S2_ENCODER_ADAPTER_MODEL.load_state_dict(
        pilot_encoder_initial,
        strict=True,
    )
    S2_SHARED_DECODER_MODEL.load_state_dict(
        pilot_decoder_initial,
        strict=True,
    )
    S2_OPTIMIZER.load_state_dict(
        pilot_optimizer_initial
    )
    S2_SCHEDULER.load_state_dict(
        pilot_scheduler_initial
    )
    S2_GRAD_SCALER.load_state_dict(
        pilot_scaler_initial
    )
    restore_rng_state(pilot_rng_initial)

    loaded_pilot_checkpoint = (
        load_s2_training_checkpoint(
            TEMPORARY_PILOT_CHECKPOINT_PATH
        )
    )
    assert loaded_pilot_checkpoint[
        "progress"
    ]["global_sample_cursor"] == 256
    assert loaded_pilot_checkpoint[
        "progress"
    ]["optimizer_step"] == 4
    assert trees_exact(
        model_state_cpu(
            S2_ENCODER_ADAPTER_MODEL
        ),
        pilot_encoder_trained,
    )
    assert trees_exact(
        model_state_cpu(
            S2_SHARED_DECODER_MODEL
        ),
        pilot_decoder_trained,
    )
    assert trees_exact(
        cpu_clone_tree(
            S2_OPTIMIZER.state_dict()
        ),
        pilot_optimizer_trained,
    )
    assert trees_exact(
        cpu_clone_tree(
            S2_SCHEDULER.state_dict()
        ),
        pilot_scheduler_trained,
    )
    assert trees_exact(
        cpu_clone_tree(
            S2_GRAD_SCALER.state_dict()
        ),
        pilot_scaler_trained,
    )
    assert trees_exact(
        capture_rng_state(),
        pilot_rng_trained,
    )
    pilot_resume_exact = True
    pilot_passed = True
    del loaded_pilot_checkpoint

finally:
    S2_ENCODER_ADAPTER_MODEL.load_state_dict(
        pilot_encoder_initial,
        strict=True,
    )
    S2_SHARED_DECODER_MODEL.load_state_dict(
        pilot_decoder_initial,
        strict=True,
    )
    S2_OPTIMIZER.load_state_dict(
        pilot_optimizer_initial
    )
    S2_SCHEDULER.load_state_dict(
        pilot_scheduler_initial
    )
    S2_GRAD_SCALER.load_state_dict(
        pilot_scaler_initial
    )
    S2_OPTIMIZER.zero_grad(set_to_none=True)
    restore_rng_state(pilot_rng_initial)
    S2_ENCODER_ADAPTER_MODEL.eval()
    S2_SHARED_DECODER_MODEL.eval()

    for temporary_path in temporary_paths:
        if temporary_path.exists():
            temporary_path.unlink()


assert pilot_passed
assert pilot_resume_exact
assert all(
    not path.exists()
    for path in temporary_paths
)
assert trees_exact(
    model_state_cpu(
        S2_ENCODER_ADAPTER_MODEL
    ),
    pilot_encoder_initial,
)
assert trees_exact(
    model_state_cpu(
        S2_SHARED_DECODER_MODEL
    ),
    pilot_decoder_initial,
)
assert trees_exact(
    cpu_clone_tree(
        S2_OPTIMIZER.state_dict()
    ),
    pilot_optimizer_initial,
)
assert trees_exact(
    cpu_clone_tree(
        S2_SCHEDULER.state_dict()
    ),
    pilot_scheduler_initial,
)
assert trees_exact(
    cpu_clone_tree(
        S2_GRAD_SCALER.state_dict()
    ),
    pilot_scaler_initial,
)
assert trees_exact(
    capture_rng_state(),
    pilot_rng_initial,
)
assert all(
    parameter.grad is None
    for _, parameter
    in named_trainable_parameters()
)
assert sha256_file(
    S2_ENCODER_INITIAL_CHECKPOINT_PATH
) == actual_encoder_checkpoint_sha256
assert sha256_file(
    SHARED_DECODER_INITIAL_CHECKPOINT_PATH
) == actual_decoder_checkpoint_sha256

pilot_peak_gpu_memory_mib = (
    torch.cuda.max_memory_allocated(device)
    / (1024 ** 2)
)
pilot_seconds_per_optimizer_step = (
    float(pilot_result["elapsed_seconds"])
    / int(pilot_result["optimizer_steps"])
)
estimated_epoch_training_minutes = (
    pilot_seconds_per_optimizer_step
    * 1_771
    / 60.0
)

print()
print("Pilot result")
print("Optimizer steps         :", pilot_result["optimizer_steps"])
print("Microbatches            :", pilot_result["optimizer_steps"] * 2)
print("Images                  :", pilot_result["samples"])
print("Valid tokens            :", pilot_result["valid_tokens"])
print(
    "Mean token CE           :",
    f"{pilot_result['mean_token_cross_entropy']:.8f}",
)
print(
    "Last gradient norm      :",
    f"{pilot_result['last_gradient_norm_before_clip']:.8f}",
)
print("FP16 overflow steps     :", pilot_result["skipped_overflow_steps"])
print("GradScaler scale        :", pilot_result["grad_scaler_scale"])
print(
    "Seconds/optimizer step  :",
    round(pilot_seconds_per_optimizer_step, 4),
)
print(
    "Estimated train min/epoch:",
    round(estimated_epoch_training_minutes, 2),
)
print(
    "Peak GPU memory         :",
    round(pilot_peak_gpu_memory_mib, 2),
    "MiB",
)
print("Checkpoint resume exact :", pilot_resume_exact)
print("Initial restore exact   : True")
print("Temporary artifacts     : DELETED")


S2_CELL10_TRAINER_CONTRACT_PATH = (
    RUN_CONFIG_DIR
    / "s2_production_trainer_checkpoint_early_stop_contract.json"
)
S2_CELL10_AUDIT_PATH = (
    RUN_AUDIT_DIR
    / "s2_cell10_production_trainer_discarded_pilot_audit.json"
)
S2_FULL_RUN_READY_PATH = (
    Path(RUN_ROOT)
    / "S2_SEED2026_FULL_RUN_READY.json"
)

cell_elapsed_seconds = (
    time.perf_counter() - CELL_STARTED_AT
)

trainer_contract = {
    "contract_version": (
        "phase2_s2_production_trainer_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "locked",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "entry_point": "run_s2_full_training",
    "epoch_trainer": {
        "entry_point": "train_s2_epoch",
        "images_per_full_epoch": 113_287,
        "physical_batch_size": (
            PHYSICAL_BATCH_SIZE
        ),
        "gradient_accumulation_steps": (
            GRADIENT_ACCUMULATION_STEPS
        ),
        "effective_batch_size": (
            EFFECTIVE_BATCH_SIZE
        ),
        "loss_accumulation": (
            "backward token-sum NLL / 1024; multiply unscaled "
            "gradients by 1024 / total valid tokens; final "
            "gradient equals grad(token-sum) / valid-token-count"
        ),
        "token_sum_backward_divisor": (
            TOKEN_SUM_BACKWARD_DIVISOR
        ),
        "gradient_clip_norm": (
            GRADIENT_CLIP_NORM
        ),
        "maximum_overflow_retries": (
            MAXIMUM_OVERFLOW_RETRIES
        ),
        "scheduler_step_unit": (
            "successful optimizer step"
        ),
    },
    "validation": {
        "frequency": "every completed epoch",
        "selection_split": (
            "Karpathy validation"
        ),
        "teacher_forced_captions": 25_010,
        "generation_images": 5_000,
        "primary_metric": "CIDEr-D",
        "tie_break_order": [
            "BLEU-4 higher",
            "validation Cross-Entropy lower",
            "earlier epoch",
        ],
    },
    "checkpointing": {
        "atomic": True,
        "boundary": "optimizer_step",
        "keep_last": True,
        "keep_best": True,
        "last_path": str(
            S2_LAST_CHECKPOINT_PATH
        ),
        "best_path": str(
            S2_BEST_CHECKPOINT_PATH
        ),
        "mid_epoch_resume": True,
        "periodic_last_checkpoint_steps": 250,
        "partial_accumulation_saved": False,
    },
    "early_stopping": {
        "enabled": True,
        "minimum_epochs": (
            MINIMUM_EPOCHS_BEFORE_EARLY_STOP
        ),
        "patience": EARLY_STOPPING_PATIENCE,
        "minimum_cider_d_delta": (
            EARLY_STOPPING_MINIMUM_DELTA
        ),
    },
    "logs": {
        "optimizer_steps_jsonl": str(
            S2_TRAINING_STEP_JSONL_PATH
        ),
        "epoch_history_csv": str(
            S2_TRAINING_HISTORY_CSV_PATH
        ),
        "validation_epoch_directory": str(
            S2_VALIDATION_RESULT_DIR
        ),
        "epoch_csv_atomic_rewrite": True,
        "optimizer_jsonl_write": (
            "append + flush + fsync; trailing partial "
            "line may be discarded after abrupt termination"
        ),
    },
    "pilot": {
        "successful_optimizer_steps": 4,
        "images": 256,
        "full_validation_run": False,
        "resume_exact": True,
        "initial_restore_exact": True,
        "artifacts_deleted": True,
        "weights_reused": False,
    },
    "policy": {
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_images_read": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "flickr_used": False,
        "full_training_started": False,
    },
}
write_json(
    S2_CELL10_TRAINER_CONTRACT_PATH,
    trainer_contract,
)

cell_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "student_id": STUDENT_ID,
    "pilot_optimizer_steps": 4,
    "pilot_microbatches": 8,
    "pilot_images": 256,
    "pilot_valid_tokens": (
        pilot_result["valid_tokens"]
    ),
    "pilot_mean_token_cross_entropy": (
        pilot_result[
            "mean_token_cross_entropy"
        ]
    ),
    "pilot_last_gradient_norm": (
        pilot_result[
            "last_gradient_norm_before_clip"
        ]
    ),
    "pilot_fp16_overflow_steps": (
        pilot_result[
            "skipped_overflow_steps"
        ]
    ),
    "pilot_grad_scaler_scale": (
        pilot_result["grad_scaler_scale"]
    ),
    "pilot_seconds_per_optimizer_step": (
        pilot_seconds_per_optimizer_step
    ),
    "estimated_epoch_training_minutes": (
        estimated_epoch_training_minutes
    ),
    "pilot_peak_gpu_memory_mib": (
        pilot_peak_gpu_memory_mib
    ),
    "pilot_checkpoint_size_bytes": (
        pilot_checkpoint_size
    ),
    "pilot_checkpoint_sha256_before_delete": (
        pilot_checkpoint_sha256
    ),
    "pilot_resume_exact": (
        pilot_resume_exact
    ),
    "initial_restore_exact": True,
    "temporary_artifacts_deleted": True,
    "production_functions_ready": [
        "train_s2_epoch",
        "save_s2_training_checkpoint",
        "load_s2_training_checkpoint",
        "run_s2_full_training",
    ],
    "full_validation_run": False,
    "test_parquet_opened": False,
    "test_caption_read": False,
    "test_images_read": False,
    "test_metrics_computed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "full_training_started": False,
    "elapsed_seconds": cell_elapsed_seconds,
}
write_json(
    S2_CELL10_AUDIT_PATH,
    cell_audit,
)

S2_CELL10_TRAINER_CONTRACT_SHA256 = (
    sha256_file(
        S2_CELL10_TRAINER_CONTRACT_PATH
    )
)
S2_CELL10_AUDIT_SHA256 = sha256_file(
    S2_CELL10_AUDIT_PATH
)

full_run_ready = {
    "marker_version": (
        "phase2_s2_seed2026_full_run_ready_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "ready",
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "production_trainer_ready": True,
    "atomic_last_checkpoint_ready": True,
    "atomic_best_checkpoint_ready": True,
    "mid_epoch_resume_ready": True,
    "structured_logs_ready": True,
    "validation_only_selection_ready": True,
    "early_stopping_ready": True,
    "discarded_pilot_passed": True,
    "initial_state_restored_exact": True,
    "test_set_status": "sealed",
    "test_set_used": False,
    "full_training_started": False,
    "contract": {
        "path": str(
            S2_CELL10_TRAINER_CONTRACT_PATH
        ),
        "sha256": (
            S2_CELL10_TRAINER_CONTRACT_SHA256
        ),
    },
    "audit": {
        "path": str(S2_CELL10_AUDIT_PATH),
        "sha256": S2_CELL10_AUDIT_SHA256,
    },
}
write_json(
    S2_FULL_RUN_READY_PATH,
    full_run_ready,
)
S2_FULL_RUN_READY_SHA256 = sha256_file(
    S2_FULL_RUN_READY_PATH
)


print()
print("=" * 72)
print("PHASE 2 S2 CELL 10 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Production epoch trainer  : READY")
print("Token-balanced accumulation: PASSED")
print("FP16-safe backward divisor : 1024.0")
print("Atomic last checkpoint    : READY")
print("Atomic best checkpoint    : READY")
print("Mid-epoch resume          : READY")
print("Optimizer-step JSONL      : READY")
print("Epoch-history CSV         : READY")
print("Validation artifacts      : READY")
print("Selection split           : Validation only")
print("Checkpoint primary        : CIDEr-D")
print("Early stopping            : epoch >= 5 / patience 4 / delta 0.001")
print("Discarded pilot steps     : 4 / 4 PASSED")
print("Pilot images              : 256")
print("FP16 overflow             : False")
print("Pilot resume exact        :", pilot_resume_exact)
print("Initial restore exact     : True")
print("Temporary artifacts       : DELETED")
print("Initial checkpoints intact: True")
print("Full Validation run       : False")
print("Test Parquet opened       : False")
print("Test caption read         : False")
print("Test images read          : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Full training started     : False")
print("Contract                  :", S2_CELL10_TRAINER_CONTRACT_PATH)
print("Contract SHA-256          :", S2_CELL10_TRAINER_CONTRACT_SHA256)
print("Audit                     :", S2_CELL10_AUDIT_PATH)
print("Audit SHA-256             :", S2_CELL10_AUDIT_SHA256)
print("READY marker              :", S2_FULL_RUN_READY_PATH)
print("READY SHA-256             :", S2_FULL_RUN_READY_SHA256)
print(
    "Elapsed                   :",
    round(cell_elapsed_seconds, 2),
    "giây",
)
print(
    "Next                      : launch or resume the full S2 seed-2026 "
    "run in Cell 11; do not open Held-out Test"
)

## Cell 11 — Full S2 Seed-2026 Confirmation Training

Cell này thực hiện:

- Xác minh production trainer và READY marker từ Cell 10.
- Tự động xác định chế độ:
  - new_run;
  - local_resume;
  - external_resume.
- Huấn luyện S2 trên MS COCO Karpathy train + restval.
- Chạy teacher-forced Validation và Beam-3 Validation sau mỗi epoch.
- Chọn best checkpoint chỉ bằng Validation CIDEr-D.
- Ghi atomic last checkpoint mỗi 250 optimizer steps.
- Ghi best checkpoint khi Validation được cải thiện.
- Áp dụng early stopping từ epoch 5 với patience 4.
- Ghi training history CSV, optimizer-step JSONL và Validation artifacts.
- Khi hoàn tất, tạo manifest CSV/JSON, SHA256SUMS và COMPLETE marker.
- Không truy cập Held-out Test Set.

Đây là cell huấn luyện dài. Không chạy cell khác song song trong notebook.

In [ ]:
import csv
import json
import math
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch


CELL_STARTED_AT = time.perf_counter()

required_runtime_names = [
    "STUDENT_ID",
    "STUDENT_NAME",
    "DEVELOPMENT_SEED",
    "RUN_ROOT",
    "RUN_CONFIG_DIR",
    "RUN_AUDIT_DIR",
    "RUN_CHECKPOINT_DIR",
    "training_contract",
    "validation_policy",
    "S2_FULL_RUN_READY_PATH",
    "S2_FULL_RUN_READY_SHA256",
    "S2_CELL10_TRAINER_CONTRACT_PATH",
    "S2_CELL10_TRAINER_CONTRACT_SHA256",
    "S2_CELL10_AUDIT_PATH",
    "S2_CELL10_AUDIT_SHA256",
    "S2_LAST_CHECKPOINT_PATH",
    "S2_BEST_CHECKPOINT_PATH",
    "S2_TRAINING_HISTORY_CSV_PATH",
    "S2_TRAINING_STEP_JSONL_PATH",
    "S2_VALIDATION_RESULT_DIR",
    "run_s2_full_training",
    "atomic_write_text",
    "write_json",
    "read_json",
    "sha256_file",
]
missing_runtime_names = [
    name for name in required_runtime_names if name not in globals()
]
assert not missing_runtime_names, (
    "Thiếu biến từ S2 Cell 1–10: "
    f"{missing_runtime_names}. Hãy chạy lại các cell trước."
)

assert STUDENT_ID == "S2"
assert STUDENT_NAME == "MobileViT-S"
assert DEVELOPMENT_SEED == 2026

full_run_ready = read_json(
    S2_FULL_RUN_READY_PATH
)
assert full_run_ready["status"] == "ready"
assert full_run_ready[
    "production_trainer_ready"
] is True
assert full_run_ready[
    "atomic_last_checkpoint_ready"
] is True
assert full_run_ready[
    "atomic_best_checkpoint_ready"
] is True
assert full_run_ready[
    "mid_epoch_resume_ready"
] is True
assert full_run_ready[
    "validation_only_selection_ready"
] is True
assert full_run_ready[
    "discarded_pilot_passed"
] is True
assert full_run_ready[
    "initial_state_restored_exact"
] is True
assert full_run_ready["test_set_status"] == "sealed"
assert full_run_ready["test_set_used"] is False
assert full_run_ready[
    "full_training_started"
] is False
assert (
    sha256_file(S2_FULL_RUN_READY_PATH)
    == S2_FULL_RUN_READY_SHA256
)
assert (
    sha256_file(S2_CELL10_TRAINER_CONTRACT_PATH)
    == S2_CELL10_TRAINER_CONTRACT_SHA256
)
assert (
    sha256_file(S2_CELL10_AUDIT_PATH)
    == S2_CELL10_AUDIT_SHA256
)


# Optional recovery setting.
#
# For a new run, leave this as None.
# After attaching a previous Kaggle notebook output, set it to that output's
# exact s2_mobilevit_s_seed2026_last.pt path, rerun Cells 1–10 to reconstruct
# the runtime, then run this cell.
S2_EXTERNAL_RESUME_CHECKPOINT = globals().get(
    "S2_EXTERNAL_RESUME_CHECKPOINT",
    None,
)
if S2_EXTERNAL_RESUME_CHECKPOINT is not None:
    S2_EXTERNAL_RESUME_CHECKPOINT = Path(
        S2_EXTERNAL_RESUME_CHECKPOINT
    )
    assert S2_EXTERNAL_RESUME_CHECKPOINT.is_file(), (
        "Không tìm thấy external resume checkpoint: "
        f"{S2_EXTERNAL_RESUME_CHECKPOINT}"
    )


S2_FULL_RUN_COMPLETE_PATH = (
    Path(RUN_ROOT)
    / "S2_SEED2026_FULL_RUN_COMPLETE.json"
)
S2_FINAL_MANIFEST_DIR = (
    Path(RUN_ROOT) / "manifests"
)
S2_FINAL_MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
S2_FINAL_MANIFEST_CSV_PATH = (
    S2_FINAL_MANIFEST_DIR
    / "s2_seed2026_full_run_manifest.csv"
)
S2_FINAL_MANIFEST_JSON_PATH = (
    S2_FINAL_MANIFEST_DIR
    / "s2_seed2026_full_run_manifest.json"
)
S2_FINAL_SHA256SUMS_PATH = (
    S2_FINAL_MANIFEST_DIR
    / "s2_seed2026_full_run_SHA256SUMS.txt"
)
S2_CELL11_AUDIT_PATH = (
    RUN_AUDIT_DIR
    / "s2_cell11_full_seed2026_training_run_audit.json"
)


def inspect_resume_checkpoint(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )
    assert checkpoint["schema_version"] == (
        "phase2_s2_full_training_checkpoint_v1"
    )
    assert checkpoint["student_id"] == "S2"
    assert int(checkpoint["seed"]) == DEVELOPMENT_SEED
    assert checkpoint["data_policy"][
        "test_set_status"
    ] == "sealed"
    assert checkpoint["data_policy"][
        "test_set_used"
    ] is False
    progress = checkpoint["progress"]
    assert progress[
        "checkpoint_boundary"
    ] == "optimizer_step"
    assert int(
        progress["microbatch_in_accumulation"]
    ) == 0
    return {
        "epoch": int(progress["epoch"]),
        "optimizer_step": int(
            progress["optimizer_step"]
        ),
        "global_sample_cursor": int(
            progress["global_sample_cursor"]
        ),
        "overflow_retry_count": int(
            progress["overflow_retry_count"]
        ),
        "epochs_without_significant_improvement": int(
            progress[
                "epochs_without_significant_improvement"
            ]
        ),
        "history_rows": len(
            checkpoint["epoch_history"]
        ),
        "sha256": sha256_file(
            checkpoint_path
        ),
        "size_bytes": int(
            Path(checkpoint_path).stat().st_size
        ),
    }


def repair_step_log_for_resume(
    step_log_path,
    maximum_global_optimizer_step,
):
    """
    If a session stopped after logging steps newer than the most recent
    periodic checkpoint, discard those trailing records before replay.
    """
    step_log_path = Path(step_log_path)
    if not step_log_path.exists():
        return {
            "existing_rows": 0,
            "retained_rows": 0,
            "removed_rows": 0,
        }

    records = [
        json.loads(line)
        for line in step_log_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
    retained = [
        record
        for record in records
        if int(
            record["global_optimizer_step"]
        ) <= int(
            maximum_global_optimizer_step
        )
    ]
    repaired_text = "".join(
        json.dumps(
            record,
            ensure_ascii=False,
            sort_keys=True,
        )
        + "\n"
        for record in retained
    )
    atomic_write_text(
        step_log_path,
        repaired_text,
    )
    return {
        "existing_rows": len(records),
        "retained_rows": len(retained),
        "removed_rows": (
            len(records) - len(retained)
        ),
    }


def build_final_manifest_rows(run_root):
    run_root = Path(run_root)
    excluded_paths = {
        S2_FINAL_MANIFEST_CSV_PATH.resolve(),
        S2_FINAL_MANIFEST_JSON_PATH.resolve(),
        S2_FINAL_SHA256SUMS_PATH.resolve(),
        S2_FULL_RUN_COMPLETE_PATH.resolve(),
    }
    rows = []
    for path in sorted(
        run_root.rglob("*"),
        key=lambda value: value.as_posix(),
    ):
        if not path.is_file():
            continue
        if path.resolve() in excluded_paths:
            continue
        if path.name.endswith(".tmp"):
            raise AssertionError(
                f"Phát hiện temporary artifact: {path}"
            )
        rows.append(
            {
                "relative_path": (
                    path.relative_to(
                        run_root
                    ).as_posix()
                ),
                "size_bytes": int(
                    path.stat().st_size
                ),
                "sha256": sha256_file(path),
            }
        )
    assert rows
    assert len({
        row["relative_path"]
        for row in rows
    }) == len(rows)
    return rows


def write_final_manifests(rows):
    from io import StringIO

    csv_buffer = StringIO()
    writer = csv.DictWriter(
        csv_buffer,
        fieldnames=[
            "relative_path",
            "size_bytes",
            "sha256",
        ],
    )
    writer.writeheader()
    writer.writerows(rows)
    atomic_write_text(
        S2_FINAL_MANIFEST_CSV_PATH,
        csv_buffer.getvalue(),
    )

    manifest_payload = {
        "schema_version": (
            "phase2_s2_seed2026_full_run_manifest_v1"
        ),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "student_id": STUDENT_ID,
        "seed": DEVELOPMENT_SEED,
        "artifact_count": len(rows),
        "total_size_bytes": sum(
            row["size_bytes"] for row in rows
        ),
        "artifacts": rows,
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(
        S2_FINAL_MANIFEST_JSON_PATH,
        manifest_payload,
    )

    checksum_text = "".join(
        f"{row['sha256']}  "
        f"{row['relative_path']}\n"
        for row in rows
    )
    atomic_write_text(
        S2_FINAL_SHA256SUMS_PATH,
        checksum_text,
    )

    return {
        "csv": {
            "path": str(
                S2_FINAL_MANIFEST_CSV_PATH
            ),
            "sha256": sha256_file(
                S2_FINAL_MANIFEST_CSV_PATH
            ),
        },
        "json": {
            "path": str(
                S2_FINAL_MANIFEST_JSON_PATH
            ),
            "sha256": sha256_file(
                S2_FINAL_MANIFEST_JSON_PATH
            ),
        },
        "sha256sums": {
            "path": str(
                S2_FINAL_SHA256SUMS_PATH
            ),
            "sha256": sha256_file(
                S2_FINAL_SHA256SUMS_PATH
            ),
        },
    }


if S2_FULL_RUN_COMPLETE_PATH.exists():
    raise AssertionError(
        "S2 full run đã có COMPLETE marker. Không chạy training lần hai. "
        f"Marker: {S2_FULL_RUN_COMPLETE_PATH}"
    )

if S2_EXTERNAL_RESUME_CHECKPOINT is not None:
    resume_checkpoint_path = (
        S2_EXTERNAL_RESUME_CHECKPOINT
    )
    run_mode = "external_resume"
elif S2_LAST_CHECKPOINT_PATH.exists():
    resume_checkpoint_path = (
        S2_LAST_CHECKPOINT_PATH
    )
    run_mode = "local_resume"
else:
    resume_checkpoint_path = None
    run_mode = "new_run"

resume_checkpoint_info = None
step_log_repair = {
    "existing_rows": 0,
    "retained_rows": 0,
    "removed_rows": 0,
}
if resume_checkpoint_path is not None:
    resume_checkpoint_info = (
        inspect_resume_checkpoint(
            resume_checkpoint_path
        )
    )
    step_log_repair = (
        repair_step_log_for_resume(
            S2_TRAINING_STEP_JSONL_PATH,
            resume_checkpoint_info[
                "optimizer_step"
            ],
        )
    )


print("=" * 72)
print("S2 FULL SEED-2026 CONFIRMATION TRAINING")
print("=" * 72)
print()
print("Student                  : S2 — MobileViT-S")
print("Run mode                 :", run_mode)
print(
    "Resume checkpoint        :",
    (
        resume_checkpoint_path
        if resume_checkpoint_path
        is not None
        else "NONE — initial checkpoints"
    ),
)
if resume_checkpoint_info is not None:
    print(
        "Resume progress          :",
        f"epoch={resume_checkpoint_info['epoch']} / "
        f"step={resume_checkpoint_info['optimizer_step']} / "
        f"cursor={resume_checkpoint_info['global_sample_cursor']}",
    )
    print(
        "Step-log repair          :",
        step_log_repair,
    )
print("Maximum epochs           : 20")
print("Estimated train/epoch    : ~21.09 minutes on current T4")
print("Validation every epoch   : 25,010 captions + 5,000 images")
print("Generation               : Beam-3 / length penalty 0.7")
print("Selection                : Validation CIDEr-D only")
print("Periodic last checkpoint : every 250 optimizer steps")
print("Early stopping           : min epoch 5 / patience 4 / delta 0.001")
print("Held-out Test            : SEALED")
print()


training_started_at = time.perf_counter()
training_result = run_s2_full_training(
    resume_checkpoint_path=(
        resume_checkpoint_path
    )
)
training_elapsed_seconds = (
    time.perf_counter()
    - training_started_at
)


assert 5 <= int(
    training_result["completed_epochs"]
) <= 20
assert int(
    training_result[
        "global_optimizer_step"
    ]
) > 0
assert int(
    training_result[
        "overflow_retry_count"
    ]
) <= 10
assert training_result[
    "test_set_status"
] == "sealed"
assert training_result[
    "test_set_used"
] is False
assert S2_LAST_CHECKPOINT_PATH.is_file()
assert S2_BEST_CHECKPOINT_PATH.is_file()
assert S2_TRAINING_HISTORY_CSV_PATH.is_file()
assert S2_TRAINING_STEP_JSONL_PATH.is_file()

epoch_history = training_result[
    "epoch_history"
]
assert len(epoch_history) == int(
    training_result["completed_epochs"]
)
epoch_numbers = [
    int(row["epoch"])
    for row in epoch_history
]
assert epoch_numbers == list(
    range(1, len(epoch_history) + 1)
)
best_validation_result = training_result[
    "best_validation_result"
]
best_epoch = int(
    best_validation_result["epoch"]
)
best_metrics = best_validation_result[
    "metrics"
]
best_validation_ce = float(
    best_validation_result[
        "teacher_forced"
    ]["mean_token_cross_entropy"]
)
assert 1 <= best_epoch <= len(
    epoch_history
)
assert all(
    math.isfinite(float(value))
    for value in best_metrics.values()
)
assert math.isfinite(best_validation_ce)
assert set(best_metrics) == {
    "BLEU-1",
    "BLEU-4",
    "METEOR",
    "ROUGE-L",
    "CIDEr-D",
}

validation_epoch_files = sorted(
    S2_VALIDATION_RESULT_DIR.glob(
        "epoch_*_validation.json"
    )
)
assert len(validation_epoch_files) == len(
    epoch_history
)

last_checkpoint_info = (
    inspect_resume_checkpoint(
        S2_LAST_CHECKPOINT_PATH
    )
)
best_checkpoint_info = (
    inspect_resume_checkpoint(
        S2_BEST_CHECKPOINT_PATH
    )
)
assert last_checkpoint_info[
    "history_rows"
] == len(epoch_history)
assert best_checkpoint_info[
    "history_rows"
] == best_epoch


print()
print("=" * 72)
print("S2 FULL RUN RESULT")
print("=" * 72)
print()
print("Completed epochs         :", len(epoch_history))
print(
    "Global optimizer steps   :",
    training_result["global_optimizer_step"],
)
print(
    "FP16 overflow retries    :",
    training_result["overflow_retry_count"],
)
print("Best epoch               :", best_epoch)
print(
    "Best Validation CE       :",
    f"{best_validation_ce:.8f}",
)
print(
    "Best BLEU-1              :",
    f"{float(best_metrics['BLEU-1']):.8f}",
)
print(
    "Best BLEU-4              :",
    f"{float(best_metrics['BLEU-4']):.8f}",
)
print(
    "Best METEOR              :",
    f"{float(best_metrics['METEOR']):.8f}",
)
print(
    "Best ROUGE-L             :",
    f"{float(best_metrics['ROUGE-L']):.8f}",
)
print(
    "Best CIDEr-D             :",
    f"{float(best_metrics['CIDEr-D']):.8f}",
)
print(
    "Last checkpoint SHA-256  :",
    last_checkpoint_info["sha256"],
)
print(
    "Best checkpoint SHA-256  :",
    best_checkpoint_info["sha256"],
)
print(
    "Training elapsed         :",
    round(
        training_elapsed_seconds / 3600.0,
        3,
    ),
    "giờ",
)


# Write the audit before inventory so it is included in the final manifest.
cell_elapsed_seconds = (
    time.perf_counter() - CELL_STARTED_AT
)
cell_audit = {
    "status": "passed",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "student_id": STUDENT_ID,
    "seed": DEVELOPMENT_SEED,
    "run_mode": run_mode,
    "resume_checkpoint": (
        None
        if resume_checkpoint_path is None
        else {
            "path": str(
                resume_checkpoint_path
            ),
            **resume_checkpoint_info,
        }
    ),
    "step_log_repair": step_log_repair,
    "completed_epochs": len(epoch_history),
    "global_optimizer_steps": int(
        training_result[
            "global_optimizer_step"
        ]
    ),
    "overflow_retry_count": int(
        training_result[
            "overflow_retry_count"
        ]
    ),
    "best_epoch": best_epoch,
    "best_validation_cross_entropy": (
        best_validation_ce
    ),
    "best_metrics": dict(best_metrics),
    "last_checkpoint": (
        last_checkpoint_info
    ),
    "best_checkpoint": (
        best_checkpoint_info
    ),
    "validation_artifact_count": len(
        validation_epoch_files
    ),
    "training_elapsed_seconds": (
        training_elapsed_seconds
    ),
    "test_parquet_opened": False,
    "test_caption_read": False,
    "test_images_read": False,
    "test_predictions_written": False,
    "test_metrics_computed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "cell_elapsed_seconds": (
        cell_elapsed_seconds
    ),
}
write_json(
    S2_CELL11_AUDIT_PATH,
    cell_audit,
)
S2_CELL11_AUDIT_SHA256 = sha256_file(
    S2_CELL11_AUDIT_PATH
)

manifest_rows = build_final_manifest_rows(
    RUN_ROOT
)
manifest_files = write_final_manifests(
    manifest_rows
)

# Verify every listed artifact after writing manifests.
for row in manifest_rows:
    artifact_path = (
        Path(RUN_ROOT)
        / row["relative_path"]
    )
    assert artifact_path.is_file()
    assert int(
        artifact_path.stat().st_size
    ) == int(row["size_bytes"])
    assert sha256_file(
        artifact_path
    ) == row["sha256"]

role_counts = Counter()
for row in manifest_rows:
    relative_path = row["relative_path"]
    if relative_path.startswith(
        "checkpoints/"
    ):
        role_counts["checkpoint"] += 1
    elif relative_path.startswith("logs/"):
        role_counts["training_log"] += 1
    elif relative_path.startswith(
        "validation/"
    ):
        role_counts[
            "validation_artifact"
        ] += 1
    elif relative_path.startswith(
        "configs/"
    ):
        role_counts["contract"] += 1
    elif relative_path.startswith(
        "audits/"
    ):
        role_counts["audit"] += 1
    else:
        role_counts["marker_or_other"] += 1

complete_payload = {
    "marker_version": (
        "phase2_s2_seed2026_full_run_complete_v1"
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": "complete",
    "student_id": STUDENT_ID,
    "student_name": STUDENT_NAME,
    "seed": DEVELOPMENT_SEED,
    "completed_epochs": len(epoch_history),
    "global_optimizer_steps": int(
        training_result[
            "global_optimizer_step"
        ]
    ),
    "best_epoch": best_epoch,
    "best_validation_cross_entropy": (
        best_validation_ce
    ),
    "best_validation_metrics": dict(
        best_metrics
    ),
    "last_checkpoint": {
        "path": str(
            S2_LAST_CHECKPOINT_PATH
        ),
        "sha256": (
            last_checkpoint_info["sha256"]
        ),
    },
    "best_checkpoint": {
        "path": str(
            S2_BEST_CHECKPOINT_PATH
        ),
        "sha256": (
            best_checkpoint_info["sha256"]
        ),
    },
    "manifest": manifest_files,
    "manifest_artifact_count": len(
        manifest_rows
    ),
    "manifest_role_counts": dict(
        role_counts
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
    "ready_for_phase2_s2_packaging": True,
}
write_json(
    S2_FULL_RUN_COMPLETE_PATH,
    complete_payload,
)
S2_FULL_RUN_COMPLETE_SHA256 = (
    sha256_file(
        S2_FULL_RUN_COMPLETE_PATH
    )
)


print()
print("=" * 72)
print("PHASE 2 S2 CELL 11 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Student                   : S2 — MobileViT-S")
print("Seed                      :", DEVELOPMENT_SEED)
print("Run mode                  :", run_mode)
print("Completed epochs          :", len(epoch_history))
print(
    "Global optimizer steps    :",
    training_result["global_optimizer_step"],
)
print(
    "FP16 overflow retries     :",
    training_result["overflow_retry_count"],
)
print("Validation evaluations    :", len(validation_epoch_files))
print("Best epoch                :", best_epoch)
print(
    "Best Validation CIDEr-D   :",
    f"{float(best_metrics['CIDEr-D']):.8f}",
)
print(
    "Best Validation BLEU-4    :",
    f"{float(best_metrics['BLEU-4']):.8f}",
)
print(
    "Best Validation CE        :",
    f"{best_validation_ce:.8f}",
)
print("Last checkpoint           : VERIFIED")
print("Best checkpoint           : VERIFIED")
print("Epoch-history CSV         : VERIFIED")
print("Optimizer-step JSONL      : VERIFIED")
print("Validation artifacts      :", len(validation_epoch_files), "VERIFIED")
print("Manifest artifacts        :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Selection split           : Validation only")
print("Test Parquet opened       : False")
print("Test caption read         : False")
print("Test images read          : False")
print("Test predictions written  : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Flickr data used          : False")
print("Audit                     :", S2_CELL11_AUDIT_PATH)
print("Audit SHA-256             :", S2_CELL11_AUDIT_SHA256)
print("Manifest CSV              :", S2_FINAL_MANIFEST_CSV_PATH)
print(
    "Manifest CSV SHA-256      :",
    manifest_files["csv"]["sha256"],
)
print("Manifest JSON             :", S2_FINAL_MANIFEST_JSON_PATH)
print(
    "Manifest JSON SHA-256     :",
    manifest_files["json"]["sha256"],
)
print("SHA256SUMS                :", S2_FINAL_SHA256SUMS_PATH)
print(
    "SHA256SUMS SHA-256        :",
    manifest_files["sha256sums"]["sha256"],
)
print("COMPLETE marker           :", S2_FULL_RUN_COMPLETE_PATH)
print("COMPLETE SHA-256          :", S2_FULL_RUN_COMPLETE_SHA256)
print(
    "Elapsed                   :",
    round(cell_elapsed_seconds / 3600.0, 3),
    "giờ",
)
print(
    "Next                      : Save Version with complete output; "
    "then audit/package S2 seed 2026 and continue remaining confirmation runs"
)